## Red Neuronal Convolucional (CNN) entrenada para dataset de detección visual de personajes de los Simpsons

> **Dataset nuevo: 64 personajes, una carpeta por personaje. Variante CPU (Keras 3 sobre PyTorch, sin GPU).** Sigue el
> mismo proceso que el notebook del Perceptrón Multicapa sobre este dataset
> ([`../../../PMC/notebooks/newDataset/LosSimpsonsPMC.ipynb`](../../../PMC/notebooks/newDataset/LosSimpsonsPMC.ipynb)):
> la misma partición 70/20/10 sin tope por personaje, las mismas imágenes alteradas y parecidas, las mismas
> etiquetas de escena, los mismos pesos por clase, el mismo bucle de entrenamiento, el mismo criterio de
> acierto y los mismos KPIs. Cambia el modelo:
>
> 1. **La imagen entra entera.** La red recibe el tensor de 64 × 64 × 3 píxeles de cada imagen, sin aplanarlo
>    y sin resumirlo en descriptores de color y textura elegidos a mano.
> 2. **Capas convolucionales.** Filtros de 3 × 3 que recorren la imagen con los mismos pesos en todas sus
>    posiciones, seguidos de normalización por lotes, ReLU y *max pooling*. La red aprende sus propios filtros.
> 3. **Experimentos propios de una CNN.** Además de los del MLP (activación, tasa de aprendizaje, batch,
>    dropout, pesos por clase, suavizado, escenas y alteradas), se comparan la profundidad, la cabeza
>    (pooling global frente a aplanado) y una entrada en escala de grises, que mide cuánto depende la red del
>    color.
> 4. **Grad-CAM.** El análisis de resultados muestra en qué zonas de la imagen se apoya la red para decidir, y
>    la última sección compara la corrida con la del MLP a la misma resolución.
>
> Antes de ejecutarlo hay que correr una vez el script de alteraciones; los pasos están en
> [`README.md`](README.md).
>
> El proceso es el mismo del notebook de GPU ([`LosSimpsons_GPU_4060M.ipynb`](LosSimpsons_GPU_4060M.ipynb)),
> con el mismo framework y el mismo bucle de entrenamiento: solo cambia el dispositivo, que aquí es
> la CPU. No usa TensorFlow.

#### Integrantes:

    - VINCENT FARENDEN CERÓN
    - RODRIGO IGNACIO MARTÍNEZ BECKER
    - IVÁN MANUEL PAVÓN BARRÍA
    - DIEGO IGNACIO PEÑA Y LILLO LUHRS

#### Asignatura:

    - TÉCNICAS AVANZADAS DE MACHINE LEARNING_001D

#### Profesor:

    - FRANCISCO JAVIER JEREZ SALAZAR

#### Fecha:

    - 19-10-2026

## 0. Configuración del entorno CPU

Esta corrida usa el mismo framework que la de GPU, **Keras 3 sobre PyTorch**, y no necesita
TensorFlow: el modelo se define con Keras y se entrena con el mismo bucle sobre PyTorch que el notebook de
GPU ([`LosSimpsons_GPU_4060M.ipynb`](LosSimpsons_GPU_4060M.ipynb)). Lo único que cambia es el
dispositivo: aquí todo se calcula en la CPU, aunque el equipo tenga una GPU. Así, al comparar los tiempos de
las dos corridas, la diferencia se debe al hardware y no al framework.

Para eso, antes de importar PyTorch o Keras, la celda siguiente fija tres variables de entorno:

| Variable | Valor | Para qué |
|---|---|---|
| `KERAS_BACKEND` | `torch` | Keras usa PyTorch, como en el notebook de GPU. Sin ella, Keras intenta cargar TensorFlow |
| `KERAS_TORCH_DEVICE` | `cpu` | Keras crea los pesos y los tensores del modelo en la CPU |
| `CUDA_VISIBLE_DEVICES` | `-1` | PyTorch no ve ninguna GPU. Es una garantía más: el notebook de GPU comprobó que, con una GPU visible, Keras deja en ella los tensores internos de `BatchNormalization` aunque el modelo se cree en la CPU. El valor vacío no sirve en Windows, donde una variable vacía equivale a no definirla |

Las tres solo tienen efecto si se fijan antes del primer `import torch` o `import keras` del kernel. Si el
kernel ya los importó con otra configuración, la celda lo detecta y pide reiniciarlo. El entorno es el
mismo `.venv-gpu` del notebook de GPU.

In [ ]:
import os
import platform
import sys

# Antes del primer import de torch o keras: Keras sobre PyTorch (no TensorFlow) y todo el calculo
# en la CPU. CUDA_VISIBLE_DEVICES=-1 oculta las GPU a PyTorch; un valor vacio no sirve en Windows.
ya_importados = [modulo for modulo in ("torch", "keras") if modulo in sys.modules]
os.environ["KERAS_BACKEND"] = "torch"
os.environ["KERAS_TORCH_DEVICE"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import keras
import torch

DISPOSITIVO = "cpu"

print(f"Python : {sys.version.split()[0]}")
print(f"Keras  : {keras.__version__}   (backend: {keras.backend.backend()})")
print(f"PyTorch: {torch.__version__}")

dispositivo_keras = keras.ops.zeros((1,)).device
if keras.backend.backend() != "torch" or torch.cuda.is_available() or dispositivo_keras.type != DISPOSITIVO:
    raise RuntimeError(
        f"El kernel ya habia importado {', '.join(ya_importados) or 'torch o keras'} con otra configuracion"
        f" (backend {keras.backend.backend()}, GPU visible: {torch.cuda.is_available()}, tensores de Keras"
        f" en {dispositivo_keras}). Reinicia el kernel y ejecuta el notebook desde esta celda."
    )

print(f"\nProcesador       : {platform.processor() or platform.machine()}")
print(f"Nucleos logicos  : {os.cpu_count()}")
print(f"Hilos de PyTorch : {torch.get_num_threads()}")

# Prueba real: una multiplicacion de matrices que debe ejecutarse en la CPU
prueba = torch.randn(2048, 2048)
resultado = prueba @ prueba
assert resultado.device.type == DISPOSITIVO, "La operacion de prueba no se ejecuto en la CPU"
print(f"\nOperacion de prueba ejecutada en: {resultado.device}   |   tensores de Keras en: {dispositivo_keras}")
del prueba, resultado

print("\nEntorno CPU verificado correctamente: Keras sobre PyTorch, sin GPU visible.")

## Definición del Dataset

Nuestro conjunto de datos es una carpeta por personaje, sin una partición previa en entrenamiento y prueba: cada carpeta reúne imágenes de ese personaje desde varios puntos de vista. Se construyó a partir del dataset público de Kaggle y el equipo lo amplió con más personajes y con fotogramas propios. En algunas carpetas hay imágenes cuyo nombre de archivo corresponde a otro personaje: son fotogramas elegidos porque en ellos el protagonista es el personaje de la carpeta. Por eso **la etiqueta de cada imagen la da siempre su carpeta, nunca el nombre del archivo.**

Nosotros escogimos este dataset debido a la gran popularidad de los personajes y porque tienen una identidad gráfica extremadamente reconocible: piel amarilla, siluetas simples, colores fuertes y personajes fáciles de distinguir incluso con pocos detalles. Su diseño es consistente y memorable, lo que hace que la serie destaque inmediatamente frente a otras animaciones.

| Dato | Valor |
|---|---|
| Personajes | 64 |
| Imágenes originales | 40.123 al 16-10-2026 (unos 2,1 GB), casi todas `.jpg`, con 30 `.png` y 2 `.webp` |
| Imágenes por personaje | De 3.118 (`homer_simpson`) a 94 (`louie`) |
| Tamaño de las imágenes | De 38 × 42 a 2.700 × 3.600 píxeles; mediana 640 × 480 |

Además de las originales, el script `scripts/newDataset/generar_imagenes_alteradas.py` guarda en cada carpeta una variante alterada de un cuarto de sus imágenes, con el nombre `<original>__alterada.<ext>`, y, para cada imagen casi idéntica a otra del mismo personaje, una versión filtrada `<original>__parecida.<ext>` que el notebook usa en su lugar. El dataset no se versiona en el repositorio.

![Homero](../../images/homero.png) ![Lisa](../../images/lisa.png)

## Problema de negocio, objetivos y KPIs

### Contexto y problema de negocio

Un archivo audiovisual crece mucho mas rapido de lo que un equipo humano puede
catalogarlo. Miles de fotogramas sin etiquetar son material practicamente inservible:
no se pueden buscar, no se pueden reutilizar y no permiten medir nada. Hoy esa
catalogacion se hace a mano, imagen por imagen, y su costo crece de forma lineal con
el tamano del archivo.

**Problema de negocio.** *El archivo de imagenes crece mas rapido de lo que el equipo
puede catalogarlo manualmente, lo que impide buscar material por personaje, encarece
cada proyecto de reutilizacion de contenido y deja sin explotar el catalogo
acumulado.*

Este proyecto aborda ese problema sobre un archivo de **40.123 imagenes (unos 2,1 GB)**
de 64 personajes de Los Simpson: asignar automaticamente el personaje que aparece en
cada imagen, para que el archivo pase de ser un conjunto de ficheros sueltos a un
catalogo consultable.

| Caso de uso | Que habilita el modelo |
|---|---|
| Busqueda en el archivo | Recuperar todos los planos de un personaje para un montaje, un trailer o un recopilatorio tematico. |
| Pre-clasificacion asistida | El modelo propone y una persona valida: convierte trabajo de *etiquetar* en trabajo de *revisar*, que es mucho mas rapido. |
| Metricas de contenido | Cuantificar la presencia en pantalla de cada personaje, dato relevante para licenciamiento y merchandising. |
| Control de uso de marca | Detectar personajes con derechos dentro de material aportado por terceros. |

### Objetivos del proyecto

**Objetivo general.** Implementar y evaluar una red neuronal convolucional (CNN) capaz
de asignar a cada imagen RGB uno de los 64 personajes del catalogo, mediante un flujo
reproducible cuyo desempeno se mida frente a referencias triviales y de forma
sensible al desbalance de clases.

**Objetivos especificos.**

1. Indexar las imagenes disponibles y asociar cada ruta con el personaje de su carpeta.
2. Repartir todas las imagenes de cada personaje en entrenamiento (70 %), validacion
   (20 %) y prueba (10 %) de forma estratificada y con semilla fija; cada variante
   alterada acompana a su imagen original, de modo que los tres conjuntos tienen las
   mismas condiciones.
3. Preparar los datos para la CNN: conversion a RGB, redimensionado a un tamano fijo,
   tensores de imagen sin aplanar, normalizacion y codificacion one-hot de las etiquetas.
4. Entrenar un modelo base y compararlo con variantes controladas de representacion de
   entrada, arquitectura e hiperparametros, cambiando un factor a la vez.
5. Seleccionar la configuracion final por F1 macro de validacion y medirla una sola
   vez sobre el conjunto de prueba.
6. Auditar el resultado: metricas por clase, matriz de confusion, aciertos, errores, zonas
   de la imagen en que se apoya la red (Grad-CAM) y comparacion con el MLP.

### KPIs del proyecto

Los umbrales se fijan **antes** de evaluar el modelo y traducen el escenario de uso a
cantidades medibles sobre los datos. El cumplimiento se verifica al final de la
seccion de evaluacion, con las cifras obtenidas en la prueba final.

**Criterio de acierto.** Una prediccion es correcta si nombra al personaje de la
carpeta o a otro personaje que tambien aparece en la imagen, segun la escena del
episodio o una imagen casi identica guardada en su carpeta (ver "Escenas con varios
personajes"). Con ese criterio se miden todas las metricas de prueba y los KPIs; la
accuracy estricta, que solo acepta el personaje de la carpeta, se informa como
referencia. La validacion, que decide la parada temprana y el modelo final, usa
siempre la etiqueta de la carpeta.

| KPI | Que mide en el negocio | Metrica tecnica | Umbral |
|---|---|---|---|
| Cobertura de asistencia | Con que frecuencia el catalogador encuentra la respuesta correcta entre las tres sugerencias del sistema. | Top-3 accuracy en test | >= 0,70 |
| Equidad entre personajes | Que el catalogo sirva para todos los personajes y no solo para los mas frecuentes. | F1 macro en test | >= 0,45 |
| Automatizacion sin revision | Proporcion de imagenes que el sistema resuelve sin intervencion humana. | Accuracy en test | >= 0,50 |
| Ventaja sobre el criterio trivial | Que el sistema aporte informacion real frente a etiquetar todo como el personaje mas comun. | Accuracy / baseline mayoritaria | >= 4x |
| Costo de catalogacion | Capacidad de procesar el archivo completo en un tiempo razonable. | Imagenes por segundo (entrada + inferencia) | >= 1.000 img/s |

La eleccion de **top-3 accuracy** y **F1 macro** como KPIs principales no es casual:
el sistema esta pensado como asistente de un catalogador, no como reemplazo, y un
catalogo que solo reconoce a los personajes mayoritarios no resuelve el problema de
busqueda. Por eso la seleccion de modelo se hace por F1 macro y no por accuracy.

> Las imagenes se trabajan a 64x64. A diferencia del MLP, la CNN recibe la imagen
> completa como un tensor de 64 x 64 x 3 (12.288 valores), sin aplanarla y sin
> resumirla en descriptores. Los umbrales de los KPIs son los mismos del MLP, para que
> las dos redes se midan con la misma vara.

## Definición del problema de clasificación

El problema que abordamos es de **clasificación de imágenes multiclase de etiqueta única**: dada una imagen RGB que contiene a un personaje de Los Simpsons, el modelo debe asignarle **exactamente una** de las 64 clases posibles (una por personaje). No es detección de objetos ni clasificación multi-etiqueta: no se predicen cajas delimitadoras ni varios personajes por imagen.

**Alcance verificable del trabajo.** El escenario descrito en la sección anterior es el marco de uso declarado del proyecto, no un encargo de un cliente real: no hay contrato, operación comercial ni cifras económicas que permitan calcular un retorno monetario. Por eso los KPIs se expresan en cantidades medibles sobre los datos disponibles y no en unidades de dinero.

**Variante definida: 64 personajes, una carpeta por personaje.** El dataset ya no trae una partición en `train` y `test`: el notebook recorre cada carpeta y reparte todas sus imágenes originales en entrenamiento (70 %), validación (20 %) y prueba (10 %), de forma estratificada y con semilla fija. Los tres conjuntos conservan la distribución real del dataset y tienen las mismas condiciones: las originales que les tocan, la versión filtrada de las que eran casi idénticas a otra del mismo personaje (ver «Imágenes parecidas») y la variante alterada de un cuarto de ellas, que siempre va al mismo conjunto que su original. No hay tope por personaje: el entrenamiento usa todas las originales que le corresponden.

**Criterios de éxito observables.** El desempeño se interpreta frente a tres referencias que calcula el notebook: accuracy aleatoria (1/64 ≈ 1,6 %), accuracy de predecir siempre la clase mayoritaria (en torno al 7,8 %, la proporción de `homer_simpson` en el dataset) y F1 macro, que da el mismo peso a los 64 personajes. También se reportan precision, recall, F1 por clase, matriz de confusión y top-3 accuracy. No se fijan umbrales nuevos a posteriori: los de los KPIs se declaran antes de entrenar, en la sección «Problema de negocio, objetivos y KPIs», y su cumplimiento se verifica al cerrar la sección de evaluación. Lo mismo vale para el criterio de acierto: en la prueba, una predicción es correcta si nombra al personaje de la carpeta o a otro que aparece en la misma imagen.

**Entrada y salida**

- **Entrada (X):** imagen RGB de tamaño variable (entre 38 × 42 y 2.700 × 3.600 px; mediana 640 × 480). Se redimensiona a 64 × 64 y entra a la red como un tensor de 64 × 64 × 3 píxeles normalizados, sin aplanar: las capas convolucionales trabajan sobre la vecindad de cada píxel.
- **Salida (y):** vector de 64 probabilidades (activación *softmax*); la clase predicha es la de mayor probabilidad.
- **Función de pérdida:** entropía cruzada categórica (*categorical crossentropy*).

**Conjuntos de datos**

| Conjunto | Contenido | Distribución |
|---|---|---|
| Entrenamiento | 70 % de las originales de cada personaje, más sus variantes alteradas | Real |
| Validación | 20 % de las originales de cada personaje, más sus variantes alteradas | Real |
| Prueba | 10 % de las originales de cada personaje, más sus variantes alteradas | Real |

Las cifras exactas de cada conjunto las imprime la sección de pre-procesamiento.

**Desbalance de clases**

En el dataset completo las clases están muy desbalanceadas: desde 3.118 imágenes de `homer_simpson` hasta 94 de `louie` (razón aproximada de 33:1). Esa razón se mantiene en entrenamiento, porque se usan todas las imágenes, y se compensa con pesos por clase **suavizados** (la raíz cuadrada de los balanceados): los personajes pequeños pesan más, sin que el modelo deje de reconocer a los frecuentes. Validación y prueba se miden sobre la distribución real, donde la *accuracy* favorece a los personajes frecuentes y el F1 macro a los escasos.

**Baselines de referencia**

- Clasificador aleatorio: 1/64 ≈ **1,6 %** de accuracy.
- Clasificador de clase mayoritaria (predecir siempre `homer_simpson`): en torno al **7,8 %** en prueba, la proporción de ese personaje en el dataset.

Cualquier modelo útil debe superar claramente estos valores.

**Métricas de evaluación**

Además de la *accuracy* global se reportarán **precision, recall y F1-score por clase**, el **F1 macro** (que pondera igual a todas las clases, mostrando el efecto del desbalance) y la **matriz de confusión**, útil para detectar confusiones sistemáticas entre personajes visualmente parecidos (por ejemplo, Bart y Milhouse, o Lisa y Maggie).

**Dificultades esperadas**

Los personajes aparecen en múltiples poses, con distintos encuadres, escalas y fondos, y con variaciones de color debidas a las diferentes temporadas. A esto se suman personajes parcialmente ocluidos o acompañados por otros en la misma escena. Con 64 personajes aparecen además secundarios muy parecidos entre sí —Patty y Selma, o los distintos miembros de la familia Flanders— y varios con menos de 150 imágenes en total.

**Supuestos**

Se asume que cada imagen tiene un único personaje **principal**: el que da nombre a su carpeta. Puede haber otros en el encuadre; la sección «Escenas con varios personajes» los estima a partir de los fotogramas de episodio y de las imágenes casi idénticas guardadas en la carpeta de otro personaje, los usa como etiqueta secundaria y, en la prueba, acepta como acierto nombrar a cualquiera de ellos. Se comprobó que el dataset no contiene archivos duplicados byte a byte, de modo que una misma imagen no puede caer a la vez en entrenamiento y prueba. Sí contiene imágenes casi idénticas —fotogramas seguidos de una escena, recortes repetidos—: el script las detecta y, de cada grupo, todas menos una entran con una versión filtrada, para que el modelo no acierte en prueba solo porque vio casi la misma imagen al entrenar. Cada variante alterada entra al mismo conjunto que su original: nunca hay una imagen en un conjunto y su variante en otro.

## Pre-Procesamiento de los datos

En esta sección se prepara el dataset para que pueda ser consumido por una red convolucional: se indexan las carpetas de los personajes, se reparte cada uno en entrenamiento, validación y prueba, se cargan y redimensionan las imágenes elegidas, se comprueba la normalización de los píxeles, se dejan como tensores de imagen sin aplanar y se codifican las etiquetas.

### Definición del conjunto de datos a utilizar

In [ ]:
import os
from datetime import datetime
from pathlib import Path

# Marca de inicio de la corrida. La seccion de rendimiento del final del notebook
# la usa para calcular la hora de termino y la duracion total.
INICIO_CORRIDA = datetime.now()
print(f"Inicio de la corrida: {INICIO_CORRIDA:%Y-%m-%d %H:%M:%S}\n")

# Una carpeta por personaje, sin particion previa. La ruta se puede fijar con la
# variable SIMPSONS_DATASET (la que usa Docker); si no existe, se busca en el escritorio.
DATASET_BASE = Path(
    os.environ.get("SIMPSONS_DATASET", Path.home() / "Desktop" / "simpsons_dataset" / "simpsons_dataset")
)

SEMILLA = 42
CONJUNTOS = ("train", "val", "test")
# Proporciones sobre TODAS las imagenes originales de cada personaje, sin tope:
# los tres conjuntos conservan la distribucion real del dataset.
PROPORCIONES = {"train": 0.70, "val": 0.20, "test": 0.10}
# Por debajo de este numero de originales un personaje no se puede repartir en tres
# conjuntos estratificados con un minimo de ejemplos en cada uno.
MINIMO_POR_PERSONAJE = 30

if not DATASET_BASE.is_dir():
    raise FileNotFoundError(f"No existe el dataset: {DATASET_BASE}")
if (DATASET_BASE / "train").is_dir() or (DATASET_BASE / "test").is_dir():
    raise ValueError(
        f"{DATASET_BASE} tiene la estructura antigua (train/ y test/). Este notebook trabaja "
        "con una carpeta por personaje; el del dataset anterior esta en PMC/notebooks/oldDataset/."
    )

print(f"Dataset       : {DATASET_BASE}")
print(f"Particion     : {', '.join(f'{nombre} {proporcion:.0%}' for nombre, proporcion in PROPORCIONES.items())}")
print("Tope de train : ninguno, se usan todas las originales")
print(
    "\nEl conjunto de test se reserva intacto: no se usa para ajustar pesos"
    " ni para elegir hiperparámetros, solo para la comprobación final."
)

### Carga de imágenes y etiquetas

In [ ]:
import pandas as pd

EXTENSIONES_VALIDAS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
# Sufijos con los que el script de alteraciones marca las imagenes que genera: una
# alterada es una variante extra que va al mismo conjunto que su original; una parecida
# reemplaza a su original
MARCA_ALTERADA = "__alterada"
MARCA_PARECIDA = "__parecida"


def construir_indice(directorio):
    """Recorre un directorio con una subcarpeta por personaje y devuelve un DataFrame con
    la ruta de cada imagen, su personaje y, para las alteradas, la original de la que
    provienen. Si el script reemplazó una original casi idéntica a otra por su versión
    __parecida, la fila de esa original apunta a la versión parecida. La etiqueta sale
    siempre de la carpeta, nunca del nombre del archivo."""
    registros = []

    for carpeta_clase in sorted(directorio.iterdir()):
        if not carpeta_clase.is_dir() or carpeta_clase.name.startswith("."):
            continue

        archivos = sorted(carpeta_clase.iterdir())
        nombres = {ruta.name for ruta in archivos}
        for ruta in archivos:
            # Las parecidas no son imágenes nuevas: entran abajo, en el lugar de su original
            if ruta.suffix.lower() not in EXTENSIONES_VALIDAS or ruta.stem.endswith(MARCA_PARECIDA):
                continue
            alterada = ruta.stem.endswith(MARCA_ALTERADA)
            # Para una original, su origen es ella misma
            origen = ruta.stem.removesuffix(MARCA_ALTERADA) + ruta.suffix if alterada else ruta.name
            reemplazo = f"{ruta.stem}{MARCA_PARECIDA}{ruta.suffix}"
            parecida = not alterada and reemplazo in nombres
            registros.append(
                {
                    "ruta": ruta.with_name(reemplazo) if parecida else ruta,
                    "personaje": carpeta_clase.name,
                    "alterada": alterada,
                    "parecida": parecida,
                    "origen": origen,
                }
            )

    if not registros:
        raise FileNotFoundError(f"No se encontraron imagenes en: {directorio}")

    return pd.DataFrame(registros)


# Índice de todo el dataset: originales y alteradas
indice_completo = construir_indice(DATASET_BASE)

resumen = (
    indice_completo.groupby("personaje")
    .agg(
        originales=("alterada", lambda serie: int((~serie).sum())),
        alteradas=("alterada", "sum"),
        parecidas=("parecida", "sum"),
    )
    .astype(int)
    .sort_values("originales", ascending=False)
    .reset_index()
)
resumen["porcentaje"] = (resumen["originales"] / resumen["originales"].sum() * 100).round(2)

n_originales = int(resumen["originales"].sum())
n_alteradas = int(resumen["alteradas"].sum())
n_parecidas = int(resumen["parecidas"].sum())
print(f"Dataset: {n_originales} imagenes originales y {n_alteradas} alteradas en {len(resumen)} personajes")
print(f"De las originales, {n_parecidas} eran casi identicas a otra y entran con su version filtrada (__parecida)")
if n_alteradas == 0:
    print(
        "AVISO: no hay imagenes alteradas. Ejecuta antes "
        "scripts/newDataset/generar_imagenes_alteradas.py; el notebook funciona sin ellas, "
        "pero el entrenamiento no tendra variantes de color y textura."
    )

display(resumen)

### Partición y selección por personaje

Con el dataset ya indexado, cada personaje se reparte **antes de cargar ninguna imagen**. Así solo se cargan las imágenes que realmente se van a usar, y la partición no puede depender de nada calculado sobre los píxeles.

1. **Solo se reparten las originales.** Las de cada personaje se dividen en entrenamiento (70 %), validación (20 %) y test (10 %) con `train_test_split` estratificado y semilla fija. Los tres conjuntos conservan así la distribución real del dataset.
2. **Sin tope por personaje.** Se usan todas las originales: ninguna queda fuera del reparto. El desbalance (33:1) no se corrige descartando imágenes de los personajes grandes, sino con pesos por clase (ver «Tratamiento del desbalance de clases»).
3. **Las alteradas siguen a su original.** Cada variante alterada entra al mismo conjunto que su imagen original: entrenamiento, validación o test. Así los tres conjuntos tienen las mismas condiciones —originales, parecidas y alteradas en la misma proporción— y el modelo se mide sobre el mismo tipo de imágenes con que aprende. Nunca hay una imagen en un conjunto y su variante en otro: el modelo no puede acertar en test porque vio casi la misma imagen al entrenar.
4. **Personajes demasiado pequeños.** Un personaje con menos de `MINIMO_POR_PERSONAJE` originales no se puede repartir en tres conjuntos con un mínimo de ejemplos en cada uno; se lista y se excluye.
5. **Las parecidas ocupan el lugar de su original.** Si el script reemplazó una original casi idéntica a otra por su versión filtrada (`__parecida`), esa versión se reparte como cualquier original y puede caer en cualquier conjunto; la original no se usa.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

originales = indice_completo[~indice_completo["alterada"]].reset_index(drop=True)
alteradas = indice_completo[indice_completo["alterada"]].reset_index(drop=True)

# Punto 4, que se aplica antes que los demás: personajes con muy pocas originales
# para repartirlos en tres conjuntos
conteo_originales = originales["personaje"].value_counts()
excluidos = sorted(conteo_originales[conteo_originales < MINIMO_POR_PERSONAJE].index)
if excluidos:
    print(f"Personajes excluidos (menos de {MINIMO_POR_PERSONAJE} originales): {', '.join(excluidos)}")
    originales = originales[~originales["personaje"].isin(excluidos)].reset_index(drop=True)
    alteradas = alteradas[~alteradas["personaje"].isin(excluidos)].reset_index(drop=True)

# 1. Reparto estratificado de las originales: primero se separa el 70 % de train, y el
#    30 % restante se divide entre validacion (20 %) y test (10 %).
fraccion_resto = 1 - PROPORCIONES["train"]
indices_train, indices_resto = train_test_split(
    originales.index,
    test_size=fraccion_resto,
    stratify=originales["personaje"],
    random_state=SEMILLA,
)
indices_val, indices_test = train_test_split(
    indices_resto,
    test_size=PROPORCIONES["test"] / fraccion_resto,
    stratify=originales.loc[indices_resto, "personaje"],
    random_state=SEMILLA,
)
originales["conjunto"] = ""
originales.loc[indices_train, "conjunto"] = "train"
originales.loc[indices_val, "conjunto"] = "val"
originales.loc[indices_test, "conjunto"] = "test"

# 2. Sin tope por personaje: todas las originales de train se usan para entrenar.
# 3. Cada alterada va al mismo conjunto que su original, así train, validación y test tienen las
#    mismas condiciones. Solo se descarta una alterada cuya original ya no está en el dataset.
conjunto_de_original = dict(zip(zip(originales["personaje"], originales["origen"]), originales["conjunto"]))
alteradas["conjunto"] = [
    conjunto_de_original.get((personaje, origen), "descartada")
    for personaje, origen in zip(alteradas["personaje"], alteradas["origen"])
]

datos = (
    pd.concat([originales, alteradas], ignore_index=True)
    .query("conjunto != 'descartada'")
    .sort_values(["personaje", "origen", "alterada"])
    .reset_index(drop=True)
)

# Lista ordenada de clases y su mapeo a índices enteros
clases = sorted(datos["personaje"].unique())
clase_a_indice = {clase: indice for indice, clase in enumerate(clases)}
indice_a_clase = {indice: clase for clase, indice in clase_a_indice.items()}

# Etiqueta numérica de cada imagen: la de su carpeta
datos["etiqueta"] = datos["personaje"].map(clase_a_indice)


def contar(filtro):
    return datos[filtro].groupby("personaje").size()


reparto = (
    pd.DataFrame(
        {
            "disponibles": originales.groupby("personaje").size(),
            "train_originales": contar((datos["conjunto"] == "train") & ~datos["alterada"]),
            "train_alteradas": contar((datos["conjunto"] == "train") & datos["alterada"]),
            "val": contar(datos["conjunto"] == "val"),
            "val_alteradas": contar((datos["conjunto"] == "val") & datos["alterada"]),
            "test": contar(datos["conjunto"] == "test"),
            "test_alteradas": contar((datos["conjunto"] == "test") & datos["alterada"]),
        }
    )
    .fillna(0)
    .astype(int)
)
reparto["train_total"] = reparto["train_originales"] + reparto["train_alteradas"]
reparto = reparto.sort_values("disponibles", ascending=False)
display(reparto)

totales = datos["conjunto"].value_counts()
alteradas_por_conjunto = datos[datos["alterada"]].groupby("conjunto").size()
print(
    "   |   ".join(
        f"{nombre}: {totales[nombre]} imagenes ({int(alteradas_por_conjunto.get(nombre, 0))} alteradas)"
        for nombre in CONJUNTOS
    )
    + f"   |   personajes: {len(clases)}"
)
print(
    f"Desbalance en train: {reparto['train_total'].max() / reparto['train_total'].min():.1f}:1"
    f"   (en el dataset completo: {conteo_originales.max() / conteo_originales.min():.1f}:1)"
)
parecidas_por_conjunto = datos[datos["parecida"]].groupby("conjunto").size()
print(
    "Imagenes parecidas (version filtrada en lugar de la original): "
    + "   |   ".join(f"{nombre} {int(parecidas_por_conjunto.get(nombre, 0))}" for nombre in CONJUNTOS)
)
n_por_conjunto = originales["conjunto"].value_counts()
print(
    "Reparto de las originales: "
    + "   |   ".join(f"{nombre} {n_por_conjunto[nombre] / len(originales):.1%}" for nombre in CONJUNTOS)
)

### Redimensionado a un tamaño fijo



In [ ]:
from PIL import Image
import numpy as np

# Dimensiones originales de cada imagen del conjunto (sin cargar los píxeles)
dimensiones = []
for ruta in datos["ruta"]:
    with Image.open(ruta) as img:
        dimensiones.append(img.size)  # (ancho, alto)

datos["ancho"] = [ancho for ancho, _ in dimensiones]
datos["alto"] = [alto for _, alto in dimensiones]

resumen_dimensiones = datos[["ancho", "alto"]].describe().T
resumen_dimensiones["mediana"] = datos[["ancho", "alto"]].median()
resumen_dimensiones = resumen_dimensiones[
    ["count", "min", "25%", "mediana", "75%", "max", "mean", "std"]
].round(1)

display(resumen_dimensiones)

**Elección y justificación del tamaño objetivo**

Las imágenes del conjunto tienen dimensiones variables, y una red, convolucional o no, necesita que todos los
ejemplos de un lote tengan la misma forma. Por lo tanto, el primer paso del pre-procesamiento consiste en
llevar todas las imágenes a una misma resolución.

En una CNN la resolución no cambia el número de parámetros de las capas convolucionales —un filtro de
3 x 3 tiene los mismos pesos a 32 x 32 que a 128 x 128—, pero sí el **costo de cómputo**: cada filtro se
aplica en todas las posiciones de la imagen, así que el trabajo crece con el número de píxeles, cuatro veces
cada vez que se dobla el lado. Medido en este equipo con la CNN base y el bucle de entrenamiento del
notebook:

| Resolución | Valores por imagen | Activaciones de la primera capa (32 filtros) | Segundos por época en GPU | Segundos por época en CPU |
|---|---|---|---|---|
| 32 x 32 | 3.072 | 32.768 | ≈ 5 | ≈ 32 |
| **64 x 64 (esta corrida)** | **12.288** | **131.072** | **≈ 10** | **≈ 130** |
| 128 x 128 | 49.152 | 524.288 | ≈ 37 | — |

Este notebook fija **64 x 64 píxeles** en la celda siguiente. Es la resolución que el estudio del MLP
identificó como el mejor equilibrio entre resultado y costo, y a 128 x 128 cada época de la CNN cuesta casi
cuatro veces más. A 64 x 64 se distinguen todavía la silueta, el peinado y la ropa de cada personaje, que
es lo que una CNN puede aprovechar y un histograma de color no.

In [ ]:
import time

from joblib import Parallel, delayed

ALTO_OBJETIVO, ANCHO_OBJETIVO = 64, 64
TAMANO_OBJETIVO = (ANCHO_OBJETIVO, ALTO_OBJETIVO)  # PIL usa (ancho, alto)


def cargar_y_redimensionar(ruta, tamano=TAMANO_OBJETIVO):
    """Abre una imagen, la convierte a RGB y la redimensiona al tamaño fijo."""
    with Image.open(ruta) as img:
        img = img.convert("RGB").resize(tamano, Image.Resampling.BILINEAR)
        return np.asarray(img, dtype=np.uint8)


# Matriz de imágenes redimensionadas: (n_imagenes, alto, ancho, 3). Decodificar y
# redimensionar libera el GIL, así que varios hilos cargan en paralelo.
inicio_carga = time.perf_counter()
imagenes = np.stack(
    Parallel(n_jobs=-1, prefer="threads")(
        delayed(cargar_y_redimensionar)(ruta) for ruta in datos["ruta"]
    )
)
segundos_carga = time.perf_counter() - inicio_carga
etiquetas = datos["etiqueta"].to_numpy()

print(f"imagenes: {imagenes.shape}  dtype={imagenes.dtype}  ({imagenes.nbytes / 1024 ** 3:.2f} GB)")
print(f"etiquetas: {etiquetas.shape}")
print(f"Carga y redimensionado: {segundos_carga:.1f} s ({len(imagenes) / segundos_carga:,.0f} imagenes/s)")

**Comprobación del redimensionado**

In [ ]:
comprobaciones = pd.DataFrame(
    [
        ("Número de imágenes", imagenes.shape[0], len(datos), imagenes.shape[0] == len(datos)),
        ("Alto", imagenes.shape[1], ALTO_OBJETIVO, imagenes.shape[1] == ALTO_OBJETIVO),
        ("Ancho", imagenes.shape[2], ANCHO_OBJETIVO, imagenes.shape[2] == ANCHO_OBJETIVO),
        ("Canales", imagenes.shape[3], 3, imagenes.shape[3] == 3),
        ("Tipo de dato", str(imagenes.dtype), "uint8", imagenes.dtype == np.uint8),
        ("Valor mínimo", int(imagenes.min()), ">= 0", imagenes.min() >= 0),
        ("Valor máximo", int(imagenes.max()), "<= 255", imagenes.max() <= 255),
        ("Etiquetas alineadas", len(etiquetas), imagenes.shape[0], len(etiquetas) == imagenes.shape[0]),
    ],
    columns=["comprobacion", "obtenido", "esperado", "ok"],
)

memoria_mb = imagenes.nbytes / 1024**2
print(f"Memoria ocupada por la matriz de imagenes: {memoria_mb:.1f} MB")
print(f"Valores por imagen (alto x ancho x 3): {ALTO_OBJETIVO * ANCHO_OBJETIVO * 3}, sin aplanar")

display(comprobaciones)


In [ ]:
import matplotlib.pyplot as plt

# Comparación visual original vs redimensionada para una muestra aleatoria
generador = np.random.default_rng(42)
muestra = generador.choice(len(datos), size=5, replace=False)

figura, ejes = plt.subplots(
    2,
    len(muestra),
    figsize=(2.4 * len(muestra), 5.8),
    layout="constrained",
)
for columna, indice in enumerate(muestra):
    fila = datos.iloc[indice]
    with Image.open(fila["ruta"]) as img:
        original = img.convert("RGB")

    personaje = fila["personaje"].replace("_", " ")
    ejes[0, columna].imshow(original)
    ejes[1, columna].imshow(imagenes[indice])

    # El texto va debajo de la imagen para que no invada la fila de arriba
    ejes[0, columna].set_xlabel(
        f"{personaje}\n{original.size[0]}x{original.size[1]}",
        fontsize=8,
        labelpad=6,
        linespacing=1.5,
    )

    for eje in (ejes[0, columna], ejes[1, columna]):
        eje.set_xticks([])
        eje.set_yticks([])
        for lado in eje.spines.values():
            lado.set_edgecolor("#999999")
            lado.set_linewidth(1.2)

# Etiqueta de fila a la izquierda, en vez de repetir el mismo título en cada columna
ejes[0, 0].set_ylabel("Original", fontsize=10, fontweight="bold", labelpad=8)
ejes[1, 0].set_ylabel(
    f"{ANCHO_OBJETIVO}x{ALTO_OBJETIVO}", fontsize=10, fontweight="bold", labelpad=8
)

figura.suptitle("Original (arriba) vs redimensionada (abajo)", fontsize=13, fontweight="bold")
plt.show()


### Normalización de los píxeles



**Por qué normalizar**
Al dividir por 255 todas las características de entrada quedan en la misma escala acotada
[0, 1]. Esto evita que unos píxeles pesen más que otros por su magnitud, mejora el
condicionamiento del descenso del gradiente y permite que la inicialización estándar de
los pesos funcione como se espera.

**Dónde se aplica**
El escalado **no se materializa aquí**, sino dentro del modelo, en la capa `Rescaling`
que es la primera capa de la CNN (sección del modelo base). El motivo es de memoria: convertir todas las imágenes a `float32` multiplica
por cuatro el espacio ocupado —la celda siguiente imprime las cifras de esta corrida—, y la
matriz de píxeles se copia además al separar train, validación y test.

Manteniendo los píxeles en `uint8`, cada copia ocupa la cuarta parte y el modelo escala cada
lote al recibirlo, a coste despreciable. Es el mismo esquema del notebook de GPU, así que los
modelos guardados por los dos reciben exactamente la misma entrada.

Esta celda comprueba sobre una muestra que el escalado de los píxeles es exactamente el esperado.

In [ ]:
# Escalado de [0, 255] a [0, 1]: el rango con el que opera la red.
VALOR_MAXIMO_PIXEL = 255.0

# El escalado real lo aplica la capa Rescaling del modelo, sobre cada lote.
# Aqui solo se verifica sobre una muestra que la operacion es la esperada.
TAMANO_MUESTRA_NORMALIZACION = 512
tamano_muestra = min(TAMANO_MUESTRA_NORMALIZACION, len(imagenes))
imagenes_norm = imagenes[:tamano_muestra].astype(np.float32) / VALOR_MAXIMO_PIXEL

print(f"imagenes      : {imagenes.shape}  dtype={imagenes.dtype}"
      f"  ->  {imagenes.nbytes / 1024 ** 3:.2f} GB")
print(f"imagenes_norm : {imagenes_norm.shape}  dtype={imagenes_norm.dtype}"
      f"  (muestra de comprobacion)")
print(f"rango: [{imagenes_norm.min():.3f}, {imagenes_norm.max():.3f}]")
print(f"media: {imagenes_norm.mean():.4f}  desviacion: {imagenes_norm.std():.4f}")
print(
    f"\nEl conjunto completo en float32 ocuparia {imagenes.nbytes * 4 / 1024 ** 3:.2f} GB;"
    f" en uint8 ocupa {imagenes.nbytes / 1024 ** 3:.2f} GB."
)

**Comprobación de la normalización**

In [ ]:
indice_prueba = 0

comprobaciones_norm = pd.DataFrame(
    [
        (
            "Forma conservada (sobre la muestra)",
            str(imagenes_norm.shape),
            str(imagenes[:tamano_muestra].shape),
            imagenes_norm.shape == imagenes[:tamano_muestra].shape,
        ),
        (
            "Tipo de dato",
            str(imagenes_norm.dtype),
            "float32",
            imagenes_norm.dtype == np.float32,
        ),
        (
            "Valor mínimo",
            round(float(imagenes_norm.min()), 4),
            ">= 0.0",
            imagenes_norm.min() >= 0.0,
        ),
        (
            "Valor máximo",
            round(float(imagenes_norm.max()), 4),
            "<= 1.0",
            imagenes_norm.max() <= 1.0,
        ),
        (
            "Sin NaN ni infinitos",
            bool(np.isfinite(imagenes_norm).all()),
            True,
            bool(np.isfinite(imagenes_norm).all()),
        ),
        (
            "Escalado exacto (x255 == original)",
            "imagenes_norm * 255",
            "imagenes",
            np.allclose(
                imagenes_norm[indice_prueba] * VALOR_MAXIMO_PIXEL,
                imagenes[indice_prueba].astype(np.float32),
            ),
        ),
    ],
    columns=["Comprobacion", "Obtenido", "Esperado", "Correcto"],
)

display(comprobaciones_norm)

if not comprobaciones_norm["Correcto"].all():
    raise ValueError("La normalización no cumple alguna de las comprobaciones.")

print("Normalización verificada correctamente.")

In [ ]:
# La imagen normalizada debe verse idéntica a la original: solo cambia la escala numérica
figura, ejes = plt.subplots(1, 2, figsize=(6.4, 3.6), layout="constrained")

paneles = (
    (imagenes[indice_prueba], "uint8 [0, 255]"),
    (imagenes_norm[indice_prueba], "float32 [0, 1]"),
)
for eje, (imagen, etiqueta) in zip(ejes, paneles):
    eje.imshow(imagen)
    eje.set_xticks([])
    eje.set_yticks([])
    for lado in eje.spines.values():
        lado.set_edgecolor("#999999")
        lado.set_linewidth(1.2)
    eje.set_xlabel(etiqueta, fontsize=9, labelpad=6)

personaje = datos.iloc[indice_prueba]["personaje"].replace("_", " ")
figura.suptitle(f"Personaje: {personaje}", fontsize=12, fontweight="bold")
plt.show()


### La imagen como tensor: la CNN no aplana

**Por qué la CNN no aplana la imagen**

El MLP necesitaba un vector: cada neurona de una capa densa se conecta con todas las entradas, y para eso
la imagen de `alto x ancho x 3` se convertía en una lista de `alto * ancho * 3` números. Ese paso es el que
**pierde la estructura espacial**: después de aplanar, la red ya no sabe que el píxel `(0, 0)` y el `(0, 1)`
eran vecinos.

Una capa convolucional trabaja al revés: recibe la imagen con su forma y aplica cada filtro sobre una
vecindad de 3 x 3 píxeles, desplazándolo por toda la imagen. Por eso la entrada de la CNN conserva las tres
dimensiones de cada imagen y el conjunto completo es un **tensor de cuatro dimensiones**:

| Dimensión | Tamaño en esta corrida | Qué cuenta |
|---|---|---|
| 0 | N | Imágenes |
| 1 | 64 | Filas (alto) |
| 2 | 64 | Columnas (ancho) |
| 3 | 3 | Canales de color R, G y B |

Es el formato `channels_last` de Keras: los tres canales de cada píxel van juntos al final. `y` sigue siendo
un vector `(N,)` con el índice de la clase de cada imagen; en «Codificación de las etiquetas» pasa a ser la
matriz one-hot `(N, 64)` que exige `categorical_crossentropy`. Lo único imprescindible es que
`X.shape[0] == y.shape[0]` y que el orden coincida: la etiqueta `y[i]` corresponde a la imagen `X[i]`.

La celda siguiente no copia los datos: `X` es la misma matriz de imágenes redimensionadas, en `uint8`. El
escalado a [0, 1] lo hace la capa `Rescaling` del modelo sobre cada lote.

In [ ]:
# La CNN recibe cada imagen con su forma (alto, ancho, 3): el conjunto es un tensor de 4 dimensiones
FORMA_ENTRADA = (ALTO_OBJETIVO, ANCHO_OBJETIVO, 3)
DIMENSION_PIXELES = ALTO_OBJETIVO * ANCHO_OBJETIVO * 3

# Sin copia ni conversion: X es la misma matriz de imagenes en uint8. El escalado a [0, 1]
# lo hace la capa Rescaling del modelo sobre cada lote.
X = imagenes
y = etiquetas

print(f"X: {X.shape}  dtype={X.dtype}  ({X.ndim} dimensiones: imagenes, alto, ancho, canales)")
print(f"y: {y.shape}")
print(f"Forma de cada imagen: {FORMA_ENTRADA}  ->  {DIMENSION_PIXELES} valores, sin aplanar")
print(f"rango de X: [{int(X.min())}, {int(X.max())}]  (uint8; se escala dentro del modelo)")

**Comprobación del tensor de entrada**

In [ ]:
comprobaciones_tensor = pd.DataFrame(
    [
        ("Número de imágenes", X.shape[0], imagenes.shape[0], X.shape[0] == imagenes.shape[0]),
        ("Número de dimensiones", X.ndim, 4, X.ndim == 4),
        ("Forma de cada imagen", str(X.shape[1:]), str(FORMA_ENTRADA), X.shape[1:] == FORMA_ENTRADA),
        ("Tipo de dato", str(X.dtype), "uint8", X.dtype == np.uint8),
        (
            "Rango conservado [0, 255]",
            f"[{int(X.min())}, {int(X.max())}]",
            "[0, 255]",
            bool(X.min() >= 0 and X.max() <= 255),
        ),
        (
            "Sin copia de las imágenes",
            "np.shares_memory(X, imagenes)",
            True,
            bool(np.shares_memory(X, imagenes)),
        ),
        (
            "Misma imagen que la redimensionada",
            "X[i]",
            "imagenes[i]",
            np.array_equal(X[indice_prueba], imagenes[indice_prueba]),
        ),
        ("Etiquetas alineadas", len(y), X.shape[0], len(y) == X.shape[0]),
    ],
    columns=["Comprobacion", "Obtenido", "Esperado", "Correcto"],
)

display(comprobaciones_tensor)

if not comprobaciones_tensor["Correcto"].all():
    raise ValueError("El tensor de entrada no cumple alguna de las comprobaciones.")

print("Tensor de entrada verificado correctamente.")

In [ ]:
# La red no recibe un vector sino tres matrices de alto x ancho, una por canal de color
figura, ejes = plt.subplots(1, 4, figsize=(12, 3.6), layout="constrained")

ejes[0].imshow(imagenes[indice_prueba])
ejes[0].set_xlabel(f"Imagen RGB {ALTO_OBJETIVO}x{ANCHO_OBJETIVO}x3", fontsize=9, labelpad=6)
for eje, canal, nombre, mapa in zip(
    ejes[1:], range(3), ("R: rojo", "G: verde", "B: azul"), ("Reds", "Greens", "Blues")
):
    eje.imshow(X[indice_prueba, :, :, canal], cmap=mapa, vmin=0, vmax=255)
    eje.set_xlabel(f"Canal {nombre}  X[i, :, :, {canal}]", fontsize=9, labelpad=6)

for eje in ejes:
    eje.set_xticks([])
    eje.set_yticks([])
    for lado in eje.spines.values():
        lado.set_edgecolor("#999999")
        lado.set_linewidth(1.2)

personaje = datos.iloc[indice_prueba]["personaje"].replace("_", " ")
figura.suptitle(f"Lo que recibe la red: tres canales de color ({personaje})", fontsize=12, fontweight="bold")
plt.show()

**Imágenes alteradas por el script**

El script `scripts/newDataset/generar_imagenes_alteradas.py` aplica a cada imagen elegida un giro de 10 a
20 grados (recortando las esquinas que quedan vacías), un cambio de color (brillo, contraste, saturación
o tono) y un filtro de textura (desenfoque, nitidez, ruido o realce de bordes), con rangos moderados para
que el personaje siga siendo reconocible. La galería muestra
algunas parejas de entrenamiento, original arriba y alterada abajo, con la transformación que registró el
script en `_alteraciones.csv`.

In [ ]:
# Parejas original / alterada del conjunto de entrenamiento, con la transformación aplicada
alteradas_train = datos[datos["alterada"] & (datos["conjunto"] == "train")]

if alteradas_train.empty:
    print("No hay imagenes alteradas en entrenamiento: ejecuta antes el script de alteraciones.")
else:
    transformaciones = {}
    registro_alteraciones = DATASET_BASE / "_alteraciones.csv"
    if registro_alteraciones.exists():
        registro_script = pd.read_csv(registro_alteraciones)
        con_giro = "rotacion" in registro_script.columns
        transformaciones = {
            (fila.personaje, fila.alterada): (f"giro {fila.rotacion:+.0f}° + " if con_giro else "")
            + f"{fila.color} + {fila.textura}"
            for fila in registro_script.itertuples()
        }

    muestra_alteradas = alteradas_train.sample(n=min(6, len(alteradas_train)), random_state=SEMILLA)
    figura, ejes = plt.subplots(
        2,
        len(muestra_alteradas),
        figsize=(2.4 * len(muestra_alteradas), 5.6),
        layout="constrained",
        squeeze=False,
    )
    for columna, fila in enumerate(muestra_alteradas.itertuples()):
        ruta_original = fila.ruta.with_name(fila.origen)
        for eje, ruta in ((ejes[0, columna], ruta_original), (ejes[1, columna], fila.ruta)):
            with Image.open(ruta) as img:
                eje.imshow(img.convert("RGB"))
            eje.set_xticks([])
            eje.set_yticks([])
        ejes[0, columna].set_title(fila.personaje.replace("_", " "), fontsize=8)
        ejes[1, columna].set_xlabel(
            transformaciones.get((fila.personaje, fila.ruta.name), "alterada"), fontsize=8
        )

    ejes[0, 0].set_ylabel("Original", fontsize=10, fontweight="bold")
    ejes[1, 0].set_ylabel("Alterada", fontsize=10, fontweight="bold")
    figura.suptitle(
        "Variantes del script: giro, cambio de color y filtro de textura", fontsize=12, fontweight="bold"
    )
    plt.show()

    proporcion = len(alteradas_train) / int((datos["conjunto"] == "train").sum())
    print(f"Imagenes alteradas en entrenamiento: {len(alteradas_train)} ({proporcion:.1%} del conjunto)")

**Imágenes parecidas**

Muchas imágenes del dataset son casi idénticas a otra del mismo personaje: fotogramas seguidos de una
misma escena o recortes repetidos. Si una cae en entrenamiento y su gemela en prueba, el modelo puede
acertar solo porque ya vio casi la misma imagen. El script compara las imágenes de cada personaje con
miniaturas a color de 16 x 16 (similitud coseno, umbral 0,90 por defecto) y agrupa las casi idénticas. En
cada grupo una queda intacta y cada una de las demás se reemplaza por una versión con giro, cambio de
color y filtro de textura (`<original>__parecida.<ext>`), que se vuelve a sortear hasta que deja de
parecerse a las del grupo. El notebook usa esa versión **en lugar** de la original, en cualquier
conjunto. La galería muestra algunos grupos: la imagen que quedó intacta, su casi idéntica y la versión
con la que esta entra al notebook.

In [ ]:
# Grupos de imágenes parecidas: la representante queda intacta y cada casi idéntica entra
# al notebook con su versión filtrada
registro_parecidas = DATASET_BASE / "_parecidas.csv"
if not registro_parecidas.exists():
    print("No hay registro de imagenes parecidas: ejecuta antes el script de alteraciones.")
else:
    parecidas_script = pd.read_csv(registro_parecidas)
    print(
        f"Imagenes parecidas reemplazadas: {len(parecidas_script)} en {parecidas_script['grupo'].nunique()} grupos"
        f"   |   similitud mediana con su gemela: {parecidas_script['similitud_original'].median():.2f} antes"
        f" y {parecidas_script['similitud_final'].median():.2f} despues del filtro"
    )
    muestra_parecidas = parecidas_script.sample(n=min(5, len(parecidas_script)), random_state=SEMILLA)
    figura, ejes = plt.subplots(
        3,
        len(muestra_parecidas),
        figsize=(2.4 * len(muestra_parecidas), 8.2),
        layout="constrained",
        squeeze=False,
    )
    for columna, fila in enumerate(muestra_parecidas.itertuples()):
        carpeta = DATASET_BASE / fila.personaje
        paneles = (
            (fila.representante, fila.personaje.replace("_", " ")),
            (fila.original, f"similitud {fila.similitud_original:.2f}"),
            (
                fila.parecida,
                f"giro {fila.rotacion:+.0f}° + {fila.color} + {fila.textura}\nsimilitud {fila.similitud_final:.2f}",
            ),
        )
        for numero_fila, (nombre_archivo, texto) in enumerate(paneles):
            eje = ejes[numero_fila, columna]
            with Image.open(carpeta / nombre_archivo) as img:
                eje.imshow(img.convert("RGB"))
            eje.set_xticks([])
            eje.set_yticks([])
            eje.set_xlabel(texto, fontsize=7)

    ejes[0, 0].set_ylabel("Intacta", fontsize=10, fontweight="bold")
    ejes[1, 0].set_ylabel("Casi idéntica", fontsize=10, fontweight="bold")
    ejes[2, 0].set_ylabel("Entra al notebook", fontsize=10, fontweight="bold")
    figura.suptitle(
        "Imágenes parecidas: una queda intacta y su gemela entra con un filtro", fontsize=12, fontweight="bold"
    )
    plt.show()

### Codificación de las etiquetas

**De nombre de carpeta a vector one-hot**

Las etiquetas nacen como texto (el nombre de la carpeta, por ejemplo `homer_simpson`) y una red no opera
con texto, así que hay que convertirlas a números. La conversión se hace en dos pasos:

1. **Índice entero.** Ya se construyó en "Carga de imágenes y etiquetas": `clases` es la lista de
   personajes ordenada alfabéticamente y `clase_a_indice` asigna a cada uno un entero de 0 a 63. Ese
   mapeo se reutiliza aquí en lugar de recalcularlo, porque debe ser **idéntico en train y en test**: si
   el índice 3 significara un personaje distinto en cada conjunto, la evaluación no tendría sentido.

2. **One-hot encoding.** El índice entero por sí solo es engañoso para el modelo: sugiere un **orden** y
   una **distancia** que no existen (que la clase 5 este "entre" la 4 y la 6, o que la 63 sea "mayor" que
   la 1). Los personajes son categorías nominales, sin orden. El one-hot elimina ese artefacto
   representando cada etiqueta como un vector de 64 posiciones con un 1 en la clase correcta y 0 en el
   resto, de modo que todas las clases quedan a la misma distancia entre sí.

Además, esta forma es la que corresponde a la salida de la red: la capa final tiene 64 neuronas con
`softmax`, que produce un vector de 64 probabilidades. La pérdida `categorical_crossentropy` compara ese
vector con el one-hot real, penalizando la probabilidad asignada a la clase verdadera.

> Existe la alternativa de dejar `y` como enteros y usar `sparse_categorical_crossentropy`, que es
> equivalente y ahorra memoria. Aquí se usa one-hot explícito porque hace visible la correspondencia con
> las 64 neuronas de salida, que es justamente lo que interesa mostrar en este trabajo.


In [ ]:
NUMERO_CLASES = len(clases)

# One-hot: la fila i de la matriz identidad es el vector con un 1 en la posición i
# Equivale a keras.utils.to_categorical(y, NUMERO_CLASES), sin cargar TensorFlow todavía
y_onehot = np.eye(NUMERO_CLASES, dtype=np.float32)[y]

print(f"Numero de clases: {NUMERO_CLASES}")
print(f"y (indices):  {y.shape}  dtype={y.dtype}")
print(f"y_onehot:     {y_onehot.shape}  dtype={y_onehot.dtype}")

# Correspondencia para una imagen concreta
posicion_activa = int(y_onehot[indice_prueba].argmax())
print(
    f"\nEjemplo (imagen {indice_prueba}): '{datos.iloc[indice_prueba]['personaje']}'"
    f" -> indice {y[indice_prueba]} -> posicion activa {posicion_activa}"
)
print(f"vector one-hot: {y_onehot[indice_prueba].astype(int)}")


**Comprobación de la codificación**

In [ ]:
comprobaciones_etiquetas = pd.DataFrame(
    [
        (
            "Número de ejemplos",
            y_onehot.shape[0],
            X.shape[0],
            y_onehot.shape[0] == X.shape[0],
        ),
        (
            "Columnas = número de clases",
            y_onehot.shape[1],
            NUMERO_CLASES,
            y_onehot.shape[1] == NUMERO_CLASES,
        ),
        ("Tipo de dato", str(y_onehot.dtype), "float32", y_onehot.dtype == np.float32),
        (
            "Un único 1 por fila",
            float(y_onehot.sum(axis=1).min()),
            1.0,
            bool(np.all(y_onehot.sum(axis=1) == 1.0)),
        ),
        (
            "Solo valores 0 o 1",
            sorted(np.unique(y_onehot).tolist()),
            [0.0, 1.0],
            bool(np.isin(np.unique(y_onehot), [0.0, 1.0]).all()),
        ),
        (
            "Coincide con el índice entero",
            "y_onehot.argmax(axis=1)",
            "y",
            bool(np.array_equal(y_onehot.argmax(axis=1), y)),
        ),
        (
            "Todas las clases presentes",
            int((y_onehot.sum(axis=0) > 0).sum()),
            NUMERO_CLASES,
            int((y_onehot.sum(axis=0) > 0).sum()) == NUMERO_CLASES,
        ),
        (
            "Mapeo estable (índice -> nombre)",
            indice_a_clase[0],
            clases[0],
            indice_a_clase[0] == clases[0],
        ),
    ],
    columns=["Comprobacion", "Obtenido", "Esperado", "Correcto"],
)

display(comprobaciones_etiquetas)

if not comprobaciones_etiquetas["Correcto"].all():
    raise ValueError("La codificación de etiquetas no cumple alguna de las comprobaciones.")

print("Codificación verificada correctamente.")


In [ ]:
# Diccionario de clases: índice, nombre y ejemplos disponibles
tabla_clases = pd.DataFrame(
    {
        "indice": range(NUMERO_CLASES),
        "personaje": clases,
        "imagenes": [int((y == indice).sum()) for indice in range(NUMERO_CLASES)],
    }
)
tabla_clases["porcentaje"] = (tabla_clases["imagenes"] / len(y) * 100).round(2)

display(tabla_clases)


### Escenas con varios personajes

La etiqueta de cada imagen es su carpeta, pero un fotograma puede mostrar a más de un personaje: Lou junto
a Wiggum, Smithers junto a Burns, Barney en el bar de Moe. Cuando el modelo responde con el otro personaje
de la escena, la métrica estricta lo cuenta como error aunque la respuesta sea razonable, y el
entrenamiento lo castiga igual que una confusión sin sentido.

Los **fotogramas de episodio** del dataset traen en su nombre el episodio y el instante
(`personaje__S02E15__t0675633.jpg`, con `t` en milisegundos). Eso permite estimar quién más aparece en la
escena sin anotar nada a mano:

- **Otro personaje en la misma escena.** Si la carpeta de otro personaje tiene un fotograma del mismo
  episodio a menos de `VENTANA_ESCENA_MS` (3 s), se asume que ese personaje también está en la escena. Es
  una heurística: tres segundos suelen ser el mismo plano, pero un corte de cámara puede caer en medio.
- **Nombre de archivo de otro personaje.** Los fotogramas que el equipo movió a la carpeta de su
  protagonista conservan el nombre del personaje con el que se extrajeron, que también está en el
  encuadre.
- **Gemela en la carpeta de otro personaje.** El script de alteraciones compara las imágenes de carpetas
  distintas y registra en `_gemelas.csv` las casi idénticas (similitud ≥ 0,90): es la misma escena
  guardada dos veces, así que los dos personajes están en ella. Esta fuente también alcanza a las
  imágenes de Kaggle. El script copia además esas imágenes a `gemelas_por_revisar/`, fuera del dataset,
  para que el equipo decida a mano si las suma también a la carpeta del otro personaje.

Las imágenes que vienen del dataset de Kaggle (`homer_simpson_0001.jpg`) no traen episodio ni instante:
solo tienen acompañantes si tienen una gemela; si no, conservan su etiqueta única.

**Cómo se usa.**

1. **Al entrenar.** La etiqueta de una imagen con escena compartida deja de ser un one-hot puro: el
   personaje de la carpeta conserva el `1 - PESO_ESCENA` (85 %) y el resto se reparte entre los
   acompañantes. El modelo sigue aprendiendo que el principal es el de la carpeta, pero nombrar al
   acompañante deja de castigarse como un error grave. El experimento S2 mide su efecto.
2. **Al evaluar.** Una predicción que nombra a un personaje presente cuenta como **acierto** en todas las
   métricas de prueba y en los KPIs. La accuracy estricta, que solo acepta el personaje de la carpeta, se
   informa como referencia. Los errores cometidos con confianza mayor que 0,80 pasan además por una
   verificación por imagen (sección «Verificación de los errores con confianza alta»).

**Sin fuga de información.** La escena que se usa al entrenar se calcula **solo con los fotogramas de
entrenamiento**. Si se usaran todos, un fotograma de test casi idéntico a uno de entrenamiento le pasaría
su etiqueta al modelo a través del objetivo de entrenamiento. Por lo mismo, al entrenar solo cuentan las gemelas que
también están en entrenamiento. La escena de la evaluación sí usa todos los fotogramas y todas las
gemelas, porque solo se lee después de predecir y no influye en el modelo. La validación, que decide la
parada temprana y el modelo final, se mide siempre con la etiqueta de la carpeta.


In [ ]:
import re

# Fotogramas de episodio: <personaje>__<episodio>__t<milisegundos>
PATRON_FOTOGRAMA = re.compile(r"^(?P<nombre>.+?)__(?P<episodio>S\d+E\d+)__t(?P<ms>\d+)$", re.IGNORECASE)
VENTANA_ESCENA_MS = 3000
PESO_ESCENA = 0.15


def leer_fotograma(nombre_archivo):
    """(personaje del nombre, episodio, ms) de un fotograma de episodio, o None si no lo es."""
    coincidencia = PATRON_FOTOGRAMA.match(Path(nombre_archivo).stem)
    if coincidencia is None:
        return None
    return coincidencia["nombre"], coincidencia["episodio"].upper(), int(coincidencia["ms"])


def construir_lineas_tiempo(tabla):
    """Fotogramas de cada episodio ordenados por instante: {episodio: (instantes, personajes)}."""
    por_episodio = {}
    for personaje, origen in zip(tabla["personaje"], tabla["origen"]):
        fotograma = leer_fotograma(origen)
        if fotograma is not None:
            por_episodio.setdefault(fotograma[1], []).append((fotograma[2], personaje))
    lineas = {}
    for episodio, fotogramas in por_episodio.items():
        fotogramas.sort()
        lineas[episodio] = (np.array([ms for ms, _ in fotogramas]), [p for _, p in fotogramas])
    return lineas


def personajes_en_escena(personaje, origen, lineas_tiempo):
    """Personajes que probablemente acompañan al de la carpeta en esa imagen (sin incluirlo)."""
    fotograma = leer_fotograma(origen)
    if fotograma is None:
        return frozenset()
    nombre, episodio, ms = fotograma
    acompanantes = {nombre} if nombre != personaje and nombre in clase_a_indice else set()
    if episodio in lineas_tiempo:
        instantes, personajes = lineas_tiempo[episodio]
        desde = np.searchsorted(instantes, ms - VENTANA_ESCENA_MS, side="left")
        hasta = np.searchsorted(instantes, ms + VENTANA_ESCENA_MS, side="right")
        acompanantes.update(otro for otro in personajes[desde:hasta] if otro != personaje)
    return frozenset(acompanantes)


# Gemelas entre personajes: la misma escena guardada en la carpeta de otro personaje. Las detecta el
# script de alteraciones comparando las originales de carpetas distintas
registro_gemelas = DATASET_BASE / "_gemelas.csv"
GEMELAS = {}
n_pares_gemelas = 0
if registro_gemelas.exists():
    pares_gemelas = pd.read_csv(registro_gemelas)
    pares_gemelas = pares_gemelas[
        pares_gemelas["personaje"].isin(clase_a_indice) & pares_gemelas["otro_personaje"].isin(clase_a_indice)
    ]
    n_pares_gemelas = len(pares_gemelas)
    for fila in pares_gemelas.itertuples():
        GEMELAS.setdefault((fila.personaje, fila.original), set()).add((fila.otro_personaje, fila.otro_original))
        GEMELAS.setdefault((fila.otro_personaje, fila.otro_original), set()).add((fila.personaje, fila.original))
else:
    print("No hay registro de gemelas entre personajes: ejecuta antes el script de alteraciones.")


def personajes_gemelos(personaje, origen, permitidas=None):
    """Personajes en cuya carpeta hay una imagen casi idéntica a esta. Con `permitidas`, solo cuentan
    las gemelas que están en ese conjunto de (personaje, original)."""
    return frozenset(
        otro
        for otro, otro_origen in GEMELAS.get((personaje, origen), ())
        if permitidas is None or (otro, otro_origen) in permitidas
    )


# Línea de tiempo para EVALUAR: todos los fotogramas originales del dataset
originales_indexados = indice_completo[
    ~indice_completo["alterada"] & indice_completo["personaje"].isin(clase_a_indice)
]
LINEAS_TIEMPO_COMPLETAS = construir_lineas_tiempo(originales_indexados)
# Línea de tiempo para ENTRENAR: solo los fotogramas originales de entrenamiento
LINEAS_TIEMPO_TRAIN = construir_lineas_tiempo(datos[(datos["conjunto"] == "train") & ~datos["alterada"]])

es_train = (datos["conjunto"] == "train").to_numpy()
# Para ENTRENAR solo cuentan las gemelas que son originales de entrenamiento
originales_train = set(
    zip(datos.loc[es_train & ~datos["alterada"].to_numpy(), "personaje"],
        datos.loc[es_train & ~datos["alterada"].to_numpy(), "origen"])
)
escena_evaluacion = [
    personajes_en_escena(p, o, LINEAS_TIEMPO_COMPLETAS) | personajes_gemelos(p, o)
    for p, o in zip(datos["personaje"], datos["origen"])
]
escena_entrenamiento = [
    personajes_en_escena(p, o, LINEAS_TIEMPO_TRAIN) | personajes_gemelos(p, o, originales_train)
    if en_train
    else frozenset()
    for p, o, en_train in zip(datos["personaje"], datos["origen"], es_train)
]
datos["escena"] = [", ".join(sorted(acompanantes)) for acompanantes in escena_evaluacion]

# Objetivo de entrenamiento con escena: el personaje de la carpeta conserva 1 - PESO_ESCENA
# y el resto se reparte entre sus acompañantes. Las filas de validación y test no cambian.
y_escena = y_onehot.copy()
for fila, acompanantes in enumerate(escena_entrenamiento):
    if acompanantes:
        y_escena[fila] *= 1 - PESO_ESCENA
        for otro in acompanantes:
            y_escena[fila, clase_a_indice[otro]] += PESO_ESCENA / len(acompanantes)

es_fotograma = np.array([leer_fotograma(origen) is not None for origen in datos["origen"]])
con_gemela = np.array([bool(personajes_gemelos(p, o)) for p, o in zip(datos["personaje"], datos["origen"])])
con_escena_evaluacion = np.array([bool(acompanantes) for acompanantes in escena_evaluacion])
con_escena_entrenamiento = np.array([bool(acompanantes) for acompanantes in escena_entrenamiento])

resumen_escena = pd.DataFrame(
    [
        (
            nombre,
            int(mascara.sum()),
            int(es_fotograma[mascara].sum()),
            int(con_gemela[mascara].sum()),
            int(con_escena_evaluacion[mascara].sum()),
            int(con_escena_entrenamiento[mascara].sum()) if nombre == "train" else "-",
        )
        for nombre in CONJUNTOS
        for mascara in [(datos["conjunto"] == nombre).to_numpy()]
    ],
    columns=["conjunto", "imagenes", "fotogramas", "con_gemela", "escena_compartida", "etiqueta_suave_al_entrenar"],
)
display(resumen_escena)

# Parejas de personajes que más comparten escena (solo originales)
pares_escena = (
    pd.DataFrame(
        [
            (personaje, otro)
            for personaje, acompanantes, alterada in zip(datos["personaje"], escena_evaluacion, datos["alterada"])
            if not alterada
            for otro in acompanantes
        ],
        columns=["personaje", "acompanante"],
    )
    .value_counts()
    .head(10)
    .reset_index(name="imagenes")
)
display(pares_escena)

comprobaciones_escena = pd.DataFrame(
    [
        (
            "Cada objetivo suma 1",
            round(float(y_escena.sum(axis=1).min()), 4),
            1.0,
            bool(np.allclose(y_escena.sum(axis=1), 1.0)),
        ),
        (
            "La carpeta sigue siendo la clase principal",
            "y_escena.argmax(axis=1)",
            "y",
            bool(np.array_equal(y_escena.argmax(axis=1), y)),
        ),
        (
            "Validación y test sin etiqueta suave",
            int((y_escena[~es_train] != y_onehot[~es_train]).any(axis=1).sum()),
            0,
            bool(np.array_equal(y_escena[~es_train], y_onehot[~es_train])),
        ),
        (
            "Ninguna imagen se acompaña a sí misma",
            "personaje fuera de su escena",
            True,
            all(p not in a for p, a in zip(datos["personaje"], escena_evaluacion)),
        ),
    ],
    columns=["Comprobacion", "Obtenido", "Esperado", "Correcto"],
)
display(comprobaciones_escena)

if not comprobaciones_escena["Correcto"].all():
    raise ValueError("Las etiquetas de escena no cumplen alguna de las comprobaciones.")

print(
    f"Fotogramas de episodio: {int(es_fotograma.sum())} de {len(datos)} imagenes"
    f"   |   con gemela en otra carpeta: {int(con_gemela.sum())} ({n_pares_gemelas} pares)"
    f"   |   con escena compartida: {int(con_escena_evaluacion.sum())}"
    f"   |   con etiqueta suave al entrenar: {int(con_escena_entrenamiento.sum())}"
)


### Partición train / validación / test

**Por qué hacen falta tres conjuntos y no dos**

Entrenar mirando el conjunto de prueba es hacer trampa: en cuanto se usa para decidir cuántas capas
poner, cuándo detener el entrenamiento o qué representación elegir, deja de ser una estimación honesta
del desempeño con datos nuevos, porque esas decisiones ya lo incorporaron. Ese fenómeno se conoce como
fuga de información.

Por eso cada personaje se repartió, en «Partición y selección por personaje», en tres conjuntos:

- **Entrenamiento (70 % de las originales):** las imágenes con las que se ajustan los pesos.
- **Validación (20 % de las originales):** imágenes que nunca actualizan pesos, pero que se miden en
  cada época. Sirven para trazar las curvas de aprendizaje, detectar sobreajuste, activar la parada
  temprana y comparar las configuraciones de la experimentación.
- **Test (10 % de las originales):** se reserva intacto y se abre **una sola vez**, en la evaluación
  final del modelo elegido.

Cada conjunto lleva además las variantes alteradas de sus propias originales y la versión filtrada de las
casi idénticas, así que los tres tienen las mismas condiciones.

**Partición estratificada.** Validación y test conservan, para cada personaje, la misma proporción que
tiene en el dataset completo. Con 64 clases muy desbalanceadas, un reparto aleatorio simple podría dejar
a un personaje minoritario con muy pocos ejemplos de validación —o con ninguno—, y entonces su métrica
sería pura casualidad.

**Semilla fija.** `random_state=SEMILLA` garantiza que la partición sea siempre la misma. Es un
requisito de la metodología de experimentación: al cambiar un solo factor por vez, la diferencia
observada debe deberse a ese factor y no a que le tocó un reparto de datos distinto.

La celda siguiente solo materializa esa partición: separa las filas de cada matriz según el conjunto
que se asignó a cada imagen.

In [ ]:
# La partición ya está decidida en la columna "conjunto" de datos: aquí solo se
# separan las imágenes de cada conjunto
MASCARAS = {nombre: (datos["conjunto"] == nombre).to_numpy() for nombre in CONJUNTOS}
PARTICION = {nombre: X[mascara] for nombre, mascara in MASCARAS.items()}

# Etiquetas e imágenes de cada conjunto (las de test se usan en el análisis de errores).
# Entrenamiento usa el objetivo con escena; validación y test, el one-hot estricto.
y_train, y_val, y_test = (y_onehot[MASCARAS[nombre]] for nombre in CONJUNTOS)
y_train_duro = y_train
y_train = y_escena[MASCARAS["train"]]
y_train_idx, y_val_idx, y_test_idx = (y[MASCARAS[nombre]] for nombre in CONJUNTOS)
imagenes_test = PARTICION["test"]
datos_test = datos[MASCARAS["test"]].reset_index(drop=True)

# Tensores de imagen de cada conjunto
X_train, X_val, X_test = (PARTICION[nombre] for nombre in CONJUNTOS)

print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")
# Las proporciones se miden sobre las originales: las alteradas son variantes extra que van
# al mismo conjunto que su original
es_alterada = datos["alterada"].to_numpy()
n_originales = {nombre: int((MASCARAS[nombre] & ~es_alterada).sum()) for nombre in CONJUNTOS}
n_alteradas = {nombre: int((MASCARAS[nombre] & es_alterada).sum()) for nombre in CONJUNTOS}
total = sum(n_originales.values())
print(
    "\nReparto de las originales: "
    + " / ".join(f"{n_originales[nombre]} {nombre} ({n_originales[nombre] / total * 100:.1f} %)" for nombre in CONJUNTOS)
)
for nombre in CONJUNTOS:
    print(
        f"{nombre:<5}: {n_originales[nombre]} originales + {n_alteradas[nombre]} alteradas"
        f" = {n_originales[nombre] + n_alteradas[nombre]} imagenes"
    )
print(f"Forma de cada imagen: {X_train.shape[1:]} ({DIMENSION_PIXELES} valores, sin aplanar)")

**Comprobación de la partición**

In [ ]:
# Proporción de cada personaje en el dataset completo frente a validación y test
proporcion_dataset = originales["personaje"].value_counts(normalize=True)
proporciones = pd.DataFrame(
    {
        "personaje": clases,
        "dataset_%": [round(float(proporcion_dataset[clase]) * 100, 2) for clase in clases],
        "val_%": [round((y_val_idx == i).mean() * 100, 2) for i in range(NUMERO_CLASES)],
        "test_%": [round((y_test_idx == i).mean() * 100, 2) for i in range(NUMERO_CLASES)],
        "n_train": [int((y_train_idx == i).sum()) for i in range(NUMERO_CLASES)],
        "n_val": [int((y_val_idx == i).sum()) for i in range(NUMERO_CLASES)],
        "n_test": [int((y_test_idx == i).sum()) for i in range(NUMERO_CLASES)],
    }
)
proporciones["desviacion_%"] = (
    proporciones[["val_%", "test_%"]].sub(proporciones["dataset_%"], axis=0).abs().max(axis=1).round(2)
)

display(proporciones)

# Una imagen y sus variantes se identifican por (personaje, original de la que salen)
origenes = {
    nombre: set(zip(datos.loc[MASCARAS[nombre], "personaje"], datos.loc[MASCARAS[nombre], "origen"]))
    for nombre in CONJUNTOS
}
compartidas = (
    len(origenes["train"] & origenes["val"])
    + len(origenes["train"] & origenes["test"])
    + len(origenes["val"] & origenes["test"])
)
# Mismas condiciones en los tres conjuntos: cuántas alteradas y parecidas hay por cada original
TOLERANCIA_CONDICIONES = 0.03


def por_original(columna):
    return {
        nombre: float(datos.loc[MASCARAS[nombre], columna].sum())
        / max(int((MASCARAS[nombre] & ~datos["alterada"].to_numpy()).sum()), 1)
        for nombre in CONJUNTOS
    }


alteradas_por_original = por_original("alterada")
parecidas_por_original = por_original("parecida")
proporciones_obtenidas = {
    nombre: (MASCARAS[nombre] & ~datos["alterada"].to_numpy()).sum() / (~datos["alterada"]).sum()
    for nombre in CONJUNTOS
}
clases_por_conjunto = [len(np.unique(indices)) for indices in (y_train_idx, y_val_idx, y_test_idx)]

comprobaciones_particion = pd.DataFrame(
    [
        (
            "Cada imagen en un solo conjunto",
            int(sum(mascara.sum() for mascara in MASCARAS.values())),
            len(datos),
            int(sum(mascara.sum() for mascara in MASCARAS.values())) == len(datos),
        ),
        ("Sin imágenes compartidas entre conjuntos", compartidas, 0, compartidas == 0),
        *[
            (
                f"{nombre} por original en train / val / test",
                " / ".join(f"{proporciones_tipo[conjunto]:.1%}" for conjunto in CONJUNTOS),
                f"iguales (± {TOLERANCIA_CONDICIONES:.0%})",
                all(
                    abs(proporciones_tipo[conjunto] - proporciones_tipo["train"]) <= TOLERANCIA_CONDICIONES
                    for conjunto in CONJUNTOS
                ),
            )
            for nombre, proporciones_tipo in (
                ("Alteradas", alteradas_por_original),
                ("Parecidas", parecidas_por_original),
            )
        ],
        (
            "Todas las originales repartidas (sin tope)",
            int((~datos["alterada"]).sum()),
            len(originales),
            int((~datos["alterada"]).sum()) == len(originales),
        ),
        (
            "Proporciones train / val / test de las originales",
            " / ".join(f"{proporciones_obtenidas[nombre]:.1%}" for nombre in CONJUNTOS),
            " / ".join(f"{PROPORCIONES[nombre]:.0%}" for nombre in CONJUNTOS),
            all(abs(proporciones_obtenidas[nombre] - PROPORCIONES[nombre]) < 0.005 for nombre in CONJUNTOS),
        ),
        (
            "Forma de las imágenes intacta",
            str(X_train.shape[1:]),
            str(FORMA_ENTRADA),
            X_train.shape[1:] == FORMA_ENTRADA,
        ),
        ("X e y alineados en train", len(y_train), len(X_train), len(y_train) == len(X_train)),
        ("X e y alineados en validación", len(y_val), len(X_val), len(y_val) == len(X_val)),
        ("X e y alineados en test", len(y_test), len(X_test), len(y_test) == len(X_test)),
        (
            "One-hot coherente con índices",
            "y_val.argmax(axis=1)",
            "y_val_idx",
            bool(np.array_equal(np.argmax(y_val, axis=1), y_val_idx)),
        ),
        (
            "Todas las clases en train / val / test",
            " / ".join(str(n) for n in clases_por_conjunto),
            NUMERO_CLASES,
            all(n == NUMERO_CLASES for n in clases_por_conjunto),
        ),
        (
            "Estratificación de val y test (desviación máxima)",
            f"{proporciones['desviacion_%'].max():.2f} %",
            "< 1 %",
            bool(proporciones["desviacion_%"].max() < 1.0),
        ),
    ],
    columns=["Comprobacion", "Obtenido", "Esperado", "Correcto"],
)

display(comprobaciones_particion)

if not comprobaciones_particion["Correcto"].all():
    raise ValueError("La partición no cumple alguna de las comprobaciones.")

print("Partición verificada correctamente.")

### Tratamiento del desbalance de clases

**El problema**

Como se describió en "Definición del problema de clasificación", el dataset está muy desbalanceado
(razón aproximada 33:1 entre `homer_simpson` y `louie`). Esto sesga el entrenamiento: la pérdida es un
promedio sobre los ejemplos del batch, así que los personajes abundantes dominan el gradiente y el
modelo aprende que, ante la duda, conviene predecir una clase mayoritaria. El resultado típico es un
accuracy global aceptable acompañado de un recall muy bajo en las clases pequeñas, que es precisamente
lo que el **F1 macro** —promedio no ponderado entre clases— deja al descubierto.

**Opciones consideradas**

| Estrategia | Qué hace | Por qué se descarta o se elige |
|---|---|---|
| Submuestreo de mayoritarias | Limita cada personaje a un tope de originales en entrenamiento | Se descarta: una corrida anterior con un tope de 300 dejaba sin usar unas 12.700 imágenes, y los personajes frecuentes, recortados en entrenamiento pero completos en validación y test, caían por debajo de un 20 % de recall (Homero 0,16, Bart 0,15) |
| Sobremuestreo por duplicado | Repite imágenes de las clases pequeñas | Se descarta: los duplicados exactos favorecen el sobreajuste a esas imágenes |
| **Aumento de datos** | Añade variantes alteradas con un giro de 10 a 20°, un cambio de color y un filtro de textura | **Se elige:** las genera una sola vez el script `generar_imagenes_alteradas.py` para un cuarto de las imágenes de cada personaje. Cada una va al mismo conjunto que su original: en entrenamiento aumenta los datos, y en validación y prueba hace que el modelo se mida en las mismas condiciones |
| **Pesos por clase suavizados** | Pondera la pérdida según la frecuencia de cada clase | **Se elige:** compensa el desbalance sin descartar datos. Se usa la raíz cuadrada del peso balanceado: con 32:1 en entrenamiento, el balanceado completo daría a `louie` unas 32 veces el peso de Homero y empujaría al modelo a no predecir a los frecuentes. Los experimentos P1 y P2 comparan sin pesos y con los balanceados completos |

**Cómo funciona `class_weight`.** Keras multiplica la pérdida de cada ejemplo por el peso de su clase, de
modo que equivocarse en un personaje raro cuesta más que equivocarse en uno frecuente. Se usa la fórmula
`n_muestras / (n_clases * n_ejemplos_de_la_clase)`, que da peso 1 a una clase de tamaño medio, mayor que 1
a las minoritarias y menor que 1 a las mayoritarias. El modelo base usa su **raíz cuadrada**, reescalada
para que el peso medio por imagen siga siendo 1: la pérdida conserva su escala y la razón entre el peso
mayor y el menor baja de unas 32 a unas 6 veces.

**Advertencia importante.** Los pesos se calculan **solo con `y_train`**, nunca con el conjunto completo
ni con validación: usar información de validación para ajustar el entrenamiento sería otra forma de fuga.
Y se aplican **solo durante el entrenamiento**: las métricas de validación y test se miden sin pesos,
porque deben reflejar el desempeño sobre la distribución real de los datos.

El efecto conjunto de estas tres decisiones se lee en la evaluación, comparando la accuracy con el F1
macro en test.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Los pesos se calculan SOLO con las etiquetas de entrenamiento
pesos_balanceados = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUMERO_CLASES),
    y=y_train_idx,
)
conteos_train = np.bincount(y_train_idx, minlength=NUMERO_CLASES)

# Pesos suavizados: raíz cuadrada del balanceado, reescalada para que el peso medio por
# imagen de entrenamiento sea 1 y la pérdida conserve su escala
pesos_raiz = np.sqrt(pesos_balanceados)
pesos_raiz = pesos_raiz * len(y_train_idx) / (pesos_raiz * conteos_train).sum()

# Esquemas que compara la experimentación (P1 y P2); el modelo base usa ESQUEMA_PESOS.
# Keras espera un diccionario {indice_de_clase: peso}
ESQUEMA_PESOS = "raiz"
PESOS_POR_ESQUEMA = {
    "ninguno": None,
    "raiz": {indice: float(peso) for indice, peso in enumerate(pesos_raiz)},
    "balanceado": {indice: float(peso) for indice, peso in enumerate(pesos_balanceados)},
}
pesos = pesos_raiz
pesos_por_clase = PESOS_POR_ESQUEMA[ESQUEMA_PESOS]

tabla_pesos = pd.DataFrame(
    {
        "indice": range(NUMERO_CLASES),
        "personaje": clases,
        "n_train": conteos_train,
        "porcentaje": (conteos_train / conteos_train.sum() * 100).round(2),
        "peso_balanceado": np.round(pesos_balanceados, 3),
        "peso": np.round(pesos, 3),
    }
).sort_values("n_train", ascending=False, ignore_index=True)

display(tabla_pesos)

razon_desbalance = conteos_train.max() / conteos_train.min()
print(f"Clase mayoritaria: {clases[int(conteos_train.argmax())]} ({conteos_train.max()} imagenes)")
print(f"Clase minoritaria: {clases[int(conteos_train.argmin())]} ({conteos_train.min()} imagenes)")
print(f"Razon de desbalance: {razon_desbalance:.1f}:1")
print(f"Rango de pesos (raiz, los del modelo base): [{pesos.min():.3f}, {pesos.max():.3f}]")
print(f"Rango de pesos balanceados completos      : [{pesos_balanceados.min():.3f}, {pesos_balanceados.max():.3f}]")


**Comprobación de los pesos y baselines de referencia**

In [ ]:
comprobaciones_pesos = pd.DataFrame(
    [
        (
            "Un peso por clase",
            len(pesos_por_clase),
            NUMERO_CLASES,
            len(pesos_por_clase) == NUMERO_CLASES,
        ),
        (
            "Todos los pesos positivos",
            round(float(pesos.min()), 3),
            "> 0",
            bool((pesos > 0).all()),
        ),
        (
            "Mayor peso a la clase minoritaria",
            clases[int(pesos.argmax())],
            clases[int(conteos_train.argmin())],
            int(pesos.argmax()) == int(conteos_train.argmin()),
        ),
        (
            "Menor peso a la clase mayoritaria",
            clases[int(pesos.argmin())],
            clases[int(conteos_train.argmax())],
            int(pesos.argmin()) == int(conteos_train.argmax()),
        ),
        (
            "Peso inverso a la frecuencia",
            "corr(peso, n_train)",
            "negativa",
            bool(float(np.corrcoef(pesos, conteos_train)[0, 1]) < 0),
        ),
        (
            "Calculados solo con train",
            len(y_train_idx),
            len(X_train),
            len(y_train_idx) == len(X_train),
        ),
    ],
    columns=["Comprobacion", "Obtenido", "Esperado", "Correcto"],
)

display(comprobaciones_pesos)

if not comprobaciones_pesos["Correcto"].all():
    raise ValueError("Los pesos por clase no cumplen alguna de las comprobaciones.")

# Baselines contra los que se comparará el modelo final. La clase mayoritaria se mide
# sobre validación, que conserva la distribución real del dataset (sus alteradas son un
# cuarto de las originales de cada personaje, igual que en train).
conteos_val = np.bincount(y_val_idx, minlength=NUMERO_CLASES)
baseline_azar = 100 / NUMERO_CLASES
baseline_mayoritaria = conteos_val.max() / conteos_val.sum() * 100

print(f"\nBaseline por azar (1/{NUMERO_CLASES}): {baseline_azar:.1f} % de accuracy")
print(f"Baseline de clase mayoritaria:  {baseline_mayoritaria:.1f} % de accuracy")
print("El modelo solo aporta valor si supera claramente ambos.")


In [ ]:
import matplotlib

# Distribución de imágenes por personaje en el conjunto de entrenamiento
figura, eje = plt.subplots(figsize=(10, 14))

orden = np.argsort(conteos_train)
eje.barh(
    [clases[i] for i in orden],
    conteos_train[orden],
    color=matplotlib.colormaps["viridis"](pesos[orden] / pesos.max()),
)
eje.set_xlabel("Número de imágenes en entrenamiento")
eje.set_title("Desbalance de clases (color = peso asignado; más claro = mayor peso)", fontsize=10)
eje.tick_params(axis="y", labelsize=7)
eje.axvline(
    conteos_train.mean(),
    color="red",
    linestyle="--",
    linewidth=1,
    label=f"media = {conteos_train.mean():.0f}",
)
eje.legend()

plt.tight_layout()
plt.show()


## Fundamentos: estructura de una red convolucional

### Capas convolucionales: filtros que recorren la imagen

**Qué hace una capa convolucional**

Una neurona de una capa densa mira **todas** las entradas a la vez, cada una con su propio peso. Una neurona
convolucional mira solo una ventana pequeña —en esta CNN, de 3 × 3 píxeles con sus tres canales— y calcula
lo mismo que cualquier neurona: una combinación lineal de esos valores más un sesgo, seguida de una
activación. Lo que la distingue es que **la misma ventana de pesos, el filtro, se desplaza por toda la
imagen**:

$$z_{f,c} = \sum_{i=-1}^{1}\sum_{j=-1}^{1}\sum_{k=1}^{3} w_{i,j,k}\, x_{f+i,\,c+j,\,k} + b$$

El resultado de aplicar un filtro en todas las posiciones es un **mapa de activación**: una imagen nueva
que vale mucho donde la ventana se parece al patrón del filtro y poco donde no. Una capa con 32 filtros
produce 32 mapas, uno por patrón, y la capa siguiente combina esos mapas en patrones más complejos.

**Tres consecuencias de compartir los pesos**

| Propiedad | Qué significa | Qué limitación del MLP resuelve |
|---|---|---|
| **Conectividad local** | Cada neurona mira una vecindad, no la imagen entera | La red sabe qué píxel está junto a cuál: la estructura espacial no se pierde |
| **Pesos compartidos** | El mismo filtro se usa en todas las posiciones | Un patrón aprendido en una zona se reconoce en cualquier otra: el peinado de Marge es el mismo arriba o al centro del encuadre |
| **Pocos parámetros** | Un filtro de 3 × 3 × 3 tiene 27 pesos, a cualquier resolución | La primera capa del MLP sobre píxeles tenía 6,3 millones de pesos a 64 × 64; la de la CNN tiene 864 |

En el MLP los filtros los elegíamos nosotros: los histogramas de color, LBP, Gabor y HOG eran, en el fondo,
filtros fijos. En la CNN **los filtros son pesos** y se aprenden con el mismo descenso del gradiente que el
resto de la red. La celda siguiente aplica a mano tres filtros clásicos sobre una imagen real para ver qué
produce una convolución.

In [ ]:
# Una convolución hecha a mano: tres filtros clásicos de 3 x 3 sobre una imagen de entrenamiento
from scipy.signal import correlate2d

indice_demo = int(np.flatnonzero(MASCARAS["train"])[0])
imagen_demo = imagenes[indice_demo].astype(np.float32) / VALOR_MAXIMO_PIXEL
# Los filtros de esta demostración trabajan sobre un solo canal: la luminancia de la imagen
gris_demo = imagen_demo @ np.array([0.299, 0.587, 0.114], dtype=np.float32)

FILTROS_DEMO = {
    "Bordes verticales (Sobel)": np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=np.float32),
    "Bordes horizontales (Sobel)": np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], dtype=np.float32),
    "Desenfoque (media 3 x 3)": np.full((3, 3), 1 / 9, dtype=np.float32),
}
# Las capas Conv2D calculan una correlación cruzada (el filtro no se invierte), que es lo que hace
# correlate2d. Con mode="same" el mapa conserva el tamaño de la imagen, como padding="same" en Keras.
mapas_demo = {
    nombre: correlate2d(gris_demo, filtro, mode="same", boundary="symm")
    for nombre, filtro in FILTROS_DEMO.items()
}

figura, ejes = plt.subplots(2, 4, figsize=(13, 7.0), layout="constrained")
ejes[0, 0].imshow(imagenes[indice_demo])
ejes[0, 0].set_title("Imagen RGB", fontsize=9)
ejes[1, 0].imshow(gris_demo, cmap="gray", vmin=0, vmax=1)
ejes[1, 0].set_title("Luminancia: entrada de los filtros", fontsize=9)

for columna, (nombre, filtro) in enumerate(FILTROS_DEMO.items(), start=1):
    eje_filtro, eje_mapa = ejes[0, columna], ejes[1, columna]
    eje_filtro.imshow(filtro, cmap="RdBu_r", vmin=-2, vmax=2)
    for fila_filtro in range(3):
        for columna_filtro in range(3):
            eje_filtro.text(
                columna_filtro, fila_filtro, f"{filtro[fila_filtro, columna_filtro]:.2g}",
                ha="center", va="center", fontsize=10,
            )
    eje_filtro.set_title(f"Filtro: {nombre}", fontsize=9)
    mapa = mapas_demo[nombre]
    limite = float(np.abs(mapa).max()) or 1.0
    if nombre.startswith("Desenfoque"):
        eje_mapa.imshow(mapa, cmap="gray", vmin=0, vmax=1)
    else:
        eje_mapa.imshow(mapa, cmap="RdBu_r", vmin=-limite, vmax=limite)
    eje_mapa.set_title(f"Mapa de activación {mapa.shape[0]}x{mapa.shape[1]}", fontsize=9)

for eje in np.ravel(ejes):
    eje.set_xticks([])
    eje.set_yticks([])

personaje = datos.iloc[indice_demo]["personaje"].replace("_", " ")
figura.suptitle(f"Una convolución a mano sobre {personaje}: filtro (arriba) y mapa que produce (abajo)",
                fontsize=12, fontweight="bold")
plt.show()

recorrido_convolucion = pd.DataFrame(
    [
        ("Entrada (luminancia)", str(gris_demo.shape), gris_demo.size),
        ("Filtro", "(3, 3)", 9),
        ("Mapa de activación (padding same)", str(mapas_demo["Bordes verticales (Sobel)"].shape),
         mapas_demo["Bordes verticales (Sobel)"].size),
    ],
    columns=["Paso", "Forma", "Número de valores"],
)
display(recorrido_convolucion)
print(
    "Con los mismos 9 pesos el filtro recorre las "
    f"{gris_demo.size:,} posiciones de la imagen: el mapa vale mucho (rojo o azul) donde la vecindad se"
    "\nparece al patrón del filtro. En la CNN esos 9 pesos (27 con los tres canales) no se eligen: se aprenden."
)

### Pooling, profundidad y campo receptivo

**Max pooling.** Después de cada bloque convolucional, una capa `MaxPooling2D` de 2 × 2 se queda con el valor
máximo de cada ventana de 2 × 2 del mapa de activación. El mapa queda con la mitad de filas y de columnas
—un cuarto de los valores— y conserva lo importante: si el patrón del filtro apareció en esa zona, aunque no
exactamente en qué píxel. Eso le da a la red una **invarianza aproximada a desplazamientos pequeños** y
abarata las capas siguientes.

**Campo receptivo.** Una neurona de la primera capa ve 3 × 3 píxeles. Una de la segunda ve 3 × 3 valores de
un mapa que ya resume ventanas de la imagen, así que en la imagen original abarca una zona mayor. Cada
bloque convolución + pooling amplía esa zona: las primeras capas detectan bordes y manchas de color, y las
últimas, partes completas —un ojo, el pelo, una camisa— porque ven una porción grande del encuadre. La
celda siguiente calcula el campo receptivo de cada bloque de la CNN base.

**Pooling global.** Al final de los bloques, `GlobalAveragePooling2D` promedia cada mapa de activación en
toda su extensión: de cada filtro queda un número, cuánto se activó en la imagen, sin importar dónde. Es la
entrada de la cabeza clasificadora, una pequeña red densa como el MLP.

In [ ]:
# Max pooling de 2 x 2 sobre el mapa de bordes verticales: la mitad de filas y de columnas
mapa_bordes = np.abs(mapas_demo["Bordes verticales (Sobel)"])
alto_par, ancho_par = mapa_bordes.shape[0] // 2 * 2, mapa_bordes.shape[1] // 2 * 2
mapa_pooling = (
    mapa_bordes[:alto_par, :ancho_par]
    .reshape(alto_par // 2, 2, ancho_par // 2, 2)
    .max(axis=(1, 3))
)

figura, ejes = plt.subplots(1, 2, figsize=(8, 4.0), layout="constrained")
for eje, mapa, titulo in (
    (ejes[0], mapa_bordes, f"Mapa de bordes {mapa_bordes.shape[0]}x{mapa_bordes.shape[1]}"),
    (ejes[1], mapa_pooling, f"Tras max pooling 2x2: {mapa_pooling.shape[0]}x{mapa_pooling.shape[1]}"),
):
    eje.imshow(mapa, cmap="inferno")
    eje.set_title(titulo, fontsize=9)
    eje.set_xticks([])
    eje.set_yticks([])
figura.suptitle("El pooling conserva dónde hubo bordes con un cuarto de los valores", fontsize=11, fontweight="bold")
plt.show()

# Campo receptivo de la CNN base (sección «Definición de la arquitectura»): cuatro bloques de una
# convolución de 3 x 3 seguida de un max pooling de 2 x 2. Cada capa amplía la zona de la imagen
# original que ve una neurona en (lado del núcleo - 1) x (distancia en píxeles entre dos neuronas vecinas).
filtros_demo = (32, 64, 128, 256)
campo, salto, lado = 1, 1, ALTO_OBJETIVO
filas_campo = []
for bloque, filtros in enumerate(filtros_demo, start=1):
    campo += (3 - 1) * salto           # convolución 3 x 3, paso 1
    campo_convolucion = campo
    campo += (2 - 1) * salto           # max pooling 2 x 2, paso 2
    salto *= 2
    filas_campo.append(
        {
            "bloque": bloque,
            "filtros": filtros,
            "mapa_entrada": f"{lado}x{lado}",
            "mapa_salida": f"{lado // 2}x{lado // 2}",
            "campo_receptivo_px": f"{min(campo_convolucion, ALTO_OBJETIVO)}x{min(campo_convolucion, ALTO_OBJETIVO)}",
            "% del lado de la imagen": round(min(campo_convolucion / ALTO_OBJETIVO, 1) * 100),
        }
    )
    lado //= 2

display(pd.DataFrame(filas_campo))
print(
    f"El ultimo bloque ve zonas de {filas_campo[-1]['campo_receptivo_px']} pixeles de una imagen de "
    f"{ALTO_OBJETIVO}x{ANCHO_OBJETIVO}: sus filtros responden a partes grandes del personaje, no a bordes sueltos."
)

### Por qué una CNN y no un MLP: el costo en parámetros

El MLP sobre píxeles conectaba cada uno de los `alto x ancho x 3` valores con cada neurona de su primera
capa, así que sus pesos crecían con la resolución. Una capa convolucional tiene un número de pesos fijo
—filtros × 3 × 3 × canales de entrada— y lo que crece con la resolución es el número de posiciones en que
se aplica, es decir, el cómputo. La celda siguiente lo cuantifica para la primera capa de cada red.

In [ ]:
# Primera capa de cada red: una densa de 512 neuronas sobre los píxeles aplanados (el MLP del
# experimento R1 del PMC) frente a una convolución de 32 filtros de 3 x 3 (la primera de la CNN base)
resoluciones_demo = (32, 64, 128)
tabla_costo = pd.DataFrame(
    {
        "resolucion": [f"{lado}x{lado}" for lado in resoluciones_demo],
        "valores_por_imagen": [lado * lado * 3 for lado in resoluciones_demo],
        "pesos_densa_512 (MLP)": [lado * lado * 3 * 512 for lado in resoluciones_demo],
        "pesos_conv_32 (CNN)": [3 * 3 * 3 * 32 for _ in resoluciones_demo],
        "multiplicaciones_conv_por_imagen": [lado * lado * 3 * 3 * 3 * 32 for lado in resoluciones_demo],
    }
)
tabla_costo["razon_pesos_MLP_CNN"] = (
    tabla_costo["pesos_densa_512 (MLP)"] / tabla_costo["pesos_conv_32 (CNN)"]
).round(0).astype(int)

display(tabla_costo)

fila_trabajo = tabla_costo[tabla_costo["resolucion"] == f"{ALTO_OBJETIVO}x{ANCHO_OBJETIVO}"]
if not fila_trabajo.empty:
    fila_trabajo = fila_trabajo.iloc[0]
    print(
        f"A {fila_trabajo['resolucion']}, la primera capa del MLP sobre pixeles tiene "
        f"{fila_trabajo['pesos_densa_512 (MLP)']:,} pesos y la de la CNN {fila_trabajo['pesos_conv_32 (CNN)']:,}:"
        f" {fila_trabajo['razon_pesos_MLP_CNN']:,} veces menos."
    )
print(
    "Los pesos de la convolucion no cambian con la resolucion; las multiplicaciones, en cambio,"
    "\nse multiplican por 4 cada vez que se dobla el lado. Por eso la CNN cuesta mas tiempo, no mas memoria."
)

### Funciones de activación

**Por qué hace falta la no linealidad**

Si se quitaran las funciones de activación, cada capa sería solo una multiplicación de matrices y la red
entera colapsaría en una sola: la composición de funciones lineales es otra función lineal. Una red de
diez capas sin activaciones tendría exactamente el mismo poder expresivo que una regresión lineal, y sería
incapaz de separar clases que no se puedan dividir con un hiperplano. La activación no lineal es lo que
permite que apilar capas sirva de algo.

**Comparación de las activaciones para las capas ocultas**

| Función | Fórmula | Rango | Ventaja | Problema |
|---|---|---|---|---|
| **Sigmoide** | $1/(1+e^{-z})$ | (0, 1) | Interpretable como probabilidad | Se satura: para $\|z\|$ grande el gradiente tiende a 0 y el aprendizaje se detiene |
| **Tanh** | $(e^z-e^{-z})/(e^z+e^{-z})$ | (-1, 1) | Centrada en cero, converge mejor que la sigmoide | También se satura en los extremos |
| **ReLU** | $\max(0, z)$ | [0, inf) | Gradiente constante 1 para $z>0$: no se satura; muy barata de calcular; produce activaciones dispersas | "Neuronas muertas": si una unidad queda siempre en $z<0$, su gradiente es 0 y no vuelve a aprender |

**Elección: ReLU en las capas ocultas.** El problema de la saturación es decisivo. En una red profunda,
los gradientes se multiplican capa a capa al retropropagarse; si cada factor es menor que 1 -- como pasa
con sigmoide, cuya derivada máxima es 0,25 -- el gradiente se desvanece antes de llegar a las primeras
capas, que son justamente las que deben aprender de los píxeles. ReLU mantiene el gradiente en 1 en toda
su zona activa, y además su cálculo es una simple comparación, lo que acelera el entrenamiento. Es el
valor por defecto razonable para capas ocultas, y de ahí partimos.

**Elección: softmax en la salida.** La capa final no busca una no linealidad cualquiera, sino convertir 64
números sin escala definida en una distribución de probabilidad:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{64} e^{z_j}}$$

Sus dos propiedades son las que necesita un problema multiclase de etiqueta única: todas las salidas son
positivas y **suman exactamente 1**. Eso introduce competencia entre clases -- si sube la probabilidad de
un personaje, baja la de los demas -- que es correcto aquí, porque cada imagen tiene un único personaje.
Si el problema fuera multietiqueta habría que usar sigmoides independientes en la salida.


In [ ]:
z = np.linspace(-6, 6, 400)

activaciones = {
    "Sigmoide": (1 / (1 + np.exp(-z)), lambda s: s * (1 - s)),
    "Tanh": (np.tanh(z), lambda s: 1 - s ** 2),
    "ReLU": (np.maximum(0, z), lambda s: (z > 0).astype(float)),
}

figura, ejes = plt.subplots(1, 2, figsize=(11, 4))

for nombre, (valores, derivada) in activaciones.items():
    ejes[0].plot(z, valores, label=nombre, linewidth=2)
    ejes[1].plot(z, derivada(valores), label=nombre, linewidth=2)

ejes[0].set_title("Funciones de activación", fontsize=10)
ejes[1].set_title("Sus derivadas (lo que viaja en la retropropagación)", fontsize=10)

for eje in ejes:
    eje.axhline(0, color="gray", linewidth=0.6)
    eje.axvline(0, color="gray", linewidth=0.6)
    eje.set_xlabel("z (preactivación)")
    eje.grid(alpha=0.3)
    eje.legend()

plt.tight_layout()
plt.show()

print(
    "En el gráfico de la derecha se ve el problema de la saturación: las derivadas de sigmoide y tanh"
    "\ncaen a cero en los extremos (el gradiente se desvanece), mientras que la de ReLU vale 1 para"
    "\ntodo z > 0. Nótese además que la derivada de la sigmoide no supera 0.25."
)


In [ ]:
# Softmax se gráfica aparte: no es una activación elemento a elemento como las anteriores,
# sino una función vectorial que mira todas las salidas a la vez y las convierte en una distribución.
generador = np.random.default_rng(7)
logits_demo = generador.normal(0, 1.5, NUMERO_CLASES)
logits_demo[min(9, NUMERO_CLASES - 1)] += 4.0  # una clase claramente favorecida por la red


def softmax(z, temperatura=1.0):
    """Softmax numéricamente estable: se resta el máximo antes de exponenciar."""
    escalado = np.asarray(z, dtype=float) / temperatura
    exponenciales = np.exp(escalado - escalado.max())
    return exponenciales / exponenciales.sum()


probabilidades_demo = softmax(logits_demo)

figura, ejes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Los logits: números sin escala definida, positivos y negativos
ejes[0].bar(range(NUMERO_CLASES), logits_demo, color="#8fb8de")
ejes[0].axhline(0, color="gray", linewidth=0.8)
ejes[0].set_title(f"Entrada: {NUMERO_CLASES} logits (sin escala)", fontsize=10)
ejes[0].set_ylabel("z (preactivación de la capa de salida)")

# 2. La misma información después de softmax: probabilidades que suman 1
ejes[1].bar(range(NUMERO_CLASES), probabilidades_demo * 100, color="#1b7f3b")
ejes[1].axhline(
    100 / NUMERO_CLASES,
    color="red",
    linestyle="--",
    linewidth=1,
    label=f"azar = {100 / NUMERO_CLASES:.1f} %",
)
ejes[1].set_title(
    f"Salida: softmax (suma = {probabilidades_demo.sum():.2f})", fontsize=10
)
ejes[1].set_ylabel("probabilidad (%)")
ejes[1].legend(fontsize=8)

for eje in ejes[:2]:
    eje.set_xlabel("índice de clase")
    eje.grid(alpha=0.3, axis="y")

# 3. La competencia entre clases: subir el logit de una hunde a todas las demas
desplazamientos = np.linspace(-4, 6, 200)
seguidas = sorted({0, min(9, NUMERO_CLASES - 1), min(17, NUMERO_CLASES - 1)})  # clases a seguir
curvas = np.array(
    [softmax(np.concatenate(([logits_demo[0] + d], logits_demo[1:]))) for d in desplazamientos]
)

for indice in seguidas:
    ejes[2].plot(
        desplazamientos,
        curvas[:, indice] * 100,
        linewidth=2,
        label=f"clase {indice}" + (" (la que sube)" if indice == 0 else ""),
    )

ejes[2].plot(
    desplazamientos,
    curvas.sum(axis=1) * 100,
    color="gray",
    linestyle="--",
    linewidth=1.2,
    label=f"suma de las {NUMERO_CLASES}",
)
ejes[2].set_title("Competencia: las probabilidades se reparten", fontsize=10)
ejes[2].set_xlabel("cuánto se suma al logit de la clase 0")
ejes[2].set_ylabel("probabilidad (%)")
ejes[2].grid(alpha=0.3)
ejes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(
    "Softmax no se puede dibujar junto a ReLU o tanh porque no transforma un número en otro:"
    "\ntoma el vector completo de logits y devuelve una distribución. El denominador -- la suma"
    "\nde todas las exponenciales -- es lo que acopla las salidas entre si."
)
print(
    f"\nClase predicha: {int(probabilidades_demo.argmax())} con {probabilidades_demo.max() * 100:.1f} %"
    f" de probabilidad   |   suma de las {NUMERO_CLASES} salidas = {probabilidades_demo.sum():.6f}"
)
print(
    "En el gráfico de la derecha se ve la propiedad clave para etiqueta única: al crecer el logit"
    "\nde una clase, su probabilidad sube y la de las demas baja, mientras la suma se mantiene en 100 %."
)


### Función de pérdida, backpropagation y optimizador

El forward pass produce una predicción, pero por sí solo no enseña nada. El aprendizaje necesita tres
piezas más: una forma de **medir** el error, una forma de **repartir la culpa** de ese error entre los
parámetros, y una **regla** para corregirlos.

**1. Función de pérdida: entropía cruzada categórica**

Mide la distancia entre la distribución predicha por softmax y el vector one-hot real. Como el one-hot
tiene un 1 en la clase correcta y 0 en el resto, la suma se reduce a un único término:

$$L = -\sum_{i=1}^{64} y_i \log(\hat{y}_i) = -\log(\hat{y}_{\text{clase correcta}})$$

Es decir: la pérdida de un ejemplo depende **solo de la probabilidad que la red asignó a la clase
verdadera**. Si le dio 0,99, la pérdida es casi 0; si le dio 0,01, la pérdida se dispara. Ese crecimiento
logarítmico penaliza con dureza los errores cometidos con mucha confianza, que es el comportamiento
deseable. La pérdida del batch es el promedio de las pérdidas individuales -- y es ahí donde `class_weight`
interviene, ponderando cada término según la frecuencia de su clase.

Por qué no usar accuracy como función a minimizar: el accuracy es escalonado, su gradiente es cero en casi
todas partes y no distingue entre acertar con 0,51 o con 0,99. Sirve para interpretar resultados, no para
optimizar. Por eso se monitorean ambos: se **optimiza** la pérdida y se **lee** el accuracy.

**2. Backpropagation: de dónde viene el error**

Conocida la pérdida, hace falta saber cuánto contribuyó cada uno de los parámetros. Eso es la derivada
parcial $\partial L / \partial w$, y calcularla directamente para cientos de miles de pesos sería
inviable. La retropropagación lo resuelve aplicando la **regla de la cadena** en sentido inverso: parte
del error en la salida y lo propaga hacia atras capa por capa, reutilizando en cada paso lo ya calculado
en la capa posterior. Esto es lo que hace posible entrenar redes grandes, y también la razón de que la
derivada de la activación importe tanto: aparece como factor multiplicativo en cada capa que se atraviesa
(de ahí el desvanecimiento del gradiente con sigmoide y la ventaja de ReLU).

El resultado es el **gradiente**: un vector que indica, para cada parámetro, en qué dirección habría que
moverlo para que la pérdida aumente. Se avanza en la dirección contraria.

**3. Optimizador: cómo se corrigen los pesos**

La regla básica es el **descenso del gradiente**:

$$w \leftarrow w - \eta \frac{\partial L}{\partial w}$$

donde $\eta$ es la **tasa de aprendizaje** (learning rate), el tamaño del paso. Demasiado grande y el
entrenamiento oscila o diverge; demasiado pequeño y avanza con una lentitud inútil. En la práctica no se
calcula el gradiente sobre todo el dataset (caro) ni sobre un solo ejemplo (ruidoso), sino sobre **batches**
de unas decenas o centenas de imágenes: es el descenso del gradiente por mini-lotes. Una **época** es una
pasada completa por el conjunto de entrenamiento, es decir, muchas actualizaciones de pesos.

**Elección: Adam.** Mejora el descenso del gradiente clásico en dos aspectos: acumula un **momento** de los
gradientes anteriores, lo que suaviza las oscilaciones, y adapta el tamaño del paso **para cada parámetro**
según la magnitud típica de su gradiente. En la práctica converge más rápido y es mucho menos sensible a
la elección inicial del learning rate, razón por la que se usa como punto de partida en la mayoría de los
trabajos. Se parte de su valor por defecto, 0,001, y su efecto se contrasta en la experimentación.

**El ciclo completo de entrenamiento**

Para cada batch, y repitiendolo época tras época:

`forward pass` -> `calcular la perdida` -> `backpropagation` -> `el optimizador actualiza los pesos`


In [ ]:
# Pérdida de un ejemplo según la probabilidad asignada a la clase correcta
probabilidades = np.linspace(0.001, 1.0, 500)
perdidas = -np.log(probabilidades)

figura, ejes = plt.subplots(1, 2, figsize=(11, 4))

ejes[0].plot(probabilidades, perdidas, color="crimson", linewidth=2)
ejes[0].set_xlabel("Probabilidad asignada a la clase correcta")
ejes[0].set_ylabel("Pérdida  -log(p)")
ejes[0].set_title("Entropía cruzada: castiga los errores confiados", fontsize=10)
ejes[0].grid(alpha=0.3)

for probabilidad in (0.01, 0.1, 0.5, 0.9):
    ejes[0].scatter([probabilidad], [-np.log(probabilidad)], zorder=3, s=35)
    ejes[0].annotate(
        f"p={probabilidad}  L={-np.log(probabilidad):.2f}",
        (probabilidad, -np.log(probabilidad)),
        textcoords="offset points",
        xytext=(8, 6),
        fontsize=8,
    )

# Referencias del problema: pérdida esperada al predecir al azar entre todas las clases
perdida_azar = -np.log(1 / NUMERO_CLASES)
ejes[0].axhline(
    perdida_azar,
    color="gray",
    linestyle="--",
    linewidth=1,
    label=f"prediccion al azar = {perdida_azar:.2f}",
)
ejes[0].legend()

# Efecto del learning rate sobre el descenso del gradiente en una parabola L(w) = w^2
ejes[1].plot(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200) ** 2, color="gray", alpha=0.5)

for tasa, color in ((0.05, "tab:blue"), (0.4, "tab:green"), (1.02, "tab:red")):
    w = 2.0
    trayectoria = [w]
    for _ in range(12):
        w = w - tasa * 2 * w  # derivada de w^2
        trayectoria.append(w)
    trayectoria = np.array(trayectoria)
    ejes[1].plot(
        trayectoria,
        trayectoria ** 2,
        "o-",
        color=color,
        markersize=4,
        linewidth=1,
        label=f"lr = {tasa}",
    )

ejes[1].set_xlabel("w (parámetro)")
ejes[1].set_ylabel("L(w)")
ejes[1].set_title("Efecto del learning rate en el descenso del gradiente", fontsize=10)
ejes[1].grid(alpha=0.3)
ejes[1].legend()

plt.tight_layout()
plt.show()

print(f"Perdida esperada de un modelo sin entrenar (1/{NUMERO_CLASES} a cada clase): {perdida_azar:.3f}")
print("Es el valor de referencia con el que debería arrancar el entrenamiento en la primera época.")
print(
    "\nEn el gráfico de la derecha: un lr pequeño (azul) converge pero lento, uno adecuado (verde)"
    "\nllega rápido al mínimo, y uno demasiado grande (rojo) se aleja y diverge."
)


## Modelo base: red neuronal convolucional (CNN)

### Configuración y alcance de la implementación

El modelo base recibe la imagen completa, un tensor de **64 × 64 × 3** píxeles, y tiene **cuatro bloques
convolucionales** de 32, 64, 128 y 256 filtros de 3 × 3, cada uno con Batch Normalization, ReLU y *max
pooling* de 2 × 2. Un pooling global promedia cada mapa de activación y una cabeza densa de 256 neuronas,
con Dropout de 0,3, lleva a la salida de 64 neuronas con `softmax`. El entrenamiento es el del MLP:
`categorical_crossentropy` con suavizado de etiquetas de 0,1 y etiquetas de escena, Adam con tasa `1e-3`,
batch de 128, hasta 60 épocas y pesos por clase suavizados (raíz cuadrada de los balanceados). Todos esos
valores se declaran en las celdas de arquitectura, compilación y entrenamiento que siguen.

El modelo base no se presenta como una elección arbitraria. Se compara con una variante que cambia la
entrada (escala de grises), con siete que cambian una sola decisión de arquitectura u optimización
(profundidad, activación, tasa de aprendizaje, batch, dropout o la cabeza) y con cinco que cambian el
tratamiento de los datos (pesos por clase, suavizado, etiquetas de escena e imágenes alteradas). Al final,
una configuración combina los cambios que mejoraron al modelo base. La regla de selección es el F1 macro de
validación, como en el MLP.

### Definición de la arquitectura

La arquitectura traduce a código lo descrito en los fundamentos. La imagen entra en `uint8` y una capa
`Rescaling(1/255)` la lleva a [0, 1] dentro del modelo, sobre cada lote. Después vienen los bloques
convolucionales, cada uno con cuatro capas:

- **`Conv2D`** con filtros de 3 × 3 y `padding="same"`, para que el mapa conserve el tamaño de su entrada. No
  lleva sesgo: la normalización que sigue resta la media de cada mapa, así que un sesgo se cancelaría.
- **`BatchNormalization`.** Reescala cada mapa de activación para que mantenga una media y una varianza
  estables a lo largo del entrenamiento. Como en el MLP, permite una tasa de aprendizaje mayor y acelera la
  convergencia; en una CNN es además lo que deja entrenar varias convoluciones seguidas sin que la escala de
  las activaciones se dispare o se apague.
- **Activación** (ReLU en el modelo base).
- **`MaxPooling2D`** de 2 × 2, que reduce el mapa a la mitad de filas y de columnas.

Cada bloque duplica el número de filtros mientras el mapa se reduce a la mitad por lado: la red cambia
resolución espacial por número de patrones. Tras el último bloque, `GlobalAveragePooling2D` resume cada
mapa en un número y la **cabeza clasificadora** —Dropout, una capa densa de 256 neuronas con ReLU,
BatchNormalization y Dropout— decide el personaje con una salida `softmax`. El Dropout va solo en la
cabeza: es la parte densa de la red, la que tiende a memorizar.

La entrada en escala de grises del experimento R1 también vive dentro del modelo: una convolución de 1 × 1
con pesos fijos (la luminancia, 0,299 R + 0,587 G + 0,114 B) que no se entrena. Así todas las
configuraciones reciben la misma imagen RGB.

La función `construir_cnn` deja como parámetros los factores que se modificarán en la experimentación
(entrada en color o en grises, filtros por bloque, convoluciones por bloque, activación, tasa de dropout y
tipo de cabeza), de modo que todas las configuraciones comparadas se generen exactamente con el mismo
código.

In [ ]:
import keras
from keras import layers

# Fija las semillas de Python, NumPy y PyTorch para que el entrenamiento sea reproducible
keras.utils.set_random_seed(SEMILLA)

FILTROS_POR_BLOQUE = (32, 64, 128, 256)
CONVOLUCIONES_POR_BLOQUE = 1
TAMANO_FILTRO = 3
NEURONAS_DENSAS = 256
ACTIVACION = "relu"
TASA_DROPOUT = 0.3
CABEZA = "pooling_global"
REPRESENTACION_BASE = "rgb"
# Valores que ven las convoluciones con cada entrada: en grises, un canal en lugar de tres
DIMENSIONES = {"rgb": DIMENSION_PIXELES, "gris": ALTO_OBJETIVO * ANCHO_OBJETIVO}
DIMENSION_ENTRADA = DIMENSIONES[REPRESENTACION_BASE]
# Pesos de la conversión a escala de grises: la luminancia de ITU-R BT.601
PESOS_GRISES = np.array([0.299, 0.587, 0.114], dtype=np.float32)


def construir_cnn(
    filtros_por_bloque=FILTROS_POR_BLOQUE,
    convoluciones_por_bloque=CONVOLUCIONES_POR_BLOQUE,
    activacion=ACTIVACION,
    tasa_dropout=TASA_DROPOUT,
    cabeza=CABEZA,
    representacion=REPRESENTACION_BASE,
    numero_clases=NUMERO_CLASES,
    nombre="cnn_base",
):
    """Construye una red convolucional: imagen uint8 -> escalado a [0, 1] -> (escala de grises) ->
    bloques Conv2D + BatchNorm + activación + MaxPooling -> pooling global o aplanado ->
    cabeza densa con Dropout -> salida softmax con una neurona por clase."""
    modelo = keras.Sequential(name=nombre)
    modelo.add(layers.Input(shape=FORMA_ENTRADA, name="entrada"))
    # El escalado a [0, 1] vive dentro del modelo: las imágenes se guardan en
    # uint8 (cuatro veces menos memoria) y se escalan sobre cada lote.
    modelo.add(layers.Rescaling(1.0 / 255, name="escalado"))
    if representacion == "gris":
        # Convolución 1 x 1 con pesos fijos que combina R, G y B en la luminancia. Va dentro del
        # modelo, así que el archivo guardado recibe la misma imagen RGB que las demás configuraciones.
        modelo.add(layers.Conv2D(1, 1, use_bias=False, trainable=False, name="escala_grises"))

    for bloque, filtros in enumerate(filtros_por_bloque, start=1):
        for convolucion in range(1, convoluciones_por_bloque + 1):
            sufijo = f"{bloque}" if convoluciones_por_bloque == 1 else f"{bloque}_{convolucion}"
            modelo.add(
                layers.Conv2D(filtros, TAMANO_FILTRO, padding="same", use_bias=False, name=f"conv_{sufijo}")
            )
            modelo.add(layers.BatchNormalization(name=f"norm_{sufijo}"))
            modelo.add(layers.Activation(activacion, name=f"activacion_{sufijo}"))
        modelo.add(layers.MaxPooling2D(2, name=f"pooling_{bloque}"))

    if cabeza == "pooling_global":
        # Un número por filtro: cuánto se activó en la imagen, sin importar dónde
        modelo.add(layers.GlobalAveragePooling2D(name="pooling_global"))
    else:
        # Aplanado: la cabeza recibe el último mapa completo, con la posición de cada activación
        modelo.add(layers.Flatten(name="aplanado"))
    modelo.add(layers.Dropout(tasa_dropout, name="dropout_1"))
    modelo.add(layers.Dense(NEURONAS_DENSAS, activation=activacion, name="densa"))
    modelo.add(layers.BatchNormalization(name="norm_densa"))
    modelo.add(layers.Dropout(tasa_dropout, name="dropout_2"))
    modelo.add(layers.Dense(numero_clases, activation="softmax", name="salida"))

    if representacion == "gris":
        modelo.get_layer("escala_grises").set_weights([PESOS_GRISES.reshape(1, 1, 3, 1)])
    return modelo


modelo_base = construir_cnn()

modelo_base = modelo_base.to(DISPOSITIVO)

filtros_texto = " -> ".join(str(n) for n in FILTROS_POR_BLOQUE)
print(f"Keras {keras.__version__} sobre backend {keras.backend.backend()}")
print(f"Entrada: imagen {ALTO_OBJETIVO}x{ANCHO_OBJETIVO}x3 ({REPRESENTACION_BASE}), escalada a [0, 1] dentro del modelo")
print(
    f"Bloques convolucionales: {filtros_texto} filtros de {TAMANO_FILTRO}x{TAMANO_FILTRO},"
    f" {CONVOLUCIONES_POR_BLOQUE} convolucion por bloque, max pooling 2x2"
)
print(f"Cabeza: {CABEZA} -> densa {NEURONAS_DENSAS} -> {NUMERO_CLASES}")
print(f"Activacion: {ACTIVACION}   |   Dropout (cabeza): {TASA_DROPOUT}   |   Salida: softmax")

### Justificación técnica de la arquitectura y de los hiperparámetros iniciales

Cada decisión del modelo base responde al problema concreto ya descrito: imágenes de 64 × 64 × 3, 64 clases
desbalanceadas y unas 35.000 imágenes de entrenamiento (el 70 % de las originales de cada personaje, más sus
variantes alteradas).

| Hiperparámetro | Valor elegido | Justificación |
|---|---|---|
| **Entrada** | Imagen RGB de 64 × 64, sin aplanar | La convolución aprovecha la vecindad de los píxeles, y el color se conserva porque el MLP mostró que distingue a muchos personajes. La escala de grises se compara en R1 |
| **Bloques convolucionales** | 4, de 32, 64, 128 y 256 filtros | Con cuatro poolings el mapa pasa de 64 × 64 a 4 × 4, y el campo receptivo del último bloque cubre 38 × 38 píxeles. Duplicar los filtros en cada bloque compensa la resolución que se pierde. E1 prueba un bloque menos y E2 dos convoluciones por bloque |
| **Convoluciones por bloque** | 1 | Una convolución por bloque cuesta en este equipo la mitad que dos; E2 mide si la segunda compensa su costo |
| **Tamaño del filtro** | 3 × 3 | El menor filtro con un centro y vecinos en todas direcciones; dos capas de 3 × 3 ven lo mismo que una de 5 × 5 con menos pesos |
| **Batch Normalization** | Tras cada convolución y en la cabeza | Estabiliza la escala de las activaciones capa a capa y permite usar la tasa de aprendizaje del MLP sin que el entrenamiento se vuelva inestable |
| **Activación oculta** | ReLU | No se satura y es la más barata de calcular (ver «Funciones de activación»); E3 mide el costo de tanh |
| **Cabeza** | Pooling global + densa de 256 | El pooling global da un número por filtro sin importar dónde se activó, así que la capa densa tiene solo 65.536 pesos; E7 la compara con aplanar el último mapa, que conserva la posición |
| **Dropout** | 0,3, en la cabeza | El mismo valor del MLP, en la parte densa de la red; E6 prueba 0,5 |
| **Activación de salida** | Softmax (64 neuronas) | Distribución de probabilidad que suma 1: clasificación multiclase de etiqueta única |
| **Optimizador** | Adam, tasa 0,001 | El del MLP, para que las dos redes se entrenen igual; E4 contrasta 0,0001 |
| **Función de pérdida** | Categorical crossentropy con suavizado 0,1 | Igual que en el MLP; S1 la compara con el one-hot puro |
| **Tamaño de batch** | 128 | El del MLP; E5 prueba 512 |
| **`class_weight`** | Raíz cuadrada de los pesos balanceados | Igual que en el MLP: compensa el desbalance de 33:1 sin anular a los personajes frecuentes; P1 y P2 comparan sin pesos y con los balanceados completos |

### Compilación del modelo

In [ ]:
TASA_APRENDIZAJE = 1e-3
# Suavizado de etiquetas: fracción del objetivo de entrenamiento que se reparte entre todas las clases
SUAVIZADO_ETIQUETAS = 0.1


def compilar(modelo, tasa_aprendizaje=TASA_APRENDIZAJE):
    """Asocia al modelo la función de pérdida y las métricas.

    El entrenamiento NO usa `fit()`, sino el bucle propio de la sección siguiente.
    Se compila igualmente porque `predict()` y `evaluate()` sí se aprovechan: son
    llamadas puntuales sobre unos pocos miles de imágenes, donde el sobrecoste del
    bucle de Keras es irrelevante, y evitan reescribir código ya verificado."""
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=tasa_aprendizaje),
        loss="categorical_crossentropy",
        metrics=[
            keras.metrics.CategoricalAccuracy(name="accuracy"),
            keras.metrics.TopKCategoricalAccuracy(k=3, name="top_3"),
        ],
    )
    return modelo


compilar(modelo_base)

configuracion_base = pd.DataFrame(
    [
        ("Optimizador", "Adam"),
        ("Tasa de aprendizaje", TASA_APRENDIZAJE),
        ("Función de pérdida", "categorical_crossentropy"),
        ("Suavizado de etiquetas (solo entrenamiento)", SUAVIZADO_ETIQUETAS),
        ("Etiquetas de escena (solo entrenamiento)", f"{PESO_ESCENA:.0%} para los acompañantes"),
        ("Pesos por clase", ESQUEMA_PESOS),
        ("Metricas", "accuracy, top_3"),
        ("Filtros por bloque", str(FILTROS_POR_BLOQUE)),
        ("Convoluciones por bloque", CONVOLUCIONES_POR_BLOQUE),
        ("Tamaño del filtro", f"{TAMANO_FILTRO}x{TAMANO_FILTRO}"),
        ("Cabeza", f"{CABEZA} + densa de {NEURONAS_DENSAS}"),
        ("Activación oculta", ACTIVACION),
        ("Dropout (cabeza)", TASA_DROPOUT),
        ("Neuronas de salida", NUMERO_CLASES),
    ],
    columns=["Elemento", "Valor"],
)

display(configuracion_base)

### Resumen del modelo

In [ ]:
modelo_base.summary()

# Reparto de los parámetros entre las capas
detalle_capas = pd.DataFrame(
    [
        (
            capa.name,
            capa.__class__.__name__,
            str(capa.output.shape),
            int(sum(np.prod(peso.shape) for peso in capa.trainable_weights)),
        )
        for capa in modelo_base.layers
    ],
    columns=["capa", "tipo", "forma_salida", "parametros"],
)
detalle_capas["% del total"] = (
    detalle_capas["parametros"] / max(detalle_capas["parametros"].sum(), 1) * 100
).round(1)

display(detalle_capas)

total_parametros = int(detalle_capas["parametros"].sum())
print(f"Parametros entrenables: {total_parametros:,}")
print(f"Imagenes de entrenamiento: {len(X_train):,}")
print(f"Razon parametros / ejemplos: {total_parametros / len(X_train):.0f} a 1")
mayor = detalle_capas.loc[detalle_capas["parametros"].idxmax()]
print(
    f"\nLa capa con mas parametros es {mayor['capa']} ({mayor['% del total']:.0f} % del total): sus"
    f"\nfiltros combinan los {FILTROS_POR_BLOQUE[-2]} mapas del bloque anterior. Las convoluciones no"
    "\ndependen de la resolucion: un filtro de 3x3 tiene los mismos pesos a 32x32 que a 128x128."
)

## Entrenamiento del modelo

### Configuración del entrenamiento

### Control del entrenamiento

El entrenamiento se controla con cuatro decisiones y tres mecanismos de parada y ajuste.

| Decisión | Valor | Por qué |
| --- | --- | --- |
| Épocas máximas | 60 | Techo generoso: quien decide cuándo parar es la parada temprana, no este número |
| Tamaño de lote | 128 | El mismo que usa la corrida de GPU, para que los tiempos sigan siendo comparables |
| Pesos por clase | Raíz cuadrada del `class_weight` balanceado | Compensa el desbalance de 33:1 sin anular a los personajes frecuentes |
| Suavizado de etiquetas | 0,1 | Solo en la pérdida de entrenamiento: la de validación usa la etiqueta estricta, así que la parada temprana se decide con la misma vara en todas las configuraciones |

**Dónde viven los datos.** Las imágenes se convierten una sola vez en tensores de PyTorch, en `uint8`. En
la CPU, `torch.as_tensor` comparte la memoria del array de NumPy, así que la conversión no duplica las
imágenes, y cada paso toma su lote directamente de ellas.

**El mismo bucle que en GPU.** El entrenamiento no usa `fit()`, sino la función `entrenar()` del notebook
de GPU: un `keras.Sequential` sobre el backend de PyTorch *es* un `torch.nn.Module`, así que el mismo
bucle corre en la CPU o en la GPU según dónde estén el modelo y los datos. En GPU se escribió para que la
tarjeta no esperara al intérprete de Python entre paso y paso; aquí se mantiene para que las dos corridas
hagan exactamente el mismo cálculo —el mismo suavizado de etiquetas, los mismos pesos por clase, los mismos
criterios de parada— y la comparación de tiempos mida solo el hardware.

La función `entrenar()` reproduce el comportamiento de los tres *callbacks* de Keras:

| Mecanismo | Qué hace | Por qué |
| --- | --- | --- |
| `EarlyStopping` | Detiene el entrenamiento si `val_loss` no mejora en 12 épocas y restaura los pesos de la mejor época | Evita seguir entrenando cuando el modelo ya solo memoriza |
| `ReduceLROnPlateau` | Divide el learning rate a la mitad si `val_loss` se estanca 4 épocas | Pasos grandes al principio y ajustes finos al final, cerca del mínimo |
| `ModelCheckpoint` | Guarda en `models/newDataset/` los pesos de la mejor época | Deja el modelo persistido en disco, reutilizable sin reentrenar |

**Por qué la paciencia es 12 y no 8.** La paciencia se fijó con el dataset anterior, cuya `val_loss` era
ruidosa: una época afortunada podía marcar un mínimo que tardaba mucho en repetirse, y la paciencia se
consumía esperándolo. Medido entonces sobre tres barajados distintos, la mejor `val_loss` alcanzada
mejora al pasar de 8 a 12 épocas de paciencia (1,9612 → 1,8735 en uno de ellos; 1,8892 → 1,8684 en otro)
y no mejora más al subir a 16.

El cambio es seguro por construcción: como los pesos devueltos son siempre los de la mejor época
(`restore_best_weights`), **esperar más nunca puede devolver un modelo peor**, solo gastar más épocas. En
CPU cada época cuesta más que en GPU, pero la paciencia es la misma para que las dos corridas sigan el mismo
proceso y sus resultados se puedan comparar.

In [ ]:
import copy
import time
from pathlib import Path

import torch

EPOCAS_MAXIMAS = 60
TAMANO_BATCH = 128
DIRECTORIO_MODELOS = Path("../../models/newDataset")
DIRECTORIO_MODELOS.mkdir(parents=True, exist_ok=True)


class Historial:
    """Misma interfaz que el objeto que devuelve keras.Model.fit(): un atributo
    .history con un diccionario {metrica: [valor por epoca]}."""

    def __init__(self):
        self.history = {}

    def anotar(self, **metricas):
        for nombre, valor in metricas.items():
            self.history.setdefault(nombre, []).append(float(valor))


def _pasada(modelo, X, y, batch, entrenando, optimizador=None, pesos=None, generador=None, suavizado=0.0):
    """Recorre el conjunto una vez y devuelve (perdida, accuracy, top_3).

    Los acumuladores son tensores y se convierten a float una sola vez, al final de la
    pasada, igual que en el notebook de GPU.
    """
    modelo.train(entrenando)
    n = len(X)
    if entrenando:
        orden = torch.randperm(n, device=X.device, generator=generador)
    else:
        orden = torch.arange(n, device=X.device)

    suma_perdida = torch.zeros((), device=X.device)
    suma_top1 = torch.zeros((), device=X.device)
    suma_top3 = torch.zeros((), device=X.device)

    for i in range(0, n, batch):
        indices = orden[i:i + batch]
        lote_x, lote_y = X[indices], y[indices]
        objetivo = lote_y.argmax(1)
        if entrenando and suavizado:
            # Suavizado de etiquetas: solo al entrenar; validación mide con la etiqueta estricta
            lote_y = lote_y * (1 - suavizado) + suavizado / lote_y.shape[1]

        with torch.set_grad_enabled(entrenando):
            probabilidades = modelo(lote_x, training=entrenando)
            # Entropia cruzada categorica, muestra a muestra
            perdida_muestra = -(lote_y * torch.log(probabilidades.clamp_min(1e-7))).sum(1)
            if pesos is not None:
                # class_weight: cada ejemplo pesa segun su clase real
                perdida_muestra = perdida_muestra * pesos[objetivo]
            perdida = perdida_muestra.mean()

        if entrenando:
            optimizador.zero_grad(set_to_none=True)
            perdida.backward()
            optimizador.step()

        with torch.no_grad():
            suma_perdida += perdida_muestra.sum()
            suma_top1 += (probabilidades.argmax(1) == objetivo).sum()
            suma_top3 += (probabilidades.topk(3, dim=1).indices
                          == objetivo.unsqueeze(1)).any(1).sum()

    return (float(suma_perdida) / n, float(suma_top1) / n, float(suma_top3) / n)


def entrenar(
    modelo,
    X_entrena,
    y_entrena,
    X_valida,
    y_valida,
    epocas=EPOCAS_MAXIMAS,
    batch=TAMANO_BATCH,
    tasa_aprendizaje=TASA_APRENDIZAJE,
    pesos_por_clase=None,
    suavizado=SUAVIZADO_ETIQUETAS,
    paciencia=12,
    factor_lr=0.5,
    paciencia_lr=4,
    lr_minimo=1e-5,
    ruta_modelo=None,
    semilla=SEMILLA,
    verbose=2,
):
    """Entrena el modelo y devuelve un Historial equivalente al de model.fit().

    Reproduce los tres callbacks del notebook original: EarlyStopping con
    restore_best_weights, ReduceLROnPlateau y el guardado de la mejor epoca.
    """
    optimizador = torch.optim.Adam(modelo.parameters(), lr=tasa_aprendizaje, fused=True)

    generador = torch.Generator(device=X_entrena.device)
    generador.manual_seed(semilla)

    pesos = None
    if pesos_por_clase is not None:
        pesos = torch.tensor(
            [pesos_por_clase[i] for i in range(len(pesos_por_clase))],
            dtype=torch.float32, device=X_entrena.device,
        )

    historial = Historial()
    mejor_perdida = float("inf")
    mejor_epoca = 0
    sin_mejora_parada = 0
    sin_mejora_lr = 0
    mejores_pesos = None

    for epoca in range(1, epocas + 1):
        inicio = time.perf_counter()
        perdida, acierto, top3 = _pasada(
            modelo, X_entrena, y_entrena, batch, True, optimizador, pesos, generador, suavizado
        )
        val_perdida, val_acierto, val_top3 = _pasada(
            modelo, X_valida, y_valida, batch, False
        )
        duracion = time.perf_counter() - inicio

        historial.anotar(
            loss=perdida, accuracy=acierto, top_3=top3,
            val_loss=val_perdida, val_accuracy=val_acierto, val_top_3=val_top3,
            learning_rate=optimizador.param_groups[0]["lr"],
        )

        if val_perdida < mejor_perdida - 1e-6:
            mejor_perdida, mejor_epoca = val_perdida, epoca
            sin_mejora_parada = sin_mejora_lr = 0
            # Copia aparte de los mejores pesos: en la CPU .cpu() no copia nada, asi que
            # clone() es el que los separa de los parametros que siguen entrenando.
            mejores_pesos = {
                nombre: tensor.detach().cpu().clone()
                for nombre, tensor in modelo.state_dict().items()
            }
        else:
            sin_mejora_parada += 1
            sin_mejora_lr += 1

        if verbose >= 2:
            print(f"Epoca {epoca:3d}/{epocas}  {duracion:5.2f}s"
                  f"  loss={perdida:.4f}  acc={acierto:.4f}"
                  f"  val_loss={val_perdida:.4f}  val_acc={val_acierto:.4f}")

        # ReduceLROnPlateau
        if sin_mejora_lr >= paciencia_lr:
            lr_actual = optimizador.param_groups[0]["lr"]
            lr_nuevo = max(lr_actual * factor_lr, lr_minimo)
            if lr_nuevo < lr_actual:
                for grupo in optimizador.param_groups:
                    grupo["lr"] = lr_nuevo
                if verbose:
                    print(f"  ReduceLROnPlateau: learning rate {lr_actual:.2e} -> {lr_nuevo:.2e}")
            sin_mejora_lr = 0

        # EarlyStopping
        if sin_mejora_parada >= paciencia:
            if verbose:
                print(f"  EarlyStopping en la epoca {epoca}: val_loss no mejora "
                      f"desde la epoca {mejor_epoca}")
            break

    # restore_best_weights: se vuelve a la mejor epoca antes de devolver el modelo
    if mejores_pesos is not None:
        modelo.load_state_dict({
            nombre: tensor.to(X_entrena.device) for nombre, tensor in mejores_pesos.items()
        })

    # Equivalente a ModelCheckpoint(save_best_only=True): el archivo que queda en
    # disco es el de la mejor epoca. Se escribe una sola vez, al final, en lugar de
    # en cada mejora: el contenido es identico y no se paga la escritura de 100 MB
    # en mitad de la medicion de tiempos.
    if ruta_modelo is not None:
        modelo.save(ruta_modelo)

    return historial


# Los datos se convierten UNA vez en tensores de PyTorch: las imagenes en uint8 y las
# etiquetas en float32. En la CPU, torch.as_tensor comparte la memoria del array de NumPy,
# asi que la conversion no duplica las imagenes.
def a_tensor(arreglo):
    return torch.as_tensor(np.ascontiguousarray(arreglo)).to(DISPOSITIVO)


X_train = a_tensor(PARTICION["train"])
X_val = a_tensor(PARTICION["val"])
y_train = a_tensor(y_train)
y_val = a_tensor(y_val)
y_train_duro = a_tensor(y_train_duro)
# Filas de train que son originales: el experimento A1 entrena sin las alteradas
filas_originales_train = a_tensor(~datos.loc[MASCARAS["train"], "alterada"].to_numpy())

print(f"X_train: {tuple(X_train.shape)}  {X_train.dtype}  en {X_train.device}")
print(f"X_val  : {tuple(X_val.shape)}  {X_val.dtype}  en {X_val.device}\n")

# El tiempo se cronometra aquí para el resumen de rendimiento del final del notebook
inicio_entrenamiento_base = time.perf_counter()
historial_base = entrenar(
    modelo_base,
    X_train,
    y_train,
    X_val,
    y_val,
    epocas=EPOCAS_MAXIMAS,
    batch=TAMANO_BATCH,
    pesos_por_clase=pesos_por_clase,
    ruta_modelo=DIRECTORIO_MODELOS / "cnn_base.keras",
)
segundos_entrenamiento_base = time.perf_counter() - inicio_entrenamiento_base

epocas_ejecutadas = len(historial_base.history["loss"])
mejor_epoca = int(np.argmin(historial_base.history["val_loss"])) + 1
print(f"\nEpocas ejecutadas: {epocas_ejecutadas} de {EPOCAS_MAXIMAS} (parada temprana)")
print(f"Mejor epoca segun val_loss: {mejor_epoca}")
print(
    f"Tiempo de entrenamiento: {segundos_entrenamiento_base:.1f} s"
    f"  ({segundos_entrenamiento_base / epocas_ejecutadas:.2f} s por epoca)"
)

### Curvas de aprendizaje del modelo base

In [ ]:
def graficar_curvas(historial, titulo="Modelo base"):
    """Dibuja pérdida y accuracy de entrenamiento y validación por época."""
    registro = historial.history if hasattr(historial, "history") else historial
    epocas = range(1, len(registro["loss"]) + 1)
    epoca_optima = int(np.argmin(registro["val_loss"])) + 1

    figura, ejes = plt.subplots(1, 2, figsize=(12, 4.5))

    ejes[0].plot(epocas, registro["loss"], label="entrenamiento", linewidth=2)
    ejes[0].plot(epocas, registro["val_loss"], label="validacion", linewidth=2)
    ejes[0].set_ylabel("Pérdida (categorical crossentropy)")
    ejes[0].set_title("Pérdida por época", fontsize=10)

    ejes[1].plot(epocas, registro["accuracy"], label="entrenamiento", linewidth=2)
    ejes[1].plot(epocas, registro["val_accuracy"], label="validacion", linewidth=2)
    ejes[1].set_ylabel("Accuracy")
    ejes[1].set_title("Accuracy por época", fontsize=10)

    for eje in ejes:
        eje.axvline(
            epoca_optima,
            color="gray",
            linestyle="--",
            linewidth=1,
            label=f"mejor epoca = {epoca_optima}",
        )
        eje.set_xlabel("Epoca")
        eje.grid(alpha=0.3)
        eje.legend()

    figura.suptitle(f"Curvas de aprendizaje - {titulo}")
    plt.tight_layout()
    plt.show()


graficar_curvas(historial_base)

### Interpretación de las curvas

In [ ]:
from sklearn.metrics import f1_score

registro = historial_base.history
indice_optimo = int(np.argmin(registro["val_loss"]))

# F1 macro en validación con los pesos restaurados de la mejor época
probabilidades_val = modelo_base.predict(X_val, batch_size=256, verbose=0)
predicciones_val = probabilidades_val.argmax(axis=1)
f1_macro_val = f1_score(y_val_idx, predicciones_val, average="macro")

diagnostico = pd.DataFrame(
    [
        ("Épocas ejecutadas", len(registro["loss"])),
        ("Mejor época (val_loss)", indice_optimo + 1),
        ("Pérdida entrenamiento", round(registro["loss"][indice_optimo], 4)),
        ("Pérdida validación", round(registro["val_loss"][indice_optimo], 4)),
        ("Accuracy entrenamiento", round(registro["accuracy"][indice_optimo], 4)),
        ("Accuracy validación", round(registro["val_accuracy"][indice_optimo], 4)),
        ("Top-3 accuracy validación", round(registro["val_top_3"][indice_optimo], 4)),
        ("F1 macro validación", round(float(f1_macro_val), 4)),
        (
            "Brecha train - val (accuracy)",
            round(registro["accuracy"][indice_optimo] - registro["val_accuracy"][indice_optimo], 4),
        ),
    ],
    columns=["Indicador", "Valor"],
)

display(diagnostico)

brecha = registro["accuracy"][indice_optimo] - registro["val_accuracy"][indice_optimo]
if brecha > 0.15:
    lectura = "sobreajuste claro: la red memoriza el entrenamiento y generaliza bastante peor"
elif brecha > 0.05:
    lectura = "sobreajuste moderado, contenido por el Dropout y la parada temprana"
else:
    lectura = "sin sobreajuste apreciable; las dos curvas evolucionan juntas"

print(f"Brecha train - val en accuracy: {brecha:.3f} -> {lectura}.")
print(
    f"Accuracy de validacion ({registro['val_accuracy'][indice_optimo]:.3f}) frente a los baselines:"
    f" azar {baseline_azar / 100:.3f} y clase mayoritaria {baseline_mayoritaria / 100:.3f}."
)

**Cómo se leen estas curvas**

Hay tres cosas que mirar en los gráficos anteriores:

1. **La separación entre las dos curvas de pérdida.** Mientras ambas bajan juntas, la red está
   aprendiendo estructura real del problema. En cuanto la pérdida de validación se aplana o sube
   mientras la de entrenamiento sigue bajando, lo que la red gana ya es memorización de las imágenes
   concretas del conjunto de entrenamiento: ese punto de cruce es el sobreajuste y es justo lo que
   detecta `EarlyStopping`.
2. **La época en que la validación deja de mejorar** (línea vertical punteada). Es la que se conserva,
   porque `restore_best_weights=True` devuelve los pesos de ese momento, y no los del final.
3. **El efecto de la regularización.** El Dropout de la cabeza hace que la curva de entrenamiento vaya *por
   debajo* de su potencial —parte de la red está apagada en cada paso— y por eso la brecha entre train y
   validación es menor de la que tendría la misma red sin regularizar. El escalón que a veces se ve en las
   curvas corresponde a `ReduceLROnPlateau` bajando el learning rate.

La tabla de arriba resume el diagnóstico: la brecha entre entrenamiento y validación cuantifica el
sobreajuste, y la distancia frente a los baselines (azar y clase mayoritaria, impresos por la celda
anterior) confirma que el modelo efectivamente aprendió algo del problema.

**En la corrida de GPU a 64 × 64 del 19-10-2026** el modelo base usa las 60 épocas sin que la parada
temprana lo corte: la pérdida de validación sigue bajando despacio y toca su mínimo en la época 52 (0,977).
`ReduceLROnPlateau` divide la tasa por dos seis veces, y el primer escalón se nota mucho: tras él, en la época
15, la accuracy de validación salta de 0,58 a 0,70. El diagnóstico es de sobreajuste claro —0,975 de accuracy
en entrenamiento y 0,764 en validación, una brecha de 0,21—, pero la validación no empeora mientras la red
memoriza: lo que aprende generaliza mucho mejor que el MLP base, que llegaba a una accuracy de entrenamiento
cercana a 0,95 y se quedaba entre 0,43 y 0,48 en validación.

## Experimentación: comparación de configuraciones



### Diseño de los experimentos

La comparación sigue el principio de **un factor a la vez**, igual que en el MLP: cada experimento cambia
exactamente un factor respecto del modelo base y deja el resto idéntico —misma semilla, misma partición,
mismos pesos por clase, mismos criterios de parada—. Así, cualquier diferencia en la métrica de validación
es atribuible al factor modificado y no a la interacción de varios cambios simultáneos.

| Exp. | Factor que cambia | Configuración | Qué pone a prueba |
|---|---|---|---|
| **Base** | — | Imagen RGB, 4 bloques (32, 64, 128, 256) de una convolución de 3 × 3, ReLU, pooling global, densa 256, dropout 0,3, lr 0,001, batch 128, pesos raíz, suavizado 0,1, etiquetas de escena, con alteradas | Referencia |
| **R1** | Representación | Escala de grises | Cuánto depende la CNN del color, que era casi todo lo que usaba el MLP |
| **E1** | Profundidad | 3 bloques (32, 64, 128) | Si un bloque menos basta, con un campo receptivo de 18 × 18 píxeles en lugar de 38 × 38 |
| **E2** | Profundidad | 4 bloques de 2 convoluciones | Si más capas por bloque, al estilo de VGG, mejoran o solo encarecen |
| **E3** | Activación | tanh | El costo de una activación que se satura |
| **E4** | Tasa de aprendizaje | lr 0,0001 | Convergencia lenta pero estable frente al valor por defecto |
| **E5** | Tamaño de batch | batch 512 | Menos ruido por actualización y menos actualizaciones por época |
| **E6** | Regularización | Dropout 0,5 en la cabeza | Si más regularización reduce la brecha entre entrenamiento y validación |
| **E7** | Cabeza | Aplanado (`Flatten`) en lugar de pooling global | Si conservar la posición de cada activación ayuda o lleva a memorizar, como en el MLP |
| **P1** | Pesos por clase | Sin pesos | Cuánto aporta compensar el desbalance |
| **P2** | Pesos por clase | Balanceados completos | Si compensar del todo ayuda a los minoritarios o castiga demasiado a los frecuentes |
| **S1** | Suavizado | One-hot puro, sin suavizado | Si repartir el 10 % del objetivo ayuda con las etiquetas ambiguas |
| **S2** | Etiquetas de escena | Solo la etiqueta de la carpeta | Si informar al modelo de los otros personajes de la escena mejora la generalización |
| **A1** | Aumento de datos | Sin imágenes alteradas | Si las variantes con giro, cambio de color y filtro de textura aportan |
| **C1** | Combinación | Los cambios que superaron a la base, juntos | Si las mejoras individuales se suman |

Las métricas registradas para cada configuración son la **accuracy** de validación —comparable con los
baselines— y el **F1 macro** de validación, que promedia sin ponderar los 64 personajes y es la métrica
que realmente refleja si el modelo atiende a los minoritarios. La selección final se hace por F1 macro,
por el desbalance ya documentado.

**Combinación final (C1).** Cambiar un factor a la vez dice qué ayuda por separado, pero no si las
mejoras se suman. Por eso, tras los experimentos individuales, C1 toma para cada factor el valor del
experimento que más superó a la base en F1 macro de validación y los aplica todos juntos. Si menos de dos
factores mejoraron, C1 no se entrena: coincidiría con un experimento ya medido.

Para acotar el costo, los experimentos usan un máximo de 35 épocas y una paciencia de 9 (el modelo
base se entrenó con 60 y paciencia 12); todos comparten exactamente ese presupuesto entre sí.

**Memoria de la GPU.** En el notebook de GPU, antes de entrenar cada configuración, un paso de prueba mide
cuánta memoria usa. A
64 × 64 todas caben en los 8 GB de la tarjeta; a 128 × 128, las más grandes —batch 512, dos convoluciones
por bloque— pueden no caber, y entonces se informa y se dejan fuera de la comparación. La prueba hace falta
porque en Windows el driver no da un error al llenarse la VRAM: pasa a usar la RAM del equipo y el
entrenamiento se vuelve decenas de veces más lento.

In [ ]:
import time

EPOCAS_EXPERIMENTO = 35
PACIENCIA_EXPERIMENTO = 9

# Cada entrada cambia UN solo factor respecto del modelo base
experimentos = [
    {"nombre": "R1 - escala de grises", "representacion": "gris"},
    {"nombre": "E1 - 3 bloques (32,64,128)", "filtros": (32, 64, 128)},
    {"nombre": "E2 - 2 convoluciones por bloque", "convoluciones": 2},
    {"nombre": "E3 - activación tanh", "activacion": "tanh"},
    {"nombre": "E4 - learning rate 1e-4", "tasa_aprendizaje": 1e-4},
    {"nombre": "E5 - batch 512", "batch": 512},
    {"nombre": "E6 - dropout 0,5", "tasa_dropout": 0.5},
    {"nombre": "E7 - cabeza con aplanado", "cabeza": "aplanado"},
    {"nombre": "P1 - sin pesos por clase", "esquema_pesos": "ninguno"},
    {"nombre": "P2 - pesos balanceados", "esquema_pesos": "balanceado"},
    {"nombre": "S1 - sin suavizado", "suavizado": 0.0},
    {"nombre": "S2 - sin etiquetas de escena", "etiquetas_escena": False},
    {"nombre": "A1 - sin alteradas", "usar_alteradas": False},
]


def describir_configuracion(
    nombre, representacion, filtros, convoluciones, activacion, tasa_dropout, cabeza,
    tasa_aprendizaje, batch, esquema_pesos, suavizado, etiquetas_escena, usar_alteradas,
):
    """Columnas de la tabla de experimentos que describen una configuración."""
    return {
        "experimento": nombre,
        "representacion": representacion,
        "dimension_entrada": DIMENSIONES[representacion],
        "bloques": len(filtros),
        "filtros": " - ".join(str(n) for n in filtros),
        "convoluciones_por_bloque": convoluciones,
        "cabeza": cabeza,
        "activacion": activacion,
        "dropout": tasa_dropout,
        "learning_rate": tasa_aprendizaje,
        "batch": batch,
        "pesos": esquema_pesos,
        "suavizado": suavizado,
        "etiquetas_escena": etiquetas_escena,
        "alteradas": usar_alteradas,
    }


def entrenar_configuracion(
    nombre,
    representacion=REPRESENTACION_BASE,
    filtros=FILTROS_POR_BLOQUE,
    convoluciones=CONVOLUCIONES_POR_BLOQUE,
    activacion=ACTIVACION,
    tasa_dropout=TASA_DROPOUT,
    cabeza=CABEZA,
    tasa_aprendizaje=TASA_APRENDIZAJE,
    batch=TAMANO_BATCH,
    epocas=EPOCAS_EXPERIMENTO,
    paciencia=PACIENCIA_EXPERIMENTO,
    esquema_pesos=ESQUEMA_PESOS,
    suavizado=SUAVIZADO_ETIQUETAS,
    etiquetas_escena=True,
    usar_alteradas=True,
):
    """Entrena una configuración y devuelve el modelo, su historial y sus métricas de validación."""
    keras.utils.set_random_seed(SEMILLA)  # misma inicialización para todas las configuraciones

    identificador = nombre.split(" - ")[0].strip().lower().replace(" ", "_")
    modelo = construir_cnn(
        filtros_por_bloque=filtros,
        convoluciones_por_bloque=convoluciones,
        activacion=activacion,
        tasa_dropout=tasa_dropout,
        cabeza=cabeza,
        representacion=representacion,
        nombre=f"cnn_{identificador}",
    )
    modelo = modelo.to(DISPOSITIVO)
    compilar(modelo, tasa_aprendizaje=tasa_aprendizaje)  # solo para predict()

    X_entrena = X_train
    y_entrena = y_train if etiquetas_escena else y_train_duro
    if not usar_alteradas:
        X_entrena, y_entrena = X_entrena[filas_originales_train], y_entrena[filas_originales_train]

    inicio = time.perf_counter()
    historial = entrenar(
        modelo,
        X_entrena,
        y_entrena,
        X_val,
        y_val,
        epocas=epocas,
        batch=batch,
        tasa_aprendizaje=tasa_aprendizaje,
        pesos_por_clase=PESOS_POR_ESQUEMA[esquema_pesos],
        suavizado=suavizado,
        paciencia=paciencia,
        ruta_modelo=DIRECTORIO_MODELOS / f"cnn_{identificador}.keras",
        verbose=0,
    )
    duracion = time.perf_counter() - inicio

    indice = int(np.argmin(historial.history["val_loss"]))
    predicciones = modelo.predict(X_val, batch_size=256, verbose=0).argmax(axis=1)

    resultado = {
        **describir_configuracion(
            nombre, representacion, filtros, convoluciones, activacion, tasa_dropout, cabeza,
            tasa_aprendizaje, batch, esquema_pesos, suavizado, etiquetas_escena, usar_alteradas,
        ),
        "epocas": len(historial.history["loss"]),
        "mejor_epoca": indice + 1,
        "parametros": int(modelo.count_params()),
        "val_loss": round(historial.history["val_loss"][indice], 4),
        "val_accuracy": round(historial.history["val_accuracy"][indice], 4),
        "val_f1_macro": round(float(f1_score(y_val_idx, predicciones, average="macro")), 4),
        "segundos": round(duracion, 1),
    }
    return modelo, historial, resultado


modelos = {}
historiales = {}
resultados = []

# El modelo base ya está entrenado: se incorpora a la comparación sin volver a entrenarlo
NOMBRE_BASE = "Base - rgb, 4 bloques (32,64,128,256)"
resultados.append(
    {
        **describir_configuracion(
            NOMBRE_BASE, REPRESENTACION_BASE, FILTROS_POR_BLOQUE, CONVOLUCIONES_POR_BLOQUE, ACTIVACION,
            TASA_DROPOUT, CABEZA, TASA_APRENDIZAJE, TAMANO_BATCH, ESQUEMA_PESOS, SUAVIZADO_ETIQUETAS,
            True, True,
        ),
        "epocas": len(historial_base.history["loss"]),
        "mejor_epoca": indice_optimo + 1,
        "parametros": int(modelo_base.count_params()),
        "val_loss": round(registro["val_loss"][indice_optimo], 4),
        "val_accuracy": round(registro["val_accuracy"][indice_optimo], 4),
        "val_f1_macro": round(float(f1_macro_val), 4),
        "segundos": np.nan,
    }
)
modelos[NOMBRE_BASE] = modelo_base
historiales[NOMBRE_BASE] = historial_base.history


def cabe_en_la_gpu(factores):
    """Estima la memoria que usará un paso de entrenamiento de la configuración y comprueba que cabe en
    la tarjeta. En Windows el driver no da un error cuando la VRAM se llena: pasa a usar la RAM del
    equipo y el entrenamiento se vuelve decenas de veces más lento. Por eso no se prueba el batch
    completo, sino dos lotes pequeños que siempre caben, y se extrapola: la memoria de las
    activaciones crece en línea recta con el tamaño del lote."""
    if DISPOSITIVO != "cuda":
        return True
    keras.utils.set_random_seed(SEMILLA)
    modelo = construir_cnn(
        filtros_por_bloque=factores.get("filtros", FILTROS_POR_BLOQUE),
        convoluciones_por_bloque=factores.get("convoluciones", CONVOLUCIONES_POR_BLOQUE),
        cabeza=factores.get("cabeza", CABEZA),
        representacion=factores.get("representacion", REPRESENTACION_BASE),
        nombre="cnn_prueba_memoria",
    ).to(DISPOSITIVO)
    picos = {}
    for lote in (8, 32):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        probabilidades = modelo(X_train[:lote], training=True)
        perdida = -(y_train[:lote] * torch.log(probabilidades.clamp_min(1e-7))).sum(1).mean()
        perdida.backward()
        picos[lote] = torch.cuda.max_memory_allocated()
        modelo.zero_grad(set_to_none=True)
        del probabilidades, perdida
    del modelo
    torch.cuda.empty_cache()
    batch = factores.get("batch", TAMANO_BATCH)
    por_imagen = (picos[32] - picos[8]) / (32 - 8)
    estimada = picos[32] + por_imagen * (batch - 32)
    return estimada < 0.9 * torch.cuda.get_device_properties(0).total_memory


def registrar(nombre, factores):
    """Entrena una configuración y la añade a la comparación. Devuelve su resultado, o None si
    no cabe en la memoria de la GPU."""
    print(f"Entrenando {nombre} ...")
    if not cabe_en_la_gpu(factores):
        print("   no cabe en la memoria de la GPU a esta resolucion: queda fuera de la comparacion")
        return None
    try:
        modelo, historial, resultado = entrenar_configuracion(nombre, **factores)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("   sin memoria suficiente en la GPU: la configuracion queda fuera de la comparacion")
        return None
    modelos[nombre] = modelo
    historiales[nombre] = historial.history
    resultados.append(resultado)
    print(
        f"   val_accuracy={resultado['val_accuracy']:.4f}"
        f"   val_f1_macro={resultado['val_f1_macro']:.4f}"
        f"   epocas={resultado['epocas']}   {resultado['segundos']:.0f} s"
    )
    return resultado


resultado_por_experimento = {}
for configuracion in experimentos:
    resultado_por_experimento[configuracion["nombre"]] = registrar(
        configuracion["nombre"],
        {factor: valor for factor, valor in configuracion.items() if factor != "nombre"},
    )

# C1: para cada factor, el valor del experimento que más superó a la base en F1 macro
f1_macro_base = resultados[0]["val_f1_macro"]
mejor_por_factor = {}
for configuracion in experimentos:
    resultado = resultado_por_experimento[configuracion["nombre"]]
    if resultado is None or resultado["val_f1_macro"] <= f1_macro_base:
        continue
    for factor, valor in configuracion.items():
        if factor == "nombre":
            continue
        if factor not in mejor_por_factor or resultado["val_f1_macro"] > mejor_por_factor[factor][1]:
            mejor_por_factor[factor] = (valor, resultado["val_f1_macro"], configuracion["nombre"])

componentes_combinacion = sorted({origen for _, _, origen in mejor_por_factor.values()})
if len(mejor_por_factor) >= 2:
    print(f"\nC1 combina: {' + '.join(componentes_combinacion)}")
    registrar(
        "C1 - combinación de mejoras",
        {factor: valor for factor, (valor, _, _) in mejor_por_factor.items()},
    )
else:
    print(
        f"\nC1 no se entrena: solo {len(mejor_por_factor)} factor(es) superaron a la base"
        f" ({', '.join(componentes_combinacion) or 'ninguno'}), así que no hay nada que combinar."
    )

tabla_experimentos = pd.DataFrame(resultados)
display(tabla_experimentos)

In [ ]:
# Comparación visual de las configuraciones por accuracy y F1 macro de validación
orden_resultados = tabla_experimentos.sort_values("val_f1_macro", ascending=True)
posiciones = np.arange(len(orden_resultados))

figura, ejes = plt.subplots(1, 2, figsize=(14, 6))

ejes[0].barh(posiciones - 0.2, orden_resultados["val_accuracy"], height=0.4, label="accuracy val")
ejes[0].barh(posiciones + 0.2, orden_resultados["val_f1_macro"], height=0.4, label="F1 macro val")
ejes[0].set_yticks(posiciones)
ejes[0].set_yticklabels(orden_resultados["experimento"], fontsize=8)
ejes[0].axvline(baseline_mayoritaria / 100, color="red", linestyle="--", linewidth=1, label="baseline clase mayoritaria")
ejes[0].set_xlabel("Métrica de validación")
ejes[0].set_title("Desempeño por configuración", fontsize=10)
ejes[0].legend(fontsize=8)
ejes[0].grid(alpha=0.3, axis="x")

for nombre, registro_historial in historiales.items():
    ejes[1].plot(
        range(1, len(registro_historial["val_loss"]) + 1),
        registro_historial["val_loss"],
        label=nombre,
        linewidth=1.8,
    )
ejes[1].set_xlabel("Epoca")
ejes[1].set_ylabel("Pérdida de validación")
ejes[1].set_title("Convergencia comparada", fontsize=10)
ejes[1].grid(alpha=0.3)
ejes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

### Efecto de la representación de entrada

In [ ]:
# Solo las configuraciones que cambian la representación (misma arquitectura que el base)
comparacion_representacion = tabla_experimentos[
    tabla_experimentos["experimento"].str.match(r"Base|R\d")
][["experimento", "representacion", "dimension_entrada", "parametros", "val_accuracy", "val_f1_macro", "epocas", "segundos"]]
comparacion_representacion = comparacion_representacion.sort_values(
    "val_f1_macro", ascending=False
).reset_index(drop=True)

display(comparacion_representacion)

figura, eje = plt.subplots(figsize=(8, 3.6))
posiciones = np.arange(len(comparacion_representacion))
eje.barh(posiciones - 0.2, comparacion_representacion["val_accuracy"], height=0.4, label="accuracy val")
eje.barh(posiciones + 0.2, comparacion_representacion["val_f1_macro"], height=0.4, label="F1 macro val")
eje.set_yticks(posiciones)
eje.set_yticklabels(
    [
        f"{fila.representacion} ({fila.dimension_entrada:,} valores)"
        for fila in comparacion_representacion.itertuples()
    ],
    fontsize=9,
)
eje.invert_yaxis()
eje.axvline(
    baseline_mayoritaria / 100,
    color="red",
    linestyle="--",
    linewidth=1,
    label="baseline clase mayoritaria",
)
eje.set_xlabel("Métrica de validación")
eje.set_title("Cuánto aporta el color a la CNN", fontsize=10)
eje.grid(alpha=0.3, axis="x")
eje.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Lectura.** El color sigue siendo decisivo, también para una CNN. Sin él (R1, escala de grises), el F1
macro de validación cae de 0,692 a 0,494 —casi 20 puntos— y la accuracy, de 0,764 a 0,600, con la misma
arquitectura y prácticamente los mismos parámetros. Es la mayor caída de toda la experimentación.

La comparación con el MLP es reveladora. En la corrida de GPU a 64 × 64, el MLP con solo color (R2) llegaba a
un F1 macro de validación de 0,477; la CNN en escala de grises, sin ningún color, llega a 0,494. **La forma
sola rinde lo mismo que el color solo**, y la CNN con imagen RGB, que aprovecha las dos cosas, sube a 0,692.
Lo que distingue a estos personajes es la combinación: la paleta de cada uno y la silueta, el peinado y la
ropa que la dibujan.

### Efecto de la profundidad de la red

In [ ]:
# Solo las configuraciones que cambian la profundidad (misma activación, lr y batch que el base)
comparacion_arquitectura = tabla_experimentos[
    tabla_experimentos["experimento"].str.contains("Base|E1|E2")
][["experimento", "bloques", "filtros", "convoluciones_por_bloque", "parametros", "val_accuracy", "val_f1_macro", "epocas", "segundos"]]
comparacion_arquitectura = comparacion_arquitectura.sort_values("parametros").reset_index(drop=True)

comparacion_arquitectura["capas_convolucionales"] = (
    comparacion_arquitectura["bloques"] * comparacion_arquitectura["convoluciones_por_bloque"]
)
comparacion_arquitectura["parametros_millones"] = (
    comparacion_arquitectura["parametros"] / 1e6
).round(3)

display(comparacion_arquitectura)

figura, eje = plt.subplots(figsize=(7, 4.5))
eje.plot(
    comparacion_arquitectura["parametros_millones"],
    comparacion_arquitectura["val_f1_macro"],
    marker="o",
    linewidth=2,
    label="F1 macro val",
)
eje.plot(
    comparacion_arquitectura["parametros_millones"],
    comparacion_arquitectura["val_accuracy"],
    marker="s",
    linewidth=2,
    label="accuracy val",
)
for _, fila in comparacion_arquitectura.iterrows():
    eje.annotate(
        f"{fila['capas_convolucionales']} conv ({fila['filtros']})",
        (fila["parametros_millones"], fila["val_f1_macro"]),
        textcoords="offset points",
        xytext=(6, -12),
        fontsize=8,
    )
eje.set_xlabel("Parámetros entrenables (millones)")
eje.set_ylabel("Métrica de validación")
eje.set_title("Profundidad de la CNN frente a desempeño", fontsize=10)
eje.grid(alpha=0.3)
eje.legend()
plt.tight_layout()
plt.show()

**Lectura.** En la CNN, más profundidad sí ayuda. E2, con dos convoluciones por bloque (ocho capas
convolucionales), es la única configuración que supera a la base: F1 macro de validación de 0,717 frente a
0,692 y accuracy de 0,786 frente a 0,764, y lo consigue con 35 épocas de presupuesto contra las 60 de la
base. Un bloque menos (E1) queda por debajo (0,664): con tres bloques, las neuronas del último ven zonas de
18 × 18 píxeles, menos de un tercio del lado de la imagen, en lugar de 38 × 38.

La profundidad se paga. E2 tiene 2,7 veces los parámetros de la base (1,26 millones frente a 473.000) y cada
época le cuesta el doble (21,3 s frente a 10,6 s). Es un patrón parecido al del MLP, donde la tercera capa daba
de 2 a 3 puntos de F1 macro a cambio de más que duplicar los parámetros, pero aquí parte de una base mucho más
alta.

### Efecto de la función de activación

In [ ]:
comparacion_activacion = tabla_experimentos[
    tabla_experimentos["experimento"].str.contains("Base|E3")
][["experimento", "activacion", "val_loss", "val_accuracy", "val_f1_macro", "epocas", "mejor_epoca"]]

display(comparacion_activacion.reset_index(drop=True))

figura, ejes = plt.subplots(1, 2, figsize=(12, 4.5))

for nombre in comparacion_activacion["experimento"]:
    registro_historial = historiales[nombre]
    epocas_rango = range(1, len(registro_historial["loss"]) + 1)
    ejes[0].plot(epocas_rango, registro_historial["val_loss"], linewidth=2, label=nombre)
    ejes[1].plot(epocas_rango, registro_historial["val_accuracy"], linewidth=2, label=nombre)

ejes[0].set_ylabel("Pérdida de validación")
ejes[0].set_title("Convergencia: ReLU frente a tanh", fontsize=10)
ejes[1].set_ylabel("Accuracy de validación")
ejes[1].set_title("Accuracy de validación por época", fontsize=10)

for eje in ejes:
    eje.set_xlabel("Epoca")
    eje.grid(alpha=0.3)
    eje.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Lectura.** ReLU gana con más claridad que en el MLP. Con tanh (E3) el F1 macro de validación cae 14,5
puntos (0,547 frente a 0,692) y la accuracy, de 0,764 a 0,645; en el MLP la caída era de 5 a 6 puntos. Con
más capas apiladas, la saturación de tanh pesa más, porque el gradiente se atenúa en cada una que atraviesa.
E3 tiene además su mejor época en la última del presupuesto (35): sigue mejorando, pero muy despacio.

En el recorrido de una imagen por la red entrenada (sección «El modelo en acción sobre el conjunto de
test»), ReLU deja distintos de cero entre el 48 % de los valores del primer bloque y el 16 % del último: la
representación es dispersa, y más selectiva cuanto más profundo es el bloque.

### Efecto del número de épocas, el tamaño de batch y la tasa de aprendizaje

In [ ]:
comparacion_optimizacion = tabla_experimentos[
    tabla_experimentos["experimento"].str.contains("Base|E4|E5")
][["experimento", "learning_rate", "batch", "epocas", "mejor_epoca", "val_loss", "val_accuracy", "val_f1_macro", "segundos"]]

display(comparacion_optimizacion.reset_index(drop=True))

figura, ejes = plt.subplots(1, 2, figsize=(12, 4.5))

for nombre in comparacion_optimizacion["experimento"]:
    registro_historial = historiales[nombre]
    epocas_rango = range(1, len(registro_historial["loss"]) + 1)
    ejes[0].plot(epocas_rango, registro_historial["loss"], linewidth=1.6, linestyle="--")
    ejes[0].plot(
        epocas_rango,
        registro_historial["val_loss"],
        linewidth=2,
        label=nombre,
        color=ejes[0].lines[-1].get_color(),
    )
    ejes[1].plot(
        epocas_rango,
        np.array(registro_historial["accuracy"]) - np.array(registro_historial["val_accuracy"]),
        linewidth=2,
        label=nombre,
    )

ejes[0].set_ylabel("Perdida")
ejes[0].set_title("Entrenamiento (punteado) y validación (sólido)", fontsize=10)
ejes[1].axhline(0, color="gray", linewidth=0.8)
ejes[1].set_ylabel("accuracy train - accuracy val")
ejes[1].set_title("Brecha de generalización por época", fontsize=10)

for eje in ejes:
    eje.set_xlabel("Epoca")
    eje.grid(alpha=0.3)
    eje.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Lectura.** Los tres hiperparámetros de optimización están acoplados y conviene leerlos juntos:

- **Tasa de aprendizaje.** Con 1e-4 (E4) el F1 macro de validación cae 17,8 puntos (0,513 frente a 0,692):
  con un paso diez veces menor, 35 épocas no alcanzan. En el MLP la caída era de 4 a 5 puntos; una red con
  más capas tarda mucho más en converger con un paso pequeño. La base, en cambio, aprovecha bajar la tasa en
  el momento justo: el primer escalón de `ReduceLROnPlateau` (época 14) sube la accuracy de validación de
  0,58 a 0,70.
- **Tamaño de batch.** Con batch 512 (E5) el F1 macro baja 2,9 puntos (0,662) y casi no se gana tiempo: 10,0 s
  por época frente a 10,6. En el MLP, batch 512 era cuatro veces más barato porque la GPU pasaba la mayor
  parte del tiempo esperando; con la CNN la tarjeta ya trabaja el 91 % del tiempo a batch 128 (ver «Resumen
  de rendimiento»), así que agrandar el lote solo reduce las actualizaciones por época.
- **Número de épocas.** Todos los experimentos usan las 35 épocas del presupuesto, y casi todos tienen su
  mejor época entre la 31 y la 35: seguían mejorando cuando se acabó. La base, con 60, toca su mínimo en la 52.
  Eso pone a los experimentos en desventaja: en la época 35 la base tenía una accuracy de validación de
  0,752, y S1, S2, P1 y A1 quedan en ese mismo nivel. Que E2 supere a la base con menos épocas hace su ventaja
  más sólida.

### Efecto de la cabeza: pooling global frente a aplanado

La cabeza decide qué parte del último mapa de activación llega a la capa densa. Con **pooling global** (el
modelo base) llega un número por filtro —cuánto se activó en toda la imagen— y la posición se pierde a
propósito: da igual si el pelo de Marge está arriba a la izquierda o al centro. Con **aplanado** (E7) llega
el mapa completo de 4 × 4 × 256, con la posición de cada activación, como entraban los píxeles al MLP: la
capa densa tiene 16 veces más pesos y puede aprender en qué zona del encuadre suele estar cada personaje.

In [ ]:
comparacion_cabeza = tabla_experimentos[
    tabla_experimentos["experimento"].str.match(r"Base|E7")
][["experimento", "cabeza", "parametros", "val_loss", "val_accuracy", "val_f1_macro", "epocas", "mejor_epoca"]]

display(comparacion_cabeza.reset_index(drop=True))

figura, ejes = plt.subplots(1, 2, figsize=(12, 4.5))

for nombre in comparacion_cabeza["experimento"]:
    registro_historial = historiales[nombre]
    epocas_rango = range(1, len(registro_historial["loss"]) + 1)
    ejes[0].plot(epocas_rango, registro_historial["val_loss"], linewidth=2, label=nombre)
    ejes[1].plot(
        epocas_rango,
        np.array(registro_historial["accuracy"]) - np.array(registro_historial["val_accuracy"]),
        linewidth=2,
        label=nombre,
    )

ejes[0].set_ylabel("Pérdida de validación")
ejes[0].set_title("Convergencia: pooling global frente a aplanado", fontsize=10)
ejes[1].axhline(0, color="gray", linewidth=0.8)
ejes[1].set_ylabel("accuracy train - accuracy val")
ejes[1].set_title("Brecha de generalización por época", fontsize=10)

for eje in ejes:
    eje.set_xlabel("Epoca")
    eje.grid(alpha=0.3)
    eje.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Lectura.** El pooling global gana. Con aplanado (E7) el F1 macro de validación baja 5 puntos (0,641 frente a
0,692) con tres veces los parámetros (1,46 millones), y la red memoriza antes: la brecha entre entrenamiento y
validación llega a 0,25 hacia la época 18, mientras la de la base no pasa de 0,22 en 60 épocas. Su mejor época
es la 29, la más temprana de todos los experimentos.

Es lo que se esperaba. Con el mapa completo, la capa densa puede aprender en qué zona del encuadre suele
estar cada personaje, una regularidad de las imágenes de entrenamiento y no del personaje. El pooling global
obliga a decidir por qué patrones aparecen y no por dónde: es la invarianza a la posición que el MLP sobre
píxeles no tenía.

### Efecto del tratamiento de los datos, la regularización y la combinación

Estos experimentos no tocan la arquitectura, sino lo que la red ve al entrenar: cuánto pesa cada
personaje (P1, P2), cuán duro es el objetivo (S1, S2), si entran las variantes alteradas (A1) y cuánto se
regulariza (E6). La columna `delta_f1_vs_base` es la diferencia de F1 macro de validación frente al modelo
base: positiva si el cambio ayuda. C1, si se entrenó, junta los cambios que salieron positivos.

Al leer la tabla conviene mirar también la accuracy: un cambio que sube el F1 macro a costa de bajar mucho
la accuracy está desplazando aciertos de los personajes frecuentes a los escasos, no reconociendo mejor.

In [ ]:
comparacion_datos = tabla_experimentos[
    tabla_experimentos["experimento"].str.match(r"Base|E6|P\d|S\d|A\d|C\d")
][
    ["experimento", "pesos", "suavizado", "etiquetas_escena", "alteradas", "dropout",
     "val_accuracy", "val_f1_macro", "epocas", "segundos"]
].reset_index(drop=True)
fila_base = comparacion_datos["experimento"].str.startswith("Base")
comparacion_datos["delta_accuracy_vs_base"] = (
    comparacion_datos["val_accuracy"] - float(comparacion_datos.loc[fila_base, "val_accuracy"].iloc[0])
).round(4)
comparacion_datos["delta_f1_vs_base"] = (
    comparacion_datos["val_f1_macro"] - float(comparacion_datos.loc[fila_base, "val_f1_macro"].iloc[0])
).round(4)
display(comparacion_datos)

figura, eje = plt.subplots(figsize=(9, 4.5))
otras = comparacion_datos[~fila_base].reset_index(drop=True)
posiciones = np.arange(len(otras))
eje.barh(posiciones - 0.2, otras["delta_accuracy_vs_base"], height=0.4, label="accuracy val")
eje.barh(posiciones + 0.2, otras["delta_f1_vs_base"], height=0.4, label="F1 macro val")
eje.set_yticks(posiciones)
eje.set_yticklabels(otras["experimento"], fontsize=8)
eje.invert_yaxis()
eje.axvline(0, color="gray", linewidth=1)
eje.set_xlabel("Diferencia frente al modelo base (validación)")
eje.set_title("Qué cambios del tratamiento de los datos ayudan", fontsize=10)
eje.grid(alpha=0.3, axis="x")
eje.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Lectura.** Ninguno de estos cambios mejora a la base:

- **Pesos por clase.** Sin pesos (P1) la accuracy de validación casi no cambia (0,756) pero el F1 macro baja
  2,3 puntos: los pesos ayudan a los personajes con pocas imágenes. Con los balanceados completos (P2) caen las
  dos, 6 puntos la accuracy y 5 el F1: castigar tanto a los frecuentes cuesta más de lo que se gana con los
  escasos. La raíz cuadrada sigue siendo el punto medio, como en el MLP.
- **Suavizado, escenas y alteradas.** Sin suavizado (S1), sin etiquetas de escena (S2) o sin alteradas (A1), el
  F1 macro baja entre 0,8 y 1,1 puntos (0,684, 0,683 y 0,681). Con 35 épocas de presupuesto frente a las 60 de
  la base, esas diferencias caben en lo que explica el presupuesto: no se puede afirmar que estos factores
  ayuden a la CNN, solo que quitarlos no la mejora.
- **Regularización.** Más dropout en la cabeza (E6, 0,5) baja el F1 macro 2,2 puntos. La brecha de la base es
  grande (0,21), pero más dropout en la parte densa no la reduce sin perder desempeño; atacarla requeriría
  regularizar las capas convolucionales o dar más variedad a los datos (ver «Propuestas de mejora»).

Como solo E2 superó a la base, **C1 no se entrena**: no hay dos mejoras que combinar.

### Selección del modelo final

In [ ]:
import json

# El criterio de selección es el F1 macro de validación, no la accuracy:
# promedia todas las clases sin ponderar y penaliza ignorar a los personajes minoritarios
ranking = tabla_experimentos.sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
display(
    ranking[
        ["experimento", "representacion", "filtros", "convoluciones_por_bloque", "cabeza",
         "val_accuracy", "val_f1_macro", "val_loss", "parametros", "epocas"]
    ]
)

nombre_final = ranking.loc[0, "experimento"]
modelo_final = modelos[nombre_final]
representacion_final = ranking.loc[0, "representacion"]
arquitectura_final = (
    f"{ranking.loc[0, 'filtros']} filtros, {ranking.loc[0, 'convoluciones_por_bloque']} conv. por bloque,"
    f" {ranking.loc[0, 'cabeza']}"
)

print(f"Configuracion seleccionada: {nombre_final}")
print(f"  entrada             : imagen {ALTO_OBJETIVO}x{ANCHO_OBJETIVO} ({representacion_final})")
print(f"  arquitectura        : {arquitectura_final}")
print(f"  accuracy validacion : {ranking.loc[0, 'val_accuracy']:.4f}")
print(f"  F1 macro validacion : {ranking.loc[0, 'val_f1_macro']:.4f}")
print(f"  parametros          : {ranking.loc[0, 'parametros']:,}")
print(f"\nDiferencia en F1 macro frente a la segunda ({ranking.loc[1, 'experimento']}): {ranking.loc[0, 'val_f1_macro'] - ranking.loc[1, 'val_f1_macro']:+.4f}")

ruta_modelo_final = DIRECTORIO_MODELOS / "cnn_final.keras"
modelo_final.save(ruta_modelo_final)

# Lo que la utilidad de predicción necesita para repetir el pre-procesamiento sin volver
# a ejecutar el notebook: la resolución y el orden de las clases. La escala de grises, si
# el modelo la usa, va dentro del modelo; la entrada se guarda solo como referencia.
metadatos_final = {
    "configuracion": nombre_final,
    "representacion": representacion_final,
    "arquitectura": arquitectura_final,
    "alto": ALTO_OBJETIVO,
    "ancho": ANCHO_OBJETIVO,
    "clases": clases,
}
ruta_metadatos_final = DIRECTORIO_MODELOS / "cnn_final.json"
ruta_metadatos_final.write_text(json.dumps(metadatos_final, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"\nModelo final guardado en: {ruta_modelo_final}")
print(f"Pre-procesamiento del modelo final en: {ruta_metadatos_final}")

**Justificación de la decisión.** La configuración elegida es la que maximiza el **F1 macro de
validación**, y no la que maximiza la accuracy. Validación conserva la distribución real del dataset, con
una razón de desbalance de 33:1: la accuracy premia acertar en los personajes abundantes y apenas se
resiente si el modelo ignora a los escasos; el F1 macro, al promediar los 64 personajes sin ponderarlos,
sí lo penaliza. Como el objetivo declarado es reconocer personajes —los 64, no solo a Homero—, esa es la
métrica coherente con el problema.

En la corrida de GPU a 64 × 64 del 19-10-2026 gana **E2**, la CNN de cuatro bloques con dos convoluciones de
3 × 3 cada uno: F1 macro de validación de 0,717, 2,5 puntos por encima de la base (0,692), y también la mayor
accuracy (0,786). Es la única que supera a la base, así que los dos criterios coinciden y no hay combinación
C1. La diferencia es mayor que la variación de alrededor de un punto que el MLP mostró entre corridas que
solo cambiaban el dispositivo, y E2 la consigue con 35 épocas frente a las 60 de la base. El costo es una red
con 2,7 veces los parámetros y el doble de tiempo por época.

El modelo seleccionado es el que se evalúa en la sección siguiente sobre el **conjunto de test**, que hasta
este punto no se ha tocado ni para entrenar ni para elegir hiperparámetros.

## Evaluación y métricas de desempeño

El conjunto de evaluación es el **test** reservado en la partición: el 10 % de las imágenes originales de
cada personaje, con la distribución real del dataset y en las mismas condiciones que el entrenamiento: las
imágenes casi idénticas a otra entran con su versión filtrada y un cuarto de las originales viene con su
variante alterada. El modelo nunca vio esas imágenes ni ninguna variante de ellas.

**Criterio de acierto.** Una predicción es correcta si nombra al personaje de la carpeta o a otro personaje
que también aparece en la imagen, según la escena y las gemelas (sección «Escenas con varios personajes») o
según la verificación por imagen de los errores con confianza alta (sección siguiente). Con ese criterio se calculan la
accuracy, el top-3, el F1, el reporte por clase, la matriz de confusión y los KPIs; la accuracy estricta,
que solo acepta el personaje de la carpeta, se muestra al lado como referencia. Las galerías usan tres
colores: verde si el modelo nombra al de la carpeta, ámbar si nombra a otro personaje presente y rojo si se
equivoca.

Ya está cargado y procesado con **exactamente las mismas funciones** que el entrenamiento —el mismo
redimensionado a 64 x 64 y el mismo diccionario `clase_a_indice`—, porque las tres particiones salieron
del mismo índice. La celda siguiente comprueba que las imágenes de test son coherentes con las del
entrenamiento.

In [ ]:
# El test sale de la partición: las mismas imágenes de alto x ancho x 3 que entrenamiento y
# validación. La escala de grises, si el modelo final la usa, va dentro del modelo.
print(f"Modelo final: {nombre_final}   |   entrada: {representacion_final}")

X_test = PARTICION["test"]

# Cada alterada de test acompaña a su original, que también está en test
originales_test = set(
    zip(datos_test.loc[~datos_test["alterada"], "personaje"], datos_test.loc[~datos_test["alterada"], "origen"])
)
alteradas_test = datos_test[datos_test["alterada"]]
alteradas_test_con_original = sum(
    (personaje, origen) in originales_test
    for personaje, origen in zip(alteradas_test["personaje"], alteradas_test["origen"])
)

comprobaciones_test = pd.DataFrame(
    [
        (
            "Todas las clases presentes",
            len(np.unique(y_test_idx)),
            NUMERO_CLASES,
            len(np.unique(y_test_idx)) == NUMERO_CLASES,
        ),
        (
            "Forma de las imágenes de test",
            str(X_test.shape[1:]),
            str(FORMA_ENTRADA),
            X_test.shape[1:] == FORMA_ENTRADA,
        ),
        (
            "Alteradas de test con su original en test",
            alteradas_test_con_original,
            len(alteradas_test),
            alteradas_test_con_original == len(alteradas_test),
        ),
        ("Etiquetas alineadas", len(y_test_idx), X_test.shape[0], len(y_test_idx) == X_test.shape[0]),
        ("Imágenes alineadas", len(imagenes_test), X_test.shape[0], len(imagenes_test) == X_test.shape[0]),
    ],
    columns=["Comprobacion", "Obtenido", "Esperado", "Correcto"],
)

display(comprobaciones_test)

if not comprobaciones_test["Correcto"].all():
    raise ValueError("El conjunto de test no cumple alguna de las comprobaciones.")

print(f"Conjunto de test: {len(datos_test)} imagenes")
print(f"X_test: {X_test.shape}   y_test: {y_test.shape}")

# Como tensores de PyTorch, igual que train y validación
X_test = a_tensor(X_test)
y_test = a_tensor(y_test)
print(f"X_test en {X_test.device}")

### Verificación de los errores con confianza alta

La escena del episodio (±3 s) y las gemelas no encuentran a todos los personajes presentes: una imagen de
Kaggle no trae episodio ni instante, y dos tomas del mismo plano pueden quedar a más de tres segundos o
parecerse menos que una gemela. Por eso, antes de calcular las métricas, cada **error cometido con
confianza mayor que `UMBRAL_CONFIANZA_VERIFICACION` (0,80)** pasa por una verificación: ¿se equivocó el
modelo, o nombró a otro personaje que sí sale en la imagen?

**Cómo se verifica.** La imagen se compara con todas las de la carpeta del personaje que nombró el
modelo, con las mismas miniaturas de 16 × 16 a color con que el script de alteraciones busca las
gemelas. Si alguna supera una similitud de `UMBRAL_SIMILITUD_VERIFICACION` (0,80), es la misma toma: ese
personaje está en la imagen y la predicción cuenta como **acierto por escena**. Si ninguna la supera, es
un **error real**.

- **Por qué 0,80.** En una muestra de 4.087 fotogramas de 120 episodios, de los 8,08 millones de pares de
  personajes distintos y episodios distintos (que no pueden ser la misma toma) solo 11 superan 0,80. En
  cambio, lo superan el 39 % de los pares del mismo episodio separados por menos de 3 s y el 13 % de los
  separados por 3 a 15 s. Contra una carpeta de unas 3.000 imágenes, la probabilidad de dar por
  verificado un caso que no lo es ronda el 0,4 %. La galería de parejas permite revisarlo a ojo.
- **Qué imagen se compara.** La original de la imagen de test: las parecidas y las alteradas entran al
  modelo con un filtro que cambia su miniatura, pero muestran la misma escena.
- **Contra qué se compara.** Las originales intactas de la carpeta del predicho, de cualquier conjunto.
  Las casi idénticas a otra (parecidas) quedan representadas por la imagen de su grupo que se conserva
  intacta. Igual que con la escena de la evaluación, usar imágenes de entrenamiento no es una fuga: la
  verificación se hace después de predecir y no cambia el modelo.
- **Solo los errores confiados.** Se verifican los errores que el modelo da por seguros (confianza mayor
  que 0,80); los demás se cuentan como errores sin más.


In [ ]:
# Verificación de los errores con confianza alta: cuando el modelo nombra con seguridad a un personaje
# que no es el de la carpeta ni sale en su escena, se busca la misma toma en la carpeta de ese personaje
UMBRAL_CONFIANZA_VERIFICACION = 0.80
UMBRAL_SIMILITUD_VERIFICACION = 0.80
# Miniaturas como las del script de alteraciones: 16 x 16 a color
LADO_MINIATURA = 16
TAMANO_BLOQUE_MINIATURAS = 4096

probabilidades_test = modelo_final.predict(X_test, batch_size=256, verbose=0)
predicciones_test = probabilidades_test.argmax(axis=1)
confianza_test = probabilidades_test.max(axis=1)
# Personajes presentes según la escena del episodio y las gemelas; la verificación añade los suyos
acompanantes_test = [set(texto.split(", ")) if texto else set() for texto in datos_test["escena"]]

# Colores de las galerías de evaluación y de análisis de errores
COLOR_ACIERTO, COLOR_ESCENA, COLOR_ERROR = "#1b7f3b", "#d68910", "#c0392b"
TEXTO_RESULTADO = {COLOR_ACIERTO: "correcto", COLOR_ESCENA: "acierto por escena", COLOR_ERROR: "incorrecto"}


def miniaturas_de(lote):
    """Miniatura con la que el script de alteraciones compara las imágenes: la imagen de trabajo
    promediada por bloques hasta 16 x 16 a color, centrada y de norma 1. El producto de dos
    miniaturas es su similitud coseno."""
    paso = lote.shape[1] // LADO_MINIATURA
    vectores = (
        lote.reshape(len(lote), LADO_MINIATURA, paso, LADO_MINIATURA, paso, 3)
        .mean(axis=(2, 4), dtype=np.float32)
        .reshape(len(lote), -1)
    )
    vectores -= vectores.mean(axis=1, keepdims=True)
    return vectores / np.maximum(np.linalg.norm(vectores, axis=1, keepdims=True), 1e-6)


# Errores según la escena y las gemelas, cometidos con confianza alta
en_escena_test = np.array(
    [clases[predicho] in presentes for predicho, presentes in zip(predicciones_test, acompanantes_test)]
)
a_verificar = np.flatnonzero(
    (predicciones_test != y_test_idx) & ~en_escena_test & (confianza_test > UMBRAL_CONFIANZA_VERIFICACION)
)

COLUMNAS_VERIFICACION = [
    "indice", "carpeta", "predicho", "confianza", "similitud", "coincidencia", "conjunto_coincidencia",
    "fila_coincidencia", "otro_personaje_en_imagen",
]
filas_verificacion = []
if len(a_verificar):
    # La original de cada caso: las parecidas y las alteradas llevan un filtro que cambia la miniatura
    casos = datos_test.iloc[a_verificar]
    miniaturas_casos = miniaturas_de(
        np.stack(
            [
                cargar_y_redimensionar(DATASET_BASE / personaje / origen)
                for personaje, origen in zip(casos["personaje"], casos["origen"])
            ]
        )
    )

    # Candidatas: las originales intactas, de cualquier conjunto, de los personajes que nombró el modelo
    etiqueta_datos = datos["etiqueta"].to_numpy()
    candidatas = np.flatnonzero(
        ~datos["alterada"].to_numpy()
        & ~datos["parecida"].to_numpy()
        & np.isin(etiqueta_datos, predicciones_test[a_verificar])
    )
    miniaturas_candidatas = np.concatenate(
        [
            miniaturas_de(imagenes[candidatas[desde:desde + TAMANO_BLOQUE_MINIATURAS]])
            for desde in range(0, len(candidatas), TAMANO_BLOQUE_MINIATURAS)
        ]
    )
    etiqueta_candidatas = etiqueta_datos[candidatas]

    for miniatura_caso, indice in zip(miniaturas_casos, a_verificar):
        predicho = int(predicciones_test[indice])
        de_su_clase = np.flatnonzero(etiqueta_candidatas == predicho)
        similitudes = miniaturas_candidatas[de_su_clase] @ miniatura_caso
        mejor = int(np.argmax(similitudes))
        fila = int(candidatas[de_su_clase[mejor]])
        filas_verificacion.append(
            {
                "indice": int(indice),
                "carpeta": clases[y_test_idx[indice]],
                "predicho": clases[predicho],
                "confianza": round(float(confianza_test[indice]), 3),
                "similitud": round(float(similitudes[mejor]), 3),
                "coincidencia": datos["origen"].iloc[fila],
                "conjunto_coincidencia": datos["conjunto"].iloc[fila],
                "fila_coincidencia": fila,
                "otro_personaje_en_imagen": bool(similitudes[mejor] >= UMBRAL_SIMILITUD_VERIFICACION),
            }
        )

# Tipos explícitos: sin casos, la tabla vacía no sabría que la última columna es booleana
verificacion_test = pd.DataFrame(filas_verificacion, columns=COLUMNAS_VERIFICACION).astype(
    {"indice": int, "fila_coincidencia": int, "otro_personaje_en_imagen": bool}
)
verificados = verificacion_test[verificacion_test["otro_personaje_en_imagen"]]
errores_reales = verificacion_test[~verificacion_test["otro_personaje_en_imagen"]]

# Un caso verificado suma al predicho a los personajes presentes: desde aquí es un acierto por escena
verificado_test = np.zeros(len(y_test_idx), dtype=bool)
verificado_test[verificados["indice"].to_numpy(dtype=int)] = True
for indice in np.flatnonzero(verificado_test):
    acompanantes_test[indice].add(clases[predicciones_test[indice]])

resumen_verificacion = pd.DataFrame(
    [
        (
            f"Errores con confianza > {UMBRAL_CONFIANZA_VERIFICACION:.2f} (según escena y gemelas)",
            len(verificacion_test),
            f"{len(verificacion_test) / len(y_test_idx):.1%} del test",
        ),
        (
            f"   con la misma toma en la carpeta del predicho (similitud >= {UMBRAL_SIMILITUD_VERIFICACION:.2f})",
            len(verificados),
            f"{len(verificados) / max(len(verificacion_test), 1):.1%}: otro personaje en la imagen, acierto por escena",
        ),
        ("   sin la misma toma en esa carpeta", len(errores_reales), "errores reales"),
    ],
    columns=["Indicador", "Valor", "Detalle"],
)
display(resumen_verificacion)

if verificacion_test.empty:
    print(f"No hay errores con confianza > {UMBRAL_CONFIANZA_VERIFICACION:.2f} que verificar.")

if len(verificados):
    display(verificados[["carpeta", "predicho"]].value_counts().head(10).reset_index(name="casos"))

    # Parejas imagen de test | imagen de la misma toma en la carpeta del personaje que nombró el modelo
    PAREJAS_POR_FILA = 3
    muestra_verificados = verificados.sample(min(12, len(verificados)), random_state=SEMILLA)
    filas_figura = int(np.ceil(len(muestra_verificados) / PAREJAS_POR_FILA))
    figura, ejes = plt.subplots(
        filas_figura,
        2 * PAREJAS_POR_FILA,
        figsize=(2.0 * 2 * PAREJAS_POR_FILA, 3.0 * filas_figura),
        layout="constrained",
        squeeze=False,
    )
    parejas = ejes.reshape(-1, 2)
    for (eje_test, eje_coincidencia), caso in zip(parejas, muestra_verificados.itertuples()):
        paneles = (
            (
                eje_test,
                imagenes_test[caso.indice],
                f"test: {caso.carpeta.replace('_', ' ')}\npred: {caso.predicho.replace('_', ' ')}"
                f" ({caso.confianza:.2f})",
            ),
            (
                eje_coincidencia,
                imagenes[caso.fila_coincidencia],
                f"carpeta {caso.predicho.replace('_', ' ')}\n({caso.conjunto_coincidencia})"
                f"  similitud {caso.similitud:.2f}",
            ),
        )
        for eje, imagen, texto in paneles:
            eje.imshow(imagen)
            eje.set_xticks([])
            eje.set_yticks([])
            for lado in eje.spines.values():
                lado.set_edgecolor(COLOR_ESCENA)
                lado.set_linewidth(2.5)
            eje.set_xlabel(texto, fontsize=7, color="#9c640c", labelpad=5, linespacing=1.4)
    for eje in parejas[len(muestra_verificados):].ravel():
        eje.axis("off")
    figura.suptitle(
        "Errores confiados verificados: la misma toma está en la carpeta del personaje que nombró el modelo",
        fontsize=12,
        fontweight="bold",
    )
    plt.show()

if len(errores_reales):
    # Los errores confiados que la verificación no explica, de mayor a menor confianza
    mas_confiados = errores_reales.sort_values("confianza", ascending=False).head(10)
    figura, ejes = plt.subplots(2, 5, figsize=(11, 6.4), layout="constrained")
    ejes = np.ravel(ejes)
    for eje, caso in zip(ejes, mas_confiados.itertuples()):
        eje.imshow(imagenes_test[caso.indice])
        eje.set_xticks([])
        eje.set_yticks([])
        for lado in eje.spines.values():
            lado.set_edgecolor(COLOR_ERROR)
            lado.set_linewidth(2.5)
        eje.set_xlabel(
            f"real: {caso.carpeta.replace('_', ' ')}\npred: {caso.predicho.replace('_', ' ')} ({caso.confianza:.2f})"
            f"\nmejor similitud: {caso.similitud:.2f}",
            fontsize=7,
            color=COLOR_ERROR,
            labelpad=5,
            linespacing=1.4,
        )
    for eje in ejes[len(mas_confiados):]:
        eje.axis("off")
    figura.suptitle("Errores confiados sin la misma toma en la carpeta del predicho: errores reales", fontsize=12, fontweight="bold")
    plt.show()

print(
    f"Errores con confianza > {UMBRAL_CONFIANZA_VERIFICACION:.2f}: {len(verificacion_test)}"
    f"   |   otro personaje en la imagen: {len(verificados)}"
    f"   |   errores reales: {len(errores_reales)}"
)


### Accuracy global sobre el conjunto de prueba

In [ ]:
perdida_test, accuracy_estricta_test, top3_estricto_test = modelo_final.evaluate(
    X_test, y_test, batch_size=256, verbose=0
)

# Las predicciones y los personajes presentes salen de la verificación de los errores con confianza alta.
# Criterio de acierto: la predicción es correcta si nombra al personaje de la carpeta o a otro
# que también aparece en la imagen, según la escena del episodio, una gemela guardada en la
# carpeta de otro personaje (sección «Escenas con varios personajes») o la verificación por imagen.
# La etiqueta con la que se evalúa es la de la carpeta, salvo cuando el modelo nombra a un personaje
# presente: entonces es él.
acierto_por_escena = np.array(
    [clases[predicho] in presentes for predicho, presentes in zip(predicciones_test, acompanantes_test)]
)
y_test_eval = np.where(acierto_por_escena, predicciones_test, y_test_idx)
acierto_estricto = predicciones_test == y_test_idx
acierto_test = predicciones_test == y_test_eval
accuracy_test = float(acierto_test.mean())
accuracy_tolerante_test = accuracy_test
# Con solo la escena y las gemelas: comparable con las corridas sin verificación por imagen
accuracy_sin_verificacion_test = float((acierto_test & ~verificado_test).mean())

# Top-3 con el mismo criterio: basta con que el de la carpeta o un personaje presente esté entre los tres
clases_validas_test = [
    {int(real)} | {clase_a_indice[nombre] for nombre in presentes}
    for real, presentes in zip(y_test_idx, acompanantes_test)
]
top3_por_imagen = np.array(
    [
        bool(set(mejores.tolist()) & validas)
        for mejores, validas in zip(np.argsort(-probabilidades_test, axis=1)[:, :3], clases_validas_test)
    ]
)
top3_test = float(top3_por_imagen.mean())


def color_resultado(indice):
    """Verde si el modelo nombró al personaje de la carpeta, ámbar si nombró a otro personaje
    presente en la imagen y rojo si se equivocó."""
    if acierto_estricto[indice]:
        return COLOR_ACIERTO
    return COLOR_ESCENA if acierto_por_escena[indice] else COLOR_ERROR


def rotulo_escena(indice):
    """Rótulo de un acierto por escena: por la escena y las gemelas, o por la verificación por imagen."""
    return "acierto verificado por imagen" if verificado_test[indice] else "acierto por escena"


# Baselines definidos en "Definición del problema de clasificación"
conteos_test = np.bincount(y_test_idx, minlength=NUMERO_CLASES)
baseline_azar = 1 / NUMERO_CLASES
baseline_mayoritaria = conteos_test.max() / conteos_test.sum()
f1_macro_test = f1_score(y_test_eval, predicciones_test, average="macro")
f1_ponderado_test = f1_score(y_test_eval, predicciones_test, average="weighted")
f1_macro_estricto_test = f1_score(y_test_idx, predicciones_test, average="macro")

comparacion_baselines = pd.DataFrame(
    [
        ("Clasificador aleatorio", baseline_azar, "-"),
        ("Clase mayoritaria siempre", baseline_mayoritaria, "-"),
        ("CNN final", accuracy_test, round(float(f1_macro_test), 4)),
        ("CNN final, criterio estricto", accuracy_estricta_test, round(float(f1_macro_estricto_test), 4)),
    ],
    columns=["Modelo", "Accuracy test", "F1 macro test"],
)
comparacion_baselines["Accuracy test"] = comparacion_baselines["Accuracy test"].round(4)

display(comparacion_baselines)

print(f"Perdida en test      : {perdida_test:.4f}")
print(f"Accuracy en test     : {accuracy_test:.4f}   (estricta, solo la carpeta: {accuracy_estricta_test:.4f})")
print(f"Top-3 accuracy       : {top3_test:.4f}   (estricta: {top3_estricto_test:.4f})")
print(f"F1 macro en test     : {f1_macro_test:.4f}   (estricto: {f1_macro_estricto_test:.4f})")
print(f"F1 ponderado en test : {f1_ponderado_test:.4f}")
print(
    f"Aciertos por escena  : {int(acierto_por_escena.sum())} imagenes en las que el modelo nombro"
    " a otro personaje presente"
)
print(
    f"   verificados       : {int(verificado_test.sum())} de los {len(verificacion_test)} errores con confianza"
    f" > {UMBRAL_CONFIANZA_VERIFICACION:.2f}   (accuracy sin verificar: {accuracy_sin_verificacion_test:.4f})"
)
print(
    f"\nMejora sobre el azar: x{accuracy_test / baseline_azar:.1f}   |   "
    f"sobre la clase mayoritaria: x{accuracy_test / baseline_mayoritaria:.1f}"
)


### Precision, Recall y F1-Score por clase

In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support

# Con el criterio de acierto: la clase evaluada es la de la carpeta, o el personaje presente
# que nombró el modelo
reporte = classification_report(
    y_test_eval,
    predicciones_test,
    labels=range(NUMERO_CLASES),
    target_names=clases,
    digits=3,
    zero_division=0,
)
print(reporte)

precision, recall, f1, soporte = precision_recall_fscore_support(
    y_test_eval, predicciones_test, labels=range(NUMERO_CLASES), zero_division=0
)

metricas_por_clase = pd.DataFrame(
    {
        "personaje": clases,
        "n_test": soporte,
        "n_train": conteos_train,
        "precision": precision.round(3),
        "recall": recall.round(3),
        "f1": f1.round(3),
    }
).sort_values("f1", ascending=False, ignore_index=True)

display(metricas_por_clase)

correlacion = float(np.corrcoef(conteos_train, f1)[0, 1])
print(f"Correlacion entre imagenes de entrenamiento y F1 por clase: {correlacion:.3f}")

In [ ]:
# Relación entre la cantidad de ejemplos de entrenamiento y el F1 alcanzado por cada clase
figura, ejes = plt.subplots(1, 2, figsize=(14, 12))

orden_f1 = metricas_por_clase.sort_values("f1")
posiciones = np.arange(len(orden_f1))
ejes[0].barh(posiciones - 0.2, orden_f1["precision"], height=0.4, label="precision")
ejes[0].barh(posiciones + 0.2, orden_f1["recall"], height=0.4, label="recall")
ejes[0].set_yticks(posiciones)
ejes[0].set_yticklabels(orden_f1["personaje"], fontsize=6)
ejes[0].set_xlabel("Valor")
ejes[0].set_title("Precision y recall por personaje", fontsize=10)
ejes[0].legend(fontsize=8)
ejes[0].grid(alpha=0.3, axis="x")

ejes[1].scatter(metricas_por_clase["n_train"], metricas_por_clase["f1"], s=40)
for _, fila in metricas_por_clase.iterrows():
    ejes[1].annotate(
        fila["personaje"],
        (fila["n_train"], fila["f1"]),
        textcoords="offset points",
        xytext=(4, 3),
        fontsize=6,
    )
ejes[1].set_xscale("log")
ejes[1].set_xlabel("Imágenes de entrenamiento (escala log)")
ejes[1].set_ylabel("F1 en test")
ejes[1].set_title(f"Mas ejemplos, mejor F1 (correlacion = {correlacion:.2f})", fontsize=10)
ejes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Lectura.** La CNN reconoce bien a casi todos los personajes: 60 de los 64 tienen un F1 de 0,60 o más, y
ninguno queda por debajo de 0,30 (en el MLP eran tres o cuatro). Los mejores son Krusty (0,92), Ned Flanders y
Milhouse (0,92), Apu (0,91) y Sideshow Bob (0,91). La familia Simpson, que el MLP confundía por la piel
amarilla compartida, sube a la parte alta: Marge 0,88, Lisa 0,86, Homero 0,84 y Bart 0,81, frente a 0,67,
0,42, 0,48 y 0,35 en la corrida del MLP de GPU a 64 × 64. Con la forma, la piel amarilla deja de ser un
problema: lo que separa a Bart de Lisa es el peinado, no el color.

Los peores son Rod Flanders (0,46), Rainier Wolfcastle (0,51), Kearney (0,56), Martin Prince (0,59) y Disco
Stu (0,60). Aquí sí pesa el tamaño: la correlación entre imágenes de entrenamiento y F1 sube a 0,42 (0,19 en
el MLP), y cuatro de esos cinco personajes tienen menos de 230 imágenes de entrenamiento. Una red que aprende
sus filtros necesita ejemplos de cada personaje, y Rod, con 96, es además casi idéntico a su hermano Todd.
Disco Stu y Maggie tienen precisión 1,00 y recall bajo (0,43 y 0,56): cuando el modelo los nombra acierta
siempre, pero se le escapa la mitad.

### Matriz de confusión

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Filas: la clase evaluada (la de la carpeta, o el personaje presente que nombró el modelo)
matriz = confusion_matrix(y_test_eval, predicciones_test, labels=range(NUMERO_CLASES))
# Normalizada por fila: cada fila suma 1, de modo que el desbalance no distorsione la lectura
totales_fila = matriz.sum(axis=1, keepdims=True)
matriz_normalizada = np.divide(matriz, totales_fila, out=np.zeros(matriz.shape), where=totales_fila > 0)

figura, eje = plt.subplots(figsize=(18, 16))
sns.heatmap(
    matriz_normalizada,
    annot=False,
    cmap="viridis",
    vmin=0,
    vmax=1,
    xticklabels=clases,
    yticklabels=clases,
    cbar_kws={"label": "Proporción de la clase real"},
    ax=eje,
)
eje.set_xlabel("Prediccion")
eje.set_ylabel("Clase real")
eje.set_title("Matriz de confusión normalizada por fila (test)", fontsize=11)
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.show()

aciertos_diagonal = np.diag(matriz_normalizada)
print(f"Recall medio (diagonal): {aciertos_diagonal.mean():.3f}")
print(f"Clase mejor reconocida : {clases[int(aciertos_diagonal.argmax())]} ({aciertos_diagonal.max():.3f})")
print(f"Clase peor reconocida  : {clases[int(aciertos_diagonal.argmin())]} ({aciertos_diagonal.min():.3f})")

### Interpretación crítica de las métricas

In [ ]:
# Comparación de las tres lecturas globales y su distancia
resumen_metricas = pd.DataFrame(
    [
        ("Accuracy (test)", round(float(accuracy_test), 4), "Proporción de aciertos (carpeta o personaje presente); domina por las clases grandes"),
        ("Accuracy estricta (test)", round(float(accuracy_estricta_test), 4), "Solo acepta el personaje de la carpeta; referencia"),
        ("F1 ponderado (test)", round(float(f1_ponderado_test), 4), "Promedio de F1 pesado por el tamaño de cada clase"),
        ("F1 macro (test)", round(float(f1_macro_test), 4), "Promedio simple de todas las clases; cada personaje pesa igual"),
        ("Top-3 accuracy (test)", round(float(top3_test), 4), "El personaje de la carpeta o uno presente está entre las 3 más probables"),
    ],
    columns=["Metrica", "Valor", "Que mide"],
)
display(resumen_metricas)

brecha_metricas = float(accuracy_test) - float(f1_macro_test)

# Clases que el modelo prácticamente no reconoce
clases_debiles = metricas_por_clase[metricas_por_clase["f1"] < 0.30]
clases_fuertes = metricas_por_clase[metricas_por_clase["f1"] >= 0.60]

print(f"Brecha accuracy - F1 macro: {brecha_metricas:.3f}")
print(f"Clases con F1 >= 0.60 : {len(clases_fuertes)} de {NUMERO_CLASES}")
print(f"Clases con F1 <  0.30 : {len(clases_debiles)} de {NUMERO_CLASES}")
if len(clases_debiles):
    print("  " + ", ".join(clases_debiles["personaje"].tolist()))
print(
    f"\nMediana de imagenes de entrenamiento en las clases fuertes: "
    f"{clases_fuertes['n_train'].median():.0f}"
)
if len(clases_debiles):
    print(
        f"Mediana de imagenes de entrenamiento en las clases debiles : "
        f"{clases_debiles['n_train'].median():.0f}"
    )

**Qué revela la distancia entre las métricas.** La accuracy (0,830) y el F1 macro (0,788) quedan a 4 puntos,
y el F1 ponderado (0,829) coincide con la accuracy: el desempeño está repartido entre personajes grandes y
pequeños, con una caída moderada en los que tienen menos imágenes.

La **accuracy estricta**, que solo acepta el personaje de la carpeta, queda 5,3 puntos por debajo de la
oficial (0,777), y el F1 macro estricto, 7,4 (0,714): son las 267 imágenes en las que el modelo nombró a otro
personaje que sí está en la escena (sección «Escenas con varios personajes: aciertos por escena»).

La **top-3 accuracy** llega a 0,932: en seis de cada diez errores, el personaje correcto está entre las tres
candidatas más probables. En el escenario de asistencia, el catalogador encuentra la respuesta entre las tres
sugerencias en más de nueve de cada diez imágenes.

### Cumplimiento de los KPIs definidos

Los cinco KPIs declarados al inicio se contrastan aqui con las cifras de la prueba
final. El umbral de cada uno se fijo antes de entrenar, de modo que la tabla es una
verificacion y no una justificacion a posteriori del resultado obtenido.

In [ ]:
# Verificacion de los KPIs definidos al inicio del notebook: cada umbral se fijo
# antes de evaluar, y aqui se contrasta con las cifras obtenidas en la prueba final.
import time

# Costo de catalogación de extremo a extremo: llevar las imágenes ya redimensionadas al
# dispositivo y pasarlas por la red. La CNN no calcula descriptores: su entrada son los píxeles.
# La lectura del disco no se incluye, igual que en el MLP.
inicio_inferencia = time.perf_counter()
entrada_inferencia = a_tensor(imagenes_test)
modelo_final.predict(entrada_inferencia, batch_size=256, verbose=0)
segundos_inferencia = time.perf_counter() - inicio_inferencia
imagenes_por_segundo = len(imagenes_test) / segundos_inferencia

ventaja_mayoritaria = float(accuracy_test) / baseline_mayoritaria

kpis = [
    ("Cobertura de asistencia", "Top-3 accuracy", 0.70, float(top3_test), "{:.4f}"),
    ("Equidad entre personajes", "F1 macro", 0.45, float(f1_macro_test), "{:.4f}"),
    ("Automatizacion sin revision", "Accuracy", 0.50, float(accuracy_test), "{:.4f}"),
    ("Ventaja sobre criterio trivial", "Accuracy / mayoritaria", 4.0, ventaja_mayoritaria, "x{:.1f}"),
    ("Costo de catalogacion", "Imagenes por segundo (entrada + red)", 1000.0, imagenes_por_segundo, "{:,.0f}"),
]

tabla_kpis = pd.DataFrame(
    [
        {
            "KPI": nombre,
            "Metrica": metrica,
            "Umbral": formato.format(umbral),
            "Obtenido": formato.format(obtenido),
            "Cumple": "SI" if obtenido >= umbral else "NO",
        }
        for nombre, metrica, umbral, obtenido, formato in kpis
    ]
)

cumplidos = sum(1 for _, _, umbral, obtenido, _ in kpis if obtenido >= umbral)
print(f"KPIs cumplidos: {cumplidos} de {len(kpis)}\n")
display(tabla_kpis)

## Análisis de resultados y de errores

### El modelo en acción sobre el conjunto de test

Las métricas anteriores resumen miles de decisiones en cuatro números. Esta sección hace lo contrario:
toma imágenes concretas del conjunto de test —las que el modelo nunca vio— y muestra el proceso completo,
desde la imagen de entrada hasta la distribución de probabilidad de salida.

El primer gráfico recorre una imagen a través de la red **ya entrenada**. En los fundamentos se aplicaron a
mano tres filtros elegidos por nosotros; aquí se ven los mapas de activación que producen los filtros que la
red aprendió, bloque por bloque: los del primer bloque responden a bordes y manchas de color en posiciones
precisas, y los del último, con mucha menos resolución, a zonas grandes de la imagen.

In [ ]:
# Recorrido de una imagen del conjunto de test por la red entrenada:
# de la imagen a los mapas de activación de cada bloque y a las probabilidades de salida
demo_aciertos = np.flatnonzero(predicciones_test == y_test_idx)
demo_indice = int(demo_aciertos[np.argsort(-confianza_test[demo_aciertos])][0])

# Activaciones capa a capa: cada capa se aplica sobre la salida de la anterior.
# training=False apaga el Dropout y hace que BatchNormalization use sus estadísticas acumuladas.
# Los tensores de PyTorch se pasan a NumPy para graficarlos (detach: sin gradiente).
activaciones = {}
tensor_demo = X_test[demo_indice : demo_indice + 1]
with torch.no_grad():
    for capa in modelo_final.layers:
        tensor_demo = capa(tensor_demo, training=False)
        activaciones[capa.name] = tensor_demo.detach().cpu().numpy()[0]

# Mapas de cada bloque: la salida de su última activación, justo antes del pooling
mapas_por_bloque = []
ultima_activacion = None
for capa in modelo_final.layers:
    if isinstance(capa, layers.Activation):
        ultima_activacion = capa.name
    elif isinstance(capa, layers.MaxPooling2D):
        mapas_por_bloque.append(ultima_activacion)

MAPAS_POR_FILA = 8
figura, ejes = plt.subplots(
    len(mapas_por_bloque),
    MAPAS_POR_FILA + 1,
    figsize=(1.75 * (MAPAS_POR_FILA + 1), 2.0 * len(mapas_por_bloque)),
    layout="constrained",
)
filas_activos = []
for fila, nombre_capa in enumerate(mapas_por_bloque):
    mapas = activaciones[nombre_capa]  # (alto, ancho, filtros)
    # Los filtros que más se activaron con esta imagen
    mas_activos = np.argsort(-mapas.mean(axis=(0, 1)))[:MAPAS_POR_FILA]
    filas_activos.append(
        {
            "bloque": fila + 1,
            "capa": nombre_capa,
            "mapa": f"{mapas.shape[0]}x{mapas.shape[1]}",
            "filtros": mapas.shape[2],
            "valores_activos_%": round(float((mapas > 0).mean()) * 100, 1),
            "filtros_que_se_activan_%": round(float((mapas.max(axis=(0, 1)) > 0).mean()) * 100, 1),
        }
    )
    ejes[fila, 0].imshow(imagenes_test[demo_indice])
    ejes[fila, 0].set_ylabel(
        f"bloque {fila + 1}\n{mapas.shape[0]}x{mapas.shape[1]}, {mapas.shape[2]} filtros", fontsize=8
    )
    for columna, filtro in enumerate(mas_activos, start=1):
        ejes[fila, columna].imshow(mapas[..., filtro], cmap="inferno")
        ejes[fila, columna].set_title(f"filtro {filtro}", fontsize=7)
for eje in np.ravel(ejes):
    eje.set_xticks([])
    eje.set_yticks([])
figura.suptitle(
    f"Mapas de activación de la red entrenada: los {MAPAS_POR_FILA} filtros más activos de cada bloque",
    fontsize=12,
    fontweight="bold",
)
plt.show()

display(pd.DataFrame(filas_activos))

# Salida: distribución softmax sobre todas las clases
salida_demo = activaciones["salida"]
real_demo = int(y_test_idx[demo_indice])
figura, (eje_entrada, eje_salida) = plt.subplots(
    1, 2, figsize=(11, 3.8), layout="constrained", gridspec_kw={"width_ratios": [1, 3]}
)
eje_entrada.imshow(imagenes_test[demo_indice])
eje_entrada.set_title(f"Entrada: {ALTO_OBJETIVO}x{ANCHO_OBJETIVO}x3 ({representacion_final})", fontsize=9)
eje_entrada.set_xlabel(clases[real_demo].replace("_", " "), fontsize=9)
eje_entrada.set_xticks([])
eje_entrada.set_yticks([])

eje_salida.bar(
    range(NUMERO_CLASES),
    salida_demo * 100,
    color=["#1b7f3b" if i == real_demo else "#8fb8de" for i in range(NUMERO_CLASES)],
)
eje_salida.axhline(
    100 / NUMERO_CLASES,
    color="red",
    linestyle="--",
    linewidth=1,
    label=f"azar = {100 / NUMERO_CLASES:.1f} %",
)
eje_salida.set_title("Salida softmax (verde = clase real)", fontsize=9)
eje_salida.set_xlabel("índice de clase")
eje_salida.set_ylabel("probabilidad (%)")
eje_salida.legend(fontsize=7)
eje_salida.grid(alpha=0.3, axis="y")

figura.suptitle(
    "Forward pass del modelo entrenado sobre una imagen de test",
    fontsize=12,
    fontweight="bold",
)
plt.show()

print(
    f"Prediccion: {clases[int(salida_demo.argmax())]} con {salida_demo.max() * 100:.1f} % de confianza"
    f"   |   clase real: {clases[real_demo]}"
)
print(
    f"Frente al {100 / NUMERO_CLASES:.1f} % que asignaria el azar: la red ya no reparte la"
    " probabilidad, la concentra."
)

**Qué muestra el recorrido.** Los mapas siguen la jerarquía que describen los fundamentos. El primer bloque
responde a bordes y contornos con la resolución completa: las ventanas del edificio, el borde del atril, las
siluetas del público. En el segundo y el tercero los mapas se fragmentan en combinaciones de bordes más
específicas. En el cuarto, de 8 × 8, los filtros más activos se encienden todos en la misma zona: la del alcalde
Quimby, el personaje de la carpeta, que en la imagen es pequeño y está rodeado de otros. A esa profundidad cada
valor del mapa ve 38 × 38 píxeles de la imagen, y lo que codifica ya no es un borde sino una parte del
personaje.

La representación además se vuelve más selectiva: el 48 % de los valores del primer bloque es distinto de
cero, y solo el 16 % de los del último, donde el 18 % de los filtros ni siquiera se activa con esta imagen. A la
salida, la red asigna un 99,6 % de probabilidad a Quimby, frente al 1,6 % del azar.

#### Predicciones individuales: las cinco clases más probables

En lugar de quedarse con la clase ganadora, aquí se abre la distribución completa. La barra de cada
candidata muestra cuánta probabilidad le asignó el modelo, lo que permite distinguir entre una decisión
tomada con holgura y una en la que dos personajes quedaron casi empatados.

In [ ]:
# Tres casos representativos: un acierto seguro, un error seguro y un caso donde el modelo dudó
# Errores según el criterio de acierto: nombrar a otro personaje presente no es un error
demo_errores = np.flatnonzero(~acierto_test)

casos = [
    (int(demo_aciertos[np.argmax(confianza_test[demo_aciertos])]), "Acierto con máxima confianza"),
    (int(demo_errores[np.argmax(confianza_test[demo_errores])]), "Error cometido con máxima confianza"),
    (int(np.argmin(confianza_test)), "Caso más dudoso de todo el conjunto"),
]

TOP_K = 5
figura, ejes = plt.subplots(
    len(casos),
    2,
    figsize=(11, 3.4 * len(casos)),
    layout="constrained",
    gridspec_kw={"width_ratios": [1, 2.6]},
)

for fila, (indice, descripcion) in enumerate(casos):
    real = int(y_test_idx[indice])
    predicho = int(predicciones_test[indice])
    color = color_resultado(indice)

    eje_imagen = ejes[fila, 0]
    eje_imagen.imshow(imagenes_test[indice])
    eje_imagen.set_xticks([])
    eje_imagen.set_yticks([])
    for lado in eje_imagen.spines.values():
        lado.set_edgecolor(color)
        lado.set_linewidth(3)
    eje_imagen.set_xlabel(
        f"real: {clases[real].replace('_', ' ')}", fontsize=9, labelpad=6, color=color
    )
    eje_imagen.set_title(descripcion, fontsize=9, fontweight="bold")

    # Las TOP_K clases con mayor probabilidad, dibujadas de mayor a menor
    mejores = np.argsort(-probabilidades_test[indice])[:TOP_K][::-1]
    etiquetas = [clases[i].replace("_", " ") for i in mejores]
    colores = [
        COLOR_ACIERTO if i == real else COLOR_ESCENA if clases[i] in acompanantes_test[indice] else "#b0c4de"
        for i in mejores
    ]

    eje_barras = ejes[fila, 1]
    barras = eje_barras.barh(etiquetas, probabilidades_test[indice][mejores] * 100, color=colores)
    eje_barras.bar_label(barras, fmt="%.1f %%", fontsize=8, padding=3)
    eje_barras.set_xlim(0, 110)
    eje_barras.set_xlabel("probabilidad asignada (%)")
    eje_barras.tick_params(axis="y", labelsize=8)
    eje_barras.grid(alpha=0.3, axis="x")
    eje_barras.set_title(
        f"Top {TOP_K}: predice '{clases[predicho].replace('_', ' ')}'"
        f" ({TEXTO_RESULTADO[color]})",
        fontsize=9,
        color=color,
    )

figura.suptitle(
    "Distribucion de probabilidad en imagenes de test",
    fontsize=12,
    fontweight="bold",
)
plt.show()

#### Muestra aleatoria: el modelo trabajando sin filtros

Los ejemplos anteriores están elegidos a propósito por ser extremos. La rejilla siguiente toma una
muestra **al azar** del conjunto de test, sin seleccionar nada: es la imagen más honesta del
comportamiento real del modelo, y la proporción de marcos verdes debería parecerse a la accuracy global.

In [ ]:
# Muestra aleatoria sin seleccionar: marco verde si acertó, ámbar si nombró a otro personaje
# presente y rojo si falló
FILAS_MOSAICO, COLUMNAS_MOSAICO = 4, 6
generador_muestra = np.random.default_rng(SEMILLA)
muestra_aleatoria = generador_muestra.choice(
    len(y_test_idx), size=FILAS_MOSAICO * COLUMNAS_MOSAICO, replace=False
)

figura, ejes = plt.subplots(
    FILAS_MOSAICO,
    COLUMNAS_MOSAICO,
    figsize=(2.1 * COLUMNAS_MOSAICO, 2.9 * FILAS_MOSAICO),
    layout="constrained",
)

for eje, indice in zip(np.ravel(ejes), muestra_aleatoria):
    real = clases[y_test_idx[indice]].replace("_", " ")
    predicho = clases[predicciones_test[indice]].replace("_", " ")
    color = color_resultado(indice)

    eje.imshow(imagenes_test[indice])
    eje.set_xticks([])
    eje.set_yticks([])
    for lado in eje.spines.values():
        lado.set_edgecolor(color)
        lado.set_linewidth(2.5)
    eje.set_xlabel(
        f"real: {real}\npred: {predicho} ({confianza_test[indice]:.2f})"
        + (f"\n{rotulo_escena(indice)}" if color == COLOR_ESCENA else ""),
        fontsize=7,
        color=color,
        labelpad=5,
        linespacing=1.5,
    )

aciertos_muestra = int(acierto_test[muestra_aleatoria].sum())
figura.suptitle(
    f"Muestra aleatoria de test: {aciertos_muestra} aciertos"
    f" de {len(muestra_aleatoria)}  (accuracy global = {accuracy_test:.1%})",
    fontsize=12,
    fontweight="bold",
)
plt.show()

#### Rendimiento por personaje, con una imagen de cada uno

La tabla de métricas por clase es precisa pero abstracta. Esta galería le pone cara: un ejemplo real de
cada uno de los 64 personajes, con el marco coloreado según el **recall** que el modelo alcanza en esa
clase —verde si la reconoce casi siempre, rojo si se le escapa casi siempre—. Se ve de un vistazo que
las clases en rojo son las minoritarias.

In [ ]:
# Una imagen por personaje, con el marco coloreado según el recall de esa clase
recall_por_clase = np.diag(matriz_normalizada)
mapa_color = plt.get_cmap("RdYlGn")

FILAS_GALERIA = 8
COLUMNAS_GALERIA = int(np.ceil(NUMERO_CLASES / FILAS_GALERIA))

figura, ejes = plt.subplots(
    FILAS_GALERIA,
    COLUMNAS_GALERIA,
    figsize=(2.0 * COLUMNAS_GALERIA, 2.7 * FILAS_GALERIA),
    layout="constrained",
)
ejes = np.ravel(ejes)

# Se recorren las clases de mejor a peor recall, para que la degradación sea visible
orden_clases = np.argsort(-recall_por_clase)

for eje, clase in zip(ejes, orden_clases):
    indices_clase = np.flatnonzero(y_test_idx == clase)
    # Se prefiere una imagen acertada; si el modelo no acierta ninguna, se usa la primera disponible
    correctas = indices_clase[predicciones_test[indices_clase] == clase]
    indice = int(correctas[0]) if len(correctas) else int(indices_clase[0])

    eje.imshow(imagenes_test[indice])
    eje.set_xticks([])
    eje.set_yticks([])
    for lado in eje.spines.values():
        lado.set_edgecolor(mapa_color(recall_por_clase[clase]))
        lado.set_linewidth(3.5)
    eje.set_xlabel(
        f"{clases[clase].replace('_', ' ')}\nrecall {recall_por_clase[clase]:.0%}"
        f"  ({len(indices_clase)} img)",
        fontsize=7,
        labelpad=5,
        linespacing=1.5,
    )

for eje in ejes[NUMERO_CLASES:]:
    eje.axis("off")

figura.suptitle(
    "Un personaje por celda, ordenados por recall en test"
    "  (verde = lo reconoce, rojo = se le escapa)",
    fontsize=12,
    fontweight="bold",
)
plt.show()

print(f"Recall medio sobre las {NUMERO_CLASES} clases: {recall_por_clase.mean():.1%}")
print(
    f"Mejor: {clases[int(orden_clases[0])]} ({recall_por_clase[orden_clases[0]]:.0%})"
    f"   |   Peor: {clases[int(orden_clases[-1])]} ({recall_por_clase[orden_clases[-1]]:.0%})"
)

#### En qué se apoya el modelo

Hasta aquí se ha visto **qué** responde el modelo; este último análisis muestra **en qué zonas de la
imagen se apoya** para responderlo. En una CNN la herramienta habitual es **Grad-CAM** (Selvaraju et al.,
2017):

1. Se toman los mapas de activación del último bloque convolucional, que todavía conservan la posición y ya
   responden a partes grandes de la imagen.
2. Se calcula el gradiente de la probabilidad de la clase predicha respecto de cada uno de esos mapas. Su
   promedio dice cuánto empuja cada filtro hacia esa clase.
3. Se suman los mapas ponderados por ese peso, se descartan los valores negativos y el resultado se amplía
   al tamaño de la imagen: las zonas cálidas son las que más aumentan la probabilidad de la clase elegida.

El MLP solo podía responder esta pregunta por bloques de descriptores, con la importancia por permutación;
la CNN la responde sobre la propia imagen. La segunda figura promedia Grad-CAM sobre todo el test: muestra en
qué zonas del encuadre se apoya la red en general.

In [ ]:
# Grad-CAM sobre el último bloque convolucional: qué zonas de la imagen empujan la decisión
import torch.nn.functional as F

indices_gradcam = list(demo_aciertos[np.argsort(-confianza_test[demo_aciertos])][:3]) + list(
    demo_errores[np.argsort(-confianza_test[demo_errores])][:3]
)

# Los mapas del último bloque: la salida de su última activación, antes del pooling
capas_modelo = list(modelo_final.layers)
corte = [capa.name for capa in capas_modelo].index(mapas_por_bloque[-1]) + 1


def grad_cam(indices):
    """Mapas Grad-CAM (n, alto, ancho) en [0, 1] para la clase predicha de cada imagen de test."""
    with torch.no_grad():
        mapas = X_test[indices]
        for capa in capas_modelo[:corte]:
            mapas = capa(mapas, training=False)
    # El gradiente se toma respecto de los mapas, no de los pesos
    mapas = mapas.detach().requires_grad_(True)  # (n, alto, ancho, filtros)
    salida = mapas
    for capa in capas_modelo[corte:]:
        salida = capa(salida, training=False)
    clases_predichas = torch.as_tensor(predicciones_test[indices], dtype=torch.long, device=salida.device)
    puntajes = salida.gather(1, clases_predichas.unsqueeze(1)).sum()
    (gradientes,) = torch.autograd.grad(puntajes, mapas)
    # Peso de cada filtro: el gradiente medio sobre todas las posiciones del mapa
    pesos_filtros = gradientes.mean(dim=(1, 2), keepdim=True)
    cam = torch.relu((pesos_filtros * mapas).sum(dim=-1))
    cam = F.interpolate(
        cam.unsqueeze(1), size=(ALTO_OBJETIVO, ANCHO_OBJETIVO), mode="bilinear", align_corners=False
    )[:, 0]
    cam = cam / cam.amax(dim=(1, 2), keepdim=True).clamp_min(1e-8)
    return cam.detach().cpu().numpy()


mapas_gradcam = grad_cam(indices_gradcam)

figura, ejes = plt.subplots(
    2, len(indices_gradcam), figsize=(2.4 * len(indices_gradcam), 5.8), layout="constrained"
)
for columna, (indice, cam) in enumerate(zip(indices_gradcam, mapas_gradcam)):
    color = color_resultado(indice)
    ejes[0, columna].imshow(imagenes_test[indice])
    ejes[1, columna].imshow(imagenes_test[indice])
    ejes[1, columna].imshow(cam, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    ejes[0, columna].set_xlabel(
        f"real: {clases[y_test_idx[indice]].replace('_', ' ')}\n"
        f"pred: {clases[predicciones_test[indice]].replace('_', ' ')} ({confianza_test[indice]:.2f})",
        fontsize=8,
        color=color,
        labelpad=6,
        linespacing=1.5,
    )
    for eje in (ejes[0, columna], ejes[1, columna]):
        eje.set_xticks([])
        eje.set_yticks([])
        for lado in eje.spines.values():
            lado.set_edgecolor(color)
            lado.set_linewidth(2)

ejes[0, 0].set_ylabel("Imagen", fontsize=10, fontweight="bold", labelpad=8)
ejes[1, 0].set_ylabel("Grad-CAM", fontsize=10, fontweight="bold", labelpad=8)
figura.suptitle(
    "Zonas que determinan la decisión (tres aciertos y tres errores con la mayor confianza)",
    fontsize=12,
    fontweight="bold",
)
plt.show()

# Grad-CAM medio sobre todo el test, por lotes: en qué zonas del encuadre se apoya la red en general
TAMANO_LOTE_GRADCAM = 256
suma_cam = np.zeros((ALTO_OBJETIVO, ANCHO_OBJETIVO))
concentracion_test = []
centro_test = []
margen_alto, margen_ancho = ALTO_OBJETIVO // 4, ANCHO_OBJETIVO // 4
for desde in range(0, len(y_test_idx), TAMANO_LOTE_GRADCAM):
    lote_cam = grad_cam(list(range(desde, min(desde + TAMANO_LOTE_GRADCAM, len(y_test_idx)))))
    suma_cam += lote_cam.sum(axis=0)
    plano = lote_cam.reshape(len(lote_cam), -1)
    umbral = np.quantile(plano, 0.90, axis=1, keepdims=True)
    total_cam = np.maximum(plano.sum(axis=1), 1e-8)
    concentracion_test.append((plano * (plano >= umbral)).sum(axis=1) / total_cam)
    centro_test.append(
        lote_cam[:, margen_alto:ALTO_OBJETIVO - margen_alto, margen_ancho:ANCHO_OBJETIVO - margen_ancho].sum(axis=(1, 2))
        / total_cam
    )
cam_medio = suma_cam / len(y_test_idx)
concentracion_test = np.concatenate(concentracion_test)
centro_test = np.concatenate(centro_test)

figura, ejes = plt.subplots(1, 2, figsize=(9, 4.0), layout="constrained")
ejes[0].imshow(cam_medio / cam_medio.max(), cmap="jet", vmin=0, vmax=1)
ejes[0].add_patch(
    plt.Rectangle(
        (margen_ancho - 0.5, margen_alto - 0.5), ANCHO_OBJETIVO - 2 * margen_ancho, ALTO_OBJETIVO - 2 * margen_alto,
        fill=False, edgecolor="white", linestyle="--", linewidth=1.5,
    )
)
ejes[0].set_title("Grad-CAM medio en test (línea: mitad central)", fontsize=9)
ejes[0].set_xticks([])
ejes[0].set_yticks([])
ejes[1].hist(centro_test[acierto_test], bins=30, alpha=0.6, label="aciertos", color=COLOR_ACIERTO, density=True)
ejes[1].hist(centro_test[~acierto_test], bins=30, alpha=0.6, label="errores", color=COLOR_ERROR, density=True)
ejes[1].axvline(0.25, color="gray", linestyle="--", linewidth=1, label="reparto uniforme (0,25)")
ejes[1].set_xlabel("Parte de Grad-CAM en la mitad central de la imagen")
ejes[1].set_title("Cuánto se apoya la red en el centro del encuadre", fontsize=9)
ejes[1].legend(fontsize=8)
ejes[1].grid(alpha=0.3)
plt.show()

tabla_gradcam = pd.DataFrame(
    [
        ("Imágenes de test", len(y_test_idx), ""),
        (
            "Parte de Grad-CAM en el 10 % de píxeles más cálidos",
            round(float(concentracion_test.mean()), 3),
            "0,100 si la red mirara toda la imagen por igual",
        ),
        (
            "Parte de Grad-CAM en la mitad central (aciertos)",
            round(float(centro_test[acierto_test].mean()), 3),
            "0,250 con un reparto uniforme",
        ),
        (
            "Parte de Grad-CAM en la mitad central (errores)",
            round(float(centro_test[~acierto_test].mean()), 3),
            "0,250 con un reparto uniforme",
        ),
    ],
    columns=["Indicador", "Valor", "Referencia"],
)
display(tabla_gradcam)

**Lectura de Grad-CAM.** La red mira a los personajes, y sobre todo a sus caras. En los aciertos más
seguros, la zona cálida cae sobre la cabeza del personaje aunque ocupe una parte pequeña del encuadre: Quimby
en el atril, Frink en un costado, Skinner en primer plano. El Grad-CAM medio de todo el test se concentra en la
parte central y superior de la imagen, donde suelen estar las cabezas: el 10 % de píxeles más cálidos reúne el
48 % de la influencia (sería el 10 % si la red mirara toda la imagen por igual), y la mitad central, el 52 % en
los aciertos (25 % con un reparto uniforme).

Los errores más confiados no son de una red que mira el fondo: en los tres de la figura, la red mira a un
personaje que sí está en la imagen. En la carpeta de Homero nombra a Lisa, que está a su lado; en la de Apu, a
Manjula, la única que aparece en el encuadre; en la de Moe, a Kirk Van Houten, junto a él. La escena del
episodio y las gemelas no los detectaron, así que cuentan como errores. En los errores, la parte de Grad-CAM
en la mitad central baja algo (48 %), sin acercarse al reparto uniforme.

### Ejemplos correctamente clasificados

In [ ]:
def mostrar_ejemplos(indices, titulo, filas=2, columnas=5):
    """Dibuja una rejilla de imágenes de test con su etiqueta real, la predicha y la confianza."""
    indices = list(indices)
    figura, ejes = plt.subplots(
        filas,
        columnas,
        figsize=(2.2 * columnas, 3.0 * filas),
        layout="constrained",
    )
    ejes = np.ravel(ejes)

    for eje, indice in zip(ejes, indices):
        real = clases[y_test_idx[indice]].replace("_", " ")
        predicho = clases[predicciones_test[indice]].replace("_", " ")
        color = color_resultado(indice)

        eje.imshow(imagenes_test[indice])
        eje.set_xticks([])
        eje.set_yticks([])
        # Marco de color: verde si acertó, ámbar si nombró a otro personaje presente, rojo si falló
        for lado in eje.spines.values():
            lado.set_edgecolor(color)
            lado.set_linewidth(2.5)

        # Las etiquetas van debajo de la imagen para que no invadan la fila superior
        eje.set_xlabel(
            f"real: {real}\npred: {predicho}  ({confianza_test[indice]:.2f})"
            + (f"\n{rotulo_escena(indice)}" if color == COLOR_ESCENA else ""),
            fontsize=8,
            color=color,
            labelpad=6,
            linespacing=1.5,
        )

    for eje in ejes[len(indices):]:
        eje.axis("off")

    figura.suptitle(titulo, fontsize=13, fontweight="bold")
    plt.show()


aciertos = np.flatnonzero(acierto_test)
# Aciertos con mayor confianza, sin repetir personaje, para ver variedad de clases
aciertos_ordenados = aciertos[np.argsort(-confianza_test[aciertos])]
vistos = set()
seleccion_aciertos = []
for indice in aciertos_ordenados:
    personaje = y_test_eval[indice]
    if personaje not in vistos:
        vistos.add(personaje)
        seleccion_aciertos.append(indice)
    if len(seleccion_aciertos) == 10:
        break

mostrar_ejemplos(seleccion_aciertos, "Aciertos con mayor confianza (un personaje por imagen)")

print(f"Aciertos totales: {len(aciertos)} de {len(y_test_idx)} ({len(aciertos) / len(y_test_idx):.1%})")
print(f"Confianza media en los aciertos: {confianza_test[aciertos].mean():.3f}")

**Qué está aprovechando el modelo.** Los aciertos más seguros, todos con una confianza de 0,99 o más,
muestran lo que el MLP no podía hacer: reconocer a un personaje por su forma. Aparecen Maggie, Barney y Martin
Prince, que comparten la piel amarilla con la familia Simpson; Quimby y Sideshow Mel en escenas con otros
personajes, y Frink ocupando un costado del encuadre. El MLP, con histogramas de color, acertaba con más
seguridad en los planos donde el personaje llenaba la imagen y su paleta era única; la CNN acierta aunque el
personaje sea pequeño o comparta colores, porque reconoce el peinado, la ropa y los rasgos de la cara.

### Ejemplos mal clasificados

In [ ]:
# Errores según el criterio de acierto: los aciertos por escena no cuentan
errores = np.flatnonzero(~acierto_test)

# Los errores más graves: equivocaciones cometidas con la mayor confianza
errores_confiados = errores[np.argsort(-confianza_test[errores])][:50]
mostrar_ejemplos(errores_confiados, "Errores cometidos con mayor confianza")

# Errores con baja confianza: el modelo tampoco sabía
errores_dudosos = errores[np.argsort(confianza_test[errores])][:10]
mostrar_ejemplos(errores_dudosos, "Errores con menor confianza (el modelo dudaba)")

resumen_errores = pd.DataFrame(
    [
        ("Errores totales", len(errores), f"{len(errores) / len(y_test_idx):.1%} del test"),
        ("Confianza media en aciertos", round(float(confianza_test[aciertos].mean()), 3), ""),
        ("Confianza media en errores", round(float(confianza_test[errores].mean()), 3), ""),
        (
            "Errores con confianza > 0.90",
            int((confianza_test[errores] > 0.90).sum()),
            f"{(confianza_test[errores] > 0.90).mean():.1%} de los errores",
        ),
        (
            f"Errores con confianza > {UMBRAL_CONFIANZA_VERIFICACION:.2f}",
            int((confianza_test[errores] > UMBRAL_CONFIANZA_VERIFICACION).sum()),
            f"tras la verificación: {int(verificado_test.sum())} eran otro personaje en la imagen",
        ),
    ],
    columns=["Indicador", "Valor", "Detalle"],
)
display(resumen_errores)

**Dos tipos de error muy distintos.** La confianza separa bien aciertos de errores: es de 0,79 en los
aciertos y de 0,38 en los errores, y solo 25 de los 860 errores superan 0,90. Las predicciones con confianza
baja son las que conviene mandar a revisión humana.

Los **errores con confianza alta** son 64 (63 tras la verificación por imagen, el 1,2 % del test), y la
galería muestra que casi todos son aciertos que el criterio no alcanza a reconocer. En casi todas las imágenes
de la figura, el personaje predicho está en el encuadre: Lisa junto a Homero, Manjula en la carpeta de Apu, el juez
Snyder en su estrado, Skinner en su escritorio, Agnes junto a Skinner en el auto, Rod junto a Todd; en la
carpeta de Bart, la imagen muestra a Homero. Ni la escena del episodio (±3 s), ni las gemelas, ni la
verificación por imagen los detectaron, así que la accuracy oficial es una cota inferior de lo que acierta la
red. La CNN comete más errores confiados que el MLP a la misma resolución (64 frente a 35), pero de otra
naturaleza: los del MLP eran sobre todo paletas parecidas; los de la CNN, otro personaje presente.

Los errores con baja confianza, en cambio, son imágenes difíciles de verdad: personajes diminutos en un plano
general, cortados por el encuadre, de espaldas o disfrazados.

### Confusiones más frecuentes y posibles causas

In [ ]:
# Pares (clase real, clase predicha) más frecuentes, excluyendo la diagonal
pares = []
for real in range(NUMERO_CLASES):
    for predicho in range(NUMERO_CLASES):
        if real != predicho and matriz[real, predicho] > 0:
            pares.append(
                {
                    "real": clases[real],
                    "predicho": clases[predicho],
                    "casos": int(matriz[real, predicho]),
                    "% de la clase real": round(float(matriz_normalizada[real, predicho]) * 100, 1),
                    "n_train_real": int(conteos_train[real]),
                    "n_train_predicho": int(conteos_train[predicho]),
                }
            )

confusiones = pd.DataFrame(pares).sort_values("casos", ascending=False, ignore_index=True).head(15)
display(confusiones)

figura, eje = plt.subplots(figsize=(9, 5))
etiquetas_pares = confusiones.head(10).apply(
    lambda fila: f"{fila['real']} -> {fila['predicho']}", axis=1
)
eje.barh(etiquetas_pares[::-1], confusiones.head(10)["casos"][::-1], color="firebrick", alpha=0.8)
eje.set_xlabel("Número de imágenes de test")
eje.set_title("Las 10 confusiones más frecuentes", fontsize=10)
eje.tick_params(axis="y", labelsize=8)
eje.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

destino_frecuente = confusiones.groupby("predicho")["casos"].sum().sort_values(ascending=False)
print("Clases que más absorben errores de otras (el modelo las usa como respuesta por defecto):")
for personaje, casos in destino_frecuente.head(5).items():
    print(f"  {personaje}: {casos} imagenes de otras clases predichas como esta")

**Causas detrás de las confusiones.** Las confusiones que quedan son de personajes que se parecen de verdad
o que comparten escena:

- **Personajes dibujados casi iguales.** Selma se predice como Patty en el 21 % de los casos (16 imágenes) y
  Patty como Selma en el 11 %: son gemelas, con el mismo diseño. Homero y el abuelo, Lenny y Carl, Rod y Todd
  son variaciones de un mismo dibujo.
- **Escenas compartidas.** Wiggum con Lou, Chalmers y Agnes con Skinner, Helen con el reverendo Lovejoy (29 %
  de las imágenes de Helen) y Apu con Manjula salen juntos en pantalla. Cuando la escena lo confirma, cuentan
  como acierto; las que quedan en la tabla son las que la escena no alcanzó a detectar.
- **La familia ya no es un problema.** Lisa y Bart se confunden entre sí en un 3 a 5 % de los casos, frente
  al 12 a 19 % del MLP. Homero sigue siendo el destino más frecuente de los errores de otras clases (21
  imágenes), como era de esperar del personaje con más imágenes del dataset.

### Escenas con varios personajes: aciertos por escena

Desde la evaluación, una predicción cuenta como acierto si nombra al personaje de la carpeta o a otro que
aparece en la misma imagen, según los fotogramas del mismo episodio, una gemela guardada en la carpeta de
otro personaje (sección «Escenas con varios personajes») o, en los errores con confianza mayor que 0,80,
una imagen de la misma toma en la carpeta del personaje predicho (sección «Verificación de los errores con
confianza alta»). Esta sección separa las dos cosas que la accuracy
estricta mezcla: confundir a un personaje con otro, que es un error, y nombrar a otro personaje que sí está
en pantalla, que es un **acierto por escena**.

La escena se estima con una ventana de tiempo y no con una anotación manual: un corte de cámara dentro de
la ventana puede dar por presente a alguien que no sale en el encuadre. Las gemelas son más seguras, porque
son la misma imagen guardada en dos carpetas. La galería permite comprobarlo a ojo. Las imágenes de Kaggle
sin gemela solo tienen acompañantes si la verificación por imagen los encuentra; si no, en ellas ambas
métricas coinciden.

In [ ]:
# El criterio de acierto se aplicó en la evaluación (acierto_estricto, acierto_por_escena,
# acierto_test): aquí se separan los aciertos por escena de los errores
nombres_predichos = [clases[indice] for indice in predicciones_test]
con_escena_test = np.array([bool(presentes) for presentes in acompanantes_test])
aciertos_escena = np.flatnonzero(acierto_por_escena)
distintos_de_carpeta = int((~acierto_estricto).sum())


def proporcion(valores):
    return round(float(valores.mean()), 4) if len(valores) else np.nan


resumen_tolerante = pd.DataFrame(
    [
        ("Imágenes de test", len(y_test_idx), ""),
        (
            "   con otro personaje presente (escena, gemela o verificación)",
            int(con_escena_test.sum()),
            f"{con_escena_test.mean():.1%} del test",
        ),
        (
            "Predicciones distintas del personaje de la carpeta",
            distintos_de_carpeta,
            f"{distintos_de_carpeta / len(y_test_idx):.1%} del test",
        ),
        (
            "   que nombran a otro personaje presente (aciertos por escena)",
            len(aciertos_escena),
            f"{len(aciertos_escena) / max(distintos_de_carpeta, 1):.1%} de esas predicciones",
        ),
        (
            "      de ellos, verificados por imagen",
            int(verificado_test.sum()),
            f"de {len(verificacion_test)} errores con confianza > {UMBRAL_CONFIANZA_VERIFICACION:.2f}",
        ),
        ("Accuracy estricta (referencia)", proporcion(acierto_estricto), ""),
        (
            "Accuracy oficial (carpeta o personaje presente)",
            round(accuracy_test, 4),
            f"{accuracy_test - float(acierto_estricto.mean()):+.4f}",
        ),
        ("Accuracy estricta en imágenes con otro personaje", proporcion(acierto_estricto[con_escena_test]), ""),
        ("Accuracy oficial en imágenes con otro personaje", proporcion(acierto_test[con_escena_test]), ""),
    ],
    columns=["Indicador", "Valor", "Detalle"],
)
display(resumen_tolerante)

pares_en_escena = (
    pd.DataFrame(
        {
            "carpeta": [clases[y_test_idx[indice]] for indice in aciertos_escena],
            "predicho": [nombres_predichos[indice] for indice in aciertos_escena],
        }
    )
    .value_counts()
    .head(10)
    .reset_index(name="casos")
)
display(pares_en_escena)

if len(aciertos_escena):
    generador = np.random.default_rng(SEMILLA)
    muestra_escena = generador.choice(aciertos_escena, size=min(10, len(aciertos_escena)), replace=False)
    figura, ejes = plt.subplots(2, 5, figsize=(11, 6.6), layout="constrained")
    ejes = np.ravel(ejes)
    for eje, indice in zip(ejes, muestra_escena):
        eje.imshow(imagenes_test[indice])
        eje.set_xticks([])
        eje.set_yticks([])
        for lado in eje.spines.values():
            lado.set_edgecolor(COLOR_ESCENA)
            lado.set_linewidth(2.5)
        escena_visible = sorted(acompanantes_test[indice])
        resto = f" +{len(escena_visible) - 2}" if len(escena_visible) > 2 else ""
        eje.set_xlabel(
            f"carpeta: {clases[y_test_idx[indice]].replace('_', ' ')}\n"
            f"pred: {nombres_predichos[indice].replace('_', ' ')} ({confianza_test[indice]:.2f})\n"
            f"escena: {', '.join(n.replace('_', ' ') for n in escena_visible[:2])}{resto}",
            fontsize=7,
            color="#9c640c",
            labelpad=6,
            linespacing=1.4,
        )
    for eje in ejes[len(muestra_escena):]:
        eje.axis("off")
    figura.suptitle(
        "Aciertos por escena: el modelo nombró a otro personaje presente", fontsize=13, fontweight="bold"
    )
    plt.show()

**Lectura.** El 14,7 % del test tiene otro personaje presente según la escena, las gemelas o la
verificación, y ahí la diferencia entre los dos criterios es enorme: la accuracy estricta en esas imágenes es
de 0,468 y la oficial, de 0,828, casi la misma que en el resto del test. La CNN nombra al acompañante en una de
cada cuatro predicciones distintas de la carpeta (267 de 1.127), y los pares son los que la serie pone juntos:
Wiggum y Lou (30 casos entre los dos sentidos), Patty y Selma (19), Smithers y Burns (13), Skinner y Chalmers
(12), Lenny y Carl.

Esos aciertos por escena suben la accuracy 5,3 puntos. Aun así el criterio sigue siendo conservador: la
verificación por imagen encontró la misma toma en 1 de los 64 errores confiados, y la galería de «Ejemplos
mal clasificados» muestra muchos más en los que el personaje predicho sí aparece.

### Originales, parecidas y alteradas en test

El test tiene las mismas condiciones que el entrenamiento: originales sin transformar, versiones filtradas
de las imágenes casi idénticas a otra (parecidas) y variantes alteradas de un cuarto de las originales.
Esta tabla mide el modelo por separado en cada tipo:

- **Originales.** Las imágenes tal como están en el dataset. Es la fila comparable con las corridas
  anteriores, cuyo test no tenía alteradas.
- **Parecidas.** Antes del filtro, las imágenes de test con una gemela casi idéntica en entrenamiento se
  acertaban en torno al 95 % de las veces, frente a un 53 % del resto. Si las versiones filtradas se siguen
  acertando mucho más que las originales, el modelo reconoce la escena a pesar del filtro.
- **Alteradas.** Miden si el modelo resiste el giro, el cambio de color y el filtro de textura con los
  que aprendió.

In [ ]:
es_alterada_test = datos_test["alterada"].to_numpy()
es_parecida_test = datos_test["parecida"].to_numpy()
es_original_test = ~es_alterada_test & ~es_parecida_test

resumen_tipos_test = pd.DataFrame(
    [
        (
            nombre,
            int(mascara.sum()),
            proporcion(acierto_test[mascara]),
            proporcion(acierto_estricto[mascara]),
            proporcion(top3_por_imagen[mascara]),
        )
        for nombre, mascara in (
            ("Original, sin transformar", es_original_test),
            ("Parecida: versión filtrada de una casi idéntica a otra", es_parecida_test),
            ("Alterada: variante de una original de test", es_alterada_test),
        )
    ],
    columns=["Imágenes de test", "n", "accuracy", "accuracy estricta", "top-3"],
)
display(resumen_tipos_test)
accuracy_originales_test, accuracy_parecidas_test, accuracy_alteradas_test = resumen_tipos_test["accuracy"]


**Lectura.** El orden es el mismo del MLP —parecidas por encima, alteradas por debajo—, pero las
distancias son más cortas:

- **Alteradas: 3,4 puntos por debajo de las originales** (0,792 frente a 0,827). El giro, el cambio de color y
  el filtro de textura cuestan poco, aunque la red solo vio al entrenar una variante fija para un cuarto de las
  imágenes.
- **Parecidas: 6 puntos por encima** (0,886), frente a los 9 a 14 del MLP. El filtro reduce la ventaja de las
  tomas casi idénticas, y la CNN depende menos de ella porque reconoce al personaje y no la paleta de la toma.

La fila de **originales** (0,827 de accuracy y 0,783 estricta) es la medida más limpia del modelo, y queda a
menos de medio punto de la global.

### Fortalezas y debilidades del modelo

In [ ]:
segundos_por_epoca_base = float(globals().get("segundos_entrenamiento_base", np.nan)) / len(
    historial_base.history["loss"]
)
sintesis = pd.DataFrame(
    [
        (
            "Fortaleza",
            "Supera con holgura los baselines",
            f"accuracy {accuracy_test:.3f} frente a {baseline_azar:.3f} (azar) y {baseline_mayoritaria:.3f} (clase mayoritaria)",
        ),
        (
            "Fortaleza",
            "Buena señal en el top-3",
            f"top-3 accuracy {top3_test:.3f}: la clase correcta suele estar entre las candidatas",
        ),
        (
            "Fortaleza",
            "Reconoce con solidez a los personajes principales",
            f"{len(clases_fuertes)} clases con F1 >= 0.60",
        ),
        (
            "Fortaleza",
            "La confianza es informativa",
            f"confianza media {confianza_test[aciertos].mean():.2f} en aciertos y "
            f"{confianza_test[errores].mean():.2f} en errores",
        ),
        (
            "Fortaleza",
            "Aprende sus propios filtros con pocos parámetros",
            f"{modelo_final.count_params():,} parametros para {len(X_train):,} imagenes de entrenamiento",
        ),
        (
            "Debilidad",
            "Desempeño desigual entre clases",
            f"brecha accuracy - F1 macro = {brecha_metricas:.3f}; {len(clases_debiles)} clases con F1 < 0.30",
        ),
        (
            "Debilidad",
            "Depende del volumen de datos por clase",
            f"correlacion entre n_train y F1 por clase = {correlacion:.2f}",
        ),
        (
            "Debilidad",
            "Resolución de trabajo baja",
            f"a {ALTO_OBJETIVO}x{ANCHO_OBJETIVO} los personajes pequeños en el encuadre ocupan pocos pixeles",
        ),
        (
            "Debilidad",
            "Entrenada desde cero",
            "los filtros salen solo de este dataset; una red preentrenada partiria de filtros ya aprendidos",
        ),
        (
            "Debilidad",
            "Nada la obliga a mirar al personaje",
            f"en los errores, {centro_test[~acierto_test].mean():.0%} de Grad-CAM cae en la mitad central"
            f" (aciertos: {centro_test[acierto_test].mean():.0%})",
        ),
        (
            "Debilidad",
            "Costo de entrenamiento",
            f"{segundos_por_epoca_base:.1f} s por epoca del modelo base en {DISPOSITIVO}",
        ),
    ],
    columns=["Tipo", "Aspecto", "Evidencia"],
)

display(sintesis)

**Síntesis.** La CNN resuelve lo que el MLP no podía: reconocer a los personajes por su forma. Con el mismo
proceso, los mismos datos y la misma resolución, sube la accuracy de test de 0,550 a 0,830 y el F1 macro de
0,522 a 0,788, cumple los cinco KPIs con margen y no deja ningún personaje por debajo de 0,30 de F1. Lo que
queda son personajes dibujados casi iguales, personajes con menos de un centenar de imágenes y escenas con
varios personajes que el criterio no detecta. Sus límites son ahora de datos y de costo: un sobreajuste claro
(0,21 de brecha), una dependencia mayor del número de imágenes por personaje y un entrenamiento más de cinco
veces más caro que el del MLP.

## Limitaciones de la CNN y propuestas de mejora

### Limitaciones de esta CNN

La CNN resuelve las limitaciones estructurales que tenía el MLP: no aplana la imagen, comparte los filtros en
todas las posiciones, gana invarianza aproximada a desplazamientos con el pooling y aprende sus propias
características en lugar de recibirlas elegidas a mano. Las que siguen son las que quedan.

**1. Resolución de trabajo.** A 64 x 64 un personaje en un plano general ocupa unas pocas decenas de píxeles,
y los detalles que distinguen a dos personajes parecidos —los ojos, el peinado, un accesorio— se pierden en
el redimensionado. Subir la resolución multiplica el costo por cuatro cada vez que se dobla el lado.

**2. Entrenada desde cero.** Todos los filtros salen de unas 35.000 imágenes de entrenamiento, con 64
personajes y algunos con menos de cien. Las redes preentrenadas en corpus de millones de imágenes parten de
filtros ya aprendidos —bordes, texturas, partes de objetos— y solo ajustan los últimos.

**3. Nada la obliga a mirar al personaje.** Igual que el MLP, la red aprende cualquier regularidad que
separe las clases, y el escenario puede ser una: la taberna de Moe, la central nuclear, el aula de Edna.
Grad-CAM permite ver cuándo se apoya en el fondo, pero no lo corrige.

**4. Invarianza solo aproximada.** El pooling tolera desplazamientos pequeños, no rotaciones grandes ni
cambios de escala fuertes. Los cubren las imágenes alteradas, con un giro de 10 a 20°, pero solo un cuarto de
las imágenes tiene una y siempre la misma.

**5. Una sola etiqueta por imagen.** La salida `softmax` obliga a elegir un personaje aunque en la escena haya
varios. El criterio de acierto lo compensa en la evaluación, no en el modelo.

**6. Costo de cómputo.** Cada imagen pasa por cuatro bloques de convoluciones: entrenar la CNN cuesta
bastante más que entrenar el MLP sobre descriptores, sobre todo en la CPU.

### Propuestas de mejora para futuras iteraciones

| Propuesta | En qué consiste | Qué limitación ataca |
|---|---|---|
| **Transfer learning** | Partir de una red preentrenada (MobileNet, ResNet, EfficientNet) y ajustar sus últimas capas a los 64 personajes | Es la mejora principal: filtros aprendidos sobre millones de imágenes (2), que suelen dar un salto grande de desempeño con pocos datos propios |
| **Aumento de datos en línea** | Generar en cada época un espejo, un recorte o un cambio de color distinto para cada imagen, en lugar de una variante fija para un cuarto de ellas | Multiplica la variedad efectiva del conjunto y la invarianza a escala y posición (4), sobre todo en los personajes con menos imágenes |
| **Más resolución** | Entrenar a 96 x 96 o 128 x 128 en la GPU | Recupera los detalles finos que se pierden a 64 x 64 (1), a un costo cuatro veces mayor por época al doblar el lado |
| **Arquitecturas residuales** | Bloques con conexiones residuales (ResNet) o convoluciones separables (MobileNet) | Permiten redes más profundas sin que el gradiente se degrade, y abaratan cada capa (6) |
| **Recorte previo del personaje** | Aplicar un detector que localice al personaje y recortar esa región antes de clasificar | Elimina el fondo (3) y normaliza la escala y el encuadre (1, 4) |
| **Salida multietiqueta** | Sigmoides independientes en la salida, entrenadas con los personajes de la escena | Permite nombrar a todos los personajes presentes en lugar de uno solo (5) |
| **Validación cruzada** | Repetir el entrenamiento con varias particiones y varias semillas, reportando media y desviación | Da una estimación más robusta que una única partición 70/20/10 y permite saber si las diferencias entre configuraciones son reales o ruido |
| **Partición por grupo o por episodio** | Repartir en un mismo conjunto cada grupo de imágenes casi idénticas, o todos los fotogramas de un episodio | Evita que tomas de una misma escena caigan a la vez en entrenamiento y test |
| **Anotación de escenas** | Marcar a mano qué personajes aparecen en una muestra de imágenes de test | Hace exacto el criterio de acierto, que hoy se estima con la escena, las gemelas y la verificación por imagen |

## Conclusiones

In [ ]:
# Cifras finales del trabajo, tomadas de las secciones anteriores
conclusiones = pd.DataFrame(
    [
        ("Personajes (clases)", NUMERO_CLASES),
        ("Imágenes de entrenamiento", len(X_train)),
        ("   de ellas, alteradas", int(datos.loc[MASCARAS["train"], "alterada"].sum())),
        ("Imágenes de validación", len(X_val)),
        ("   de ellas, alteradas", int(datos.loc[MASCARAS["val"], "alterada"].sum())),
        ("Imágenes de test", len(X_test)),
        ("   de ellas, alteradas", int(datos.loc[MASCARAS["test"], "alterada"].sum())),
        ("Imágenes parecidas, con filtro (todos los conjuntos)", int(datos["parecida"].sum())),
        ("Pares de gemelas entre personajes", n_pares_gemelas),
        ("Reparto de las originales", " / ".join(f"{PROPORCIONES[nombre]:.0%}" for nombre in CONJUNTOS)),
        ("Resolución de trabajo", f"{ALTO_OBJETIVO}x{ANCHO_OBJETIVO}"),
        ("Entrada del modelo final", f"imagen {ALTO_OBJETIVO}x{ANCHO_OBJETIVO}x3 ({representacion_final})"),
        ("Configuración final", nombre_final),
        ("Arquitectura del modelo final", arquitectura_final),
        ("Parámetros del modelo final", f"{modelo_final.count_params():,}"),
        ("Accuracy en test (carpeta o personaje presente)", round(float(accuracy_test), 4)),
        ("Accuracy estricta en test (solo la carpeta)", round(float(accuracy_estricta_test), 4)),
        ("Accuracy en test sin la verificación por imagen", round(accuracy_sin_verificacion_test, 4)),
        (
            f"Errores con confianza > {UMBRAL_CONFIANZA_VERIFICACION:.2f} verificados por imagen",
            f"{int(verificado_test.sum())} de {len(verificacion_test)}",
        ),
        (
            "   en originales / parecidas / alteradas",
            " / ".join(
                f"{valor:.4f}"
                for valor in (accuracy_originales_test, accuracy_parecidas_test, accuracy_alteradas_test)
            ),
        ),
        ("Top-3 accuracy en test", round(float(top3_test), 4)),
        ("F1 macro en test", round(float(f1_macro_test), 4)),
        ("F1 ponderado en test", round(float(f1_ponderado_test), 4)),
        ("Baseline azar", round(baseline_azar, 4)),
        ("Baseline clase mayoritaria", round(float(baseline_mayoritaria), 4)),
    ],
    columns=["Concepto", "Valor"],
)

display(conclusiones)

**Sobre los resultados.** En la corrida de GPU a 64 × 64 del 19-10-2026, la CNN alcanza una accuracy de test
de 0,830 con el criterio de acierto del proyecto (0,777 con el estricto), una top-3 de 0,932 y un F1 macro de
0,788: 53 veces el azar y casi 11 veces la clase mayoritaria. Los cinco KPIs se cumplen con amplio margen; el
de costo, con 5.326 imágenes por segundo, cinco veces el umbral. Frente al MLP a la misma resolución y en el
mismo dispositivo, la accuracy sube 28 puntos, la estricta 27, la top-3 20 y el F1 macro 27.

**Sobre los objetivos planteados.** Se cumplieron los seis objetivos específicos: indexar el dataset con la
etiqueta de cada carpeta; repartirlo 70/20/10 de forma estratificada, con cada alterada en el mismo conjunto
que su original; preparar las imágenes como tensores sin aplanar; entrenar el modelo base y trece variantes
controladas; seleccionar por F1 macro de validación y medir una sola vez en test; y auditar el resultado por
clase, por tipo de imagen, por escena y con Grad-CAM, comparándolo con el MLP.

**Sobre lo que aprendimos.**

1. **La representación decide más que el ajuste fino.** Pasar de descriptores elegidos a mano a filtros
   aprendidos sube el F1 macro de test 27 puntos; ninguna variante de hiperparámetros, en el MLP o en la CNN,
   ganó más de 3.
2. **El color sigue importando.** Sin él, la misma CNN pierde 20 puntos de F1 macro de validación. La forma
   sola rinde como el color solo del MLP; las dos juntas son las que separan a los personajes.
3. **Las decisiones propias de una CNN se notan.** El pooling global supera al aplanado y una segunda
   convolución por bloque supera a la base: invarianza a la posición y profundidad son las dos propiedades que
   el MLP no tenía.
4. **El criterio de acierto se queda corto.** Casi todos los errores confiados nombran a un personaje que está
   en la imagen: la accuracy de 0,830 es una cota inferior, y anotar a mano las escenas de una muestra del test
   es lo que más precisión daría a la evaluación.
5. **El costo se mueve al entrenamiento.** La CNN tarda 5,6 veces más en entrenar que el MLP, pero clasifica
   imágenes nuevas más rápido de extremo a extremo: 5.326 imágenes por segundo, frente a 1.285–1.557 del MLP,
   que antes debe calcular sus descriptores en la CPU.

## Probar el modelo con una imagen cualquiera

Esta sección empaqueta todo lo anterior en una función de uso directo. Recibe la ruta de una imagen —del
dataset o de cualquier otro origen— y le aplica **exactamente** el mismo pre-procesamiento con el que se
entrenó la red: conversión a RGB y redimensionado a 64 x 64. Nada más: el escalado a [0, 1] y, si el modelo
la usa, la conversión a escala de grises van dentro del modelo. El modelo espera entradas con la misma forma
y la misma distribución que vio durante el entrenamiento.

La celda siguiente prepara el entorno. Si el modelo ya está en memoria lo reutiliza; si no, lo carga desde
`models/newDataset/cnn_final.keras`, junto con `cnn_final.json`, que guarda la resolución y el orden de
las clases. A diferencia del MLP con descriptores, no hace falta ejecutar antes ninguna otra celda.

In [ ]:
# Utilidad de predicción: autosuficiente, reutiliza lo que ya esté en memoria y carga lo que falte
import json
import os
import sys
from pathlib import Path

# Igual que en la sección 0: si esta celda se ejecuta sola en un kernel nuevo, Keras sobre
# PyTorch y la GPU oculta se fijan antes de importar torch y keras
if "keras" not in sys.modules:
    os.environ["KERAS_BACKEND"] = "torch"
    os.environ["KERAS_TORCH_DEVICE"] = "cpu"
if "torch" not in sys.modules:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
import keras

DIRECTORIO_MODELO_FINAL = Path("../../models/newDataset")

# 1. El modelo: si no se ejecutó el notebook completo, se carga el archivo guardado
if "modelo_final" not in globals():
    ruta_guardada = DIRECTORIO_MODELO_FINAL / "cnn_final.keras"
    if not ruta_guardada.exists():
        raise FileNotFoundError(
            f"No hay modelo en memoria ni en {ruta_guardada}. "
            "Ejecuta antes la selección del modelo final o entrena el notebook completo."
        )
    # compile=False: esta celda solo predice. El entrenamiento usa un optimizador
    # de PyTorch, asi que el optimizador de Keras guardado en el archivo esta vacio
    # y cargarlo solo produciria un aviso sin utilidad.
    modelo_final = keras.models.load_model(ruta_guardada, compile=False)
    print(f"Modelo cargado desde {ruta_guardada}")

# 2. El pre-procesamiento con el que se entrenó: resolución y orden de las clases, que define
#    qué índice de salida corresponde a qué personaje. La entrada (rgb o gris) es informativa:
#    la conversión a escala de grises, si el modelo la usa, va dentro del modelo
metadatos_modelo = json.loads((DIRECTORIO_MODELO_FINAL / "cnn_final.json").read_text(encoding="utf-8"))
representacion_modelo = metadatos_modelo["representacion"]
clases = metadatos_modelo["clases"]
ALTO_OBJETIVO, ANCHO_OBJETIVO = metadatos_modelo["alto"], metadatos_modelo["ancho"]


def predecir_imagen(ruta_imagen, top_k=5, mostrar=True):
    """Clasifica una imagen cualquiera y muestra el resultado.

    ruta_imagen : ruta a un archivo de imagen (jpg, png, ...).
    top_k       : cuántas clases candidatas mostrar, de mayor a menor probabilidad.
    Devuelve un DataFrame con esas candidatas y sus probabilidades."""
    ruta_imagen = Path(str(ruta_imagen).strip().strip('"').strip("'"))
    if not ruta_imagen.is_file():
        raise FileNotFoundError(f"No se encontro la imagen: {ruta_imagen}")

    # --- Mismo pre-procesamiento que en el entrenamiento ---
    with Image.open(ruta_imagen) as archivo:
        original = archivo.convert("RGB")
    # np.array y no np.asarray: la copia es escribible, y PyTorch avisa si recibe la
    # memoria de solo lectura de PIL
    reducida = np.array(
        original.resize((ANCHO_OBJETIVO, ALTO_OBJETIVO), Image.Resampling.BILINEAR),
        dtype=np.uint8,
    )
    # La imagen entra con su forma (1, alto, ancho, 3) y en uint8: la capa Rescaling del
    # modelo divide por 255 y, si el modelo usa escala de grises, la convierte dentro.
    tensor_imagen = reducida[np.newaxis]
    dispositivo = next(modelo_final.parameters()).device
    entrada = torch.as_tensor(tensor_imagen).to(dispositivo)

    probabilidades = modelo_final.predict(entrada, verbose=0)[0]
    mejores = np.argsort(-probabilidades)[:top_k]

    candidatas = pd.DataFrame(
        {
            "personaje": [clases[i].replace("_", " ") for i in mejores],
            "probabilidad": [round(float(probabilidades[i]), 4) for i in mejores],
        }
    )

    # Si la imagen viene de una carpeta con nombre de clase, se aprovecha para comparar
    etiqueta_real = ruta_imagen.parent.name if ruta_imagen.parent.name in clases else None
    prediccion = clases[int(mejores[0])]
    acierto = None if etiqueta_real is None else prediccion == etiqueta_real
    color = "#1b7f3b" if acierto in (None, True) else "#c0392b"

    if not mostrar:
        return candidatas

    figura, ejes = plt.subplots(
        1, 3, figsize=(13, 4.2), layout="constrained", gridspec_kw={"width_ratios": [1.2, 1, 2.2]}
    )

    # Panel 1: la imagen original, tal como está en el disco
    ejes[0].imshow(original)
    ejes[0].set_title(f"Original  {original.size[0]}x{original.size[1]}", fontsize=9)
    ejes[0].set_xlabel(ruta_imagen.name, fontsize=8, labelpad=6)

    # Panel 2: la imagen redimensionada que recibe la red
    ejes[1].imshow(reducida)
    ejes[1].set_title(f"Lo que ve el modelo  {ANCHO_OBJETIVO}x{ALTO_OBJETIVO}", fontsize=9)
    ejes[1].set_xlabel(f"tensor {ALTO_OBJETIVO}x{ANCHO_OBJETIVO}x3 ({representacion_modelo})", fontsize=8, labelpad=6)

    for eje in ejes[:2]:
        eje.set_xticks([])
        eje.set_yticks([])
        for lado in eje.spines.values():
            lado.set_edgecolor(color)
            lado.set_linewidth(3)

    # Panel 3: las candidatas con su probabilidad
    orden = candidatas.iloc[::-1]
    colores = [
        "#1b7f3b" if nombre.replace(" ", "_") == etiqueta_real else "#b0c4de"
        for nombre in orden["personaje"]
    ]
    if etiqueta_real is None:
        colores[-1] = "#2c7fb8"  # sin etiqueta conocida: se resalta solo la ganadora

    barras = ejes[2].barh(orden["personaje"], orden["probabilidad"] * 100, color=colores)
    ejes[2].bar_label(barras, fmt="%.1f %%", fontsize=8, padding=3)
    ejes[2].set_xlim(0, 112)
    ejes[2].set_xlabel("probabilidad asignada (%)")
    ejes[2].grid(alpha=0.3, axis="x")
    ejes[2].set_title(f"Top {top_k} de candidatas", fontsize=9)

    titulo = f"El modelo dice: {prediccion.replace('_', ' ').upper()}"
    titulo += f"   ({probabilidades[mejores[0]]:.1%} de confianza)"
    if etiqueta_real is not None:
        titulo += f"   |   real: {etiqueta_real.replace('_', ' ')}"
        titulo += "  ->  ACIERTO" if acierto else "  ->  ERROR"
    figura.suptitle(titulo, fontsize=13, fontweight="bold", color=color)
    plt.show()

    return candidatas


print(f"Listo ({representacion_modelo}). Usa predecir_imagen(<ruta>) en la celda siguiente.")

**Cómo usarla.** Basta con reemplazar la ruta de `RUTA_IMAGEN` por la de cualquier imagen del disco y
ejecutar la celda; se puede repetir tantas veces como se quiera sin volver a cargar el modelo. La `r`
delante de las comillas indica a Python que no interprete las barras invertidas de las rutas de Windows,
así que la ruta se puede pegar tal cual.

Cuando la imagen proviene de una carpeta con nombre de personaje —como ocurre en el dataset—, la función
lo detecta y compara la predicción con esa etiqueta. Si la imagen viene de cualquier otro lugar,
simplemente muestra la respuesta del modelo.

In [ ]:
# Cambia esta ruta por la de la imagen que quieras clasificar y vuelve a ejecutar la celda.
# Por defecto se usa la primera imagen de test, que el modelo no vio al entrenar.
RUTA_IMAGEN = datos_test["ruta"].iloc[0] if "datos_test" in globals() else r"C:\ruta\a\una\imagen.jpg"

candidatas = predecir_imagen(RUTA_IMAGEN, top_k=5)
display(candidatas)

**Advertencia sobre la interpretación.** La red siempre responde algo: el softmax reparte
probabilidad entre los 64 personajes conocidos y devuelve la mayor, aunque la imagen no contenga a ningún
personaje de Los Simpson. No existe una opción "ninguno de los anteriores". Por eso la confianza es tan
informativa como el nombre: un valor cercano al 100 % indica una decisión clara, mientras que una
ganadora con 15 % frente a candidatas de 12 % significa que el modelo no tiene criterio para decidir.

Conviene recordar también los límites vistos en la evaluación: el modelo reconoce con solidez a los
personajes con muchas imágenes de entrenamiento y falla con frecuencia en los minoritarios, y es sensible
al encuadre y al fondo. Una foto con el personaje pequeño dentro de una escena cargada tiene bastantes
menos probabilidades de clasificarse bien que un primer plano.

## Referencias

**Dataset**

- Alfaro, A. (s. f.). *Los Simpson* [conjunto de datos]. Kaggle.
  https://www.kaggle.com/datasets/alfaro96/los-simpson. Es la base del dataset de este
  notebook, que el equipo amplió a 64 personajes.

**Artículos**

- LeCun, Y., Bottou, L., Bengio, Y. y Haffner, P. (1998). Gradient-based learning applied to document
  recognition. *Proceedings of the IEEE, 86*(11), 2278-2324. Las redes convolucionales con pesos compartidos
  y pooling.
- Ioffe, S. y Szegedy, C. (2015). Batch Normalization: Accelerating deep network training by reducing internal
  covariate shift. *ICML 2015*. https://arxiv.org/abs/1502.03167
- Simonyan, K. y Zisserman, A. (2015). Very deep convolutional networks for large-scale image recognition.
  *ICLR 2015*. https://arxiv.org/abs/1409.1556. Los bloques de convoluciones de 3 × 3 (experimento E2).
- Lin, M., Chen, Q. y Yan, S. (2014). Network in network. *ICLR 2014*. https://arxiv.org/abs/1312.4400. El
  pooling global como entrada de la cabeza clasificadora.
- Selvaraju, R. R., Cogswell, M., Das, A., Vedantam, R., Parikh, D. y Batra, D. (2017). Grad-CAM: Visual
  explanations from deep networks via gradient-based localization. *ICCV 2017*. https://arxiv.org/abs/1610.02391

**Documentación de las librerías utilizadas**

- Keras — *Keras API reference*. https://keras.io/api/
- Keras — *Conv2D layer*. https://keras.io/api/layers/convolution_layers/convolution2d/
- Keras — *MaxPooling2D* y *GlobalAveragePooling2D*. https://keras.io/api/layers/pooling_layers/
- Keras — *BatchNormalization layer*. https://keras.io/api/layers/normalization_layers/batch_normalization/
- PyTorch — *Automatic differentiation package* (`torch.autograd`). https://pytorch.org/docs/stable/autograd.html
- scikit-learn — *Metrics and scoring*. https://scikit-learn.org/stable/modules/model_evaluation.html
- scikit-learn — `train_test_split`. https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- scikit-learn — `compute_class_weight`. https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html
- SciPy — `scipy.signal.correlate2d`. https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.correlate2d.html
- NumPy — *NumPy documentation*. https://numpy.org/doc/stable/
- pandas — *pandas documentation*. https://pandas.pydata.org/docs/
- Matplotlib — *Matplotlib documentation*. https://matplotlib.org/stable/index.html
- seaborn — *seaborn documentation*. https://seaborn.pydata.org/
- Pillow — *Pillow (PIL Fork) documentation*. https://pillow.readthedocs.io/en/stable/
- joblib — *Embarrassingly parallel for loops*. https://joblib.readthedocs.io/en/stable/parallel.html

**Material del curso**

- Jerez Salazar, F. J. *1.1.1 El Perceptrón*. Machine Learning, TLY1102.
- Jerez Salazar, F. J. *1.2.1 Redes Fully Connected*. Machine Learning, TLY1102.
- Jerez Salazar, F. J. *1.2.2 Backpropagation*. Machine Learning, TLY1102.
- Jerez Salazar, F. J. *1.3.1 Descenso del Gradiente*. Machine Learning, TLY1102.
- Jerez Salazar, F. J. *1.3.2 Optimizadores*. Machine Learning, TLY1102.
- Jerez Salazar, F. J. *1.4.1 Diseño y Evaluación de Modelos*. Machine Learning, TLY1102.
- Jerez Salazar, F. J. *1.4.2 Implementación con Keras*. Machine Learning, TLY1102.

## Resumen de rendimiento de la corrida (CPU vs GPU)

Esta última sección no añade nada al modelo: mide **cuánto costó obtenerlo**. Las secciones anteriores
responden qué tan bien clasifica la red; esta responde en qué hardware se entrenó, cuánto tardó cada
etapa y qué se gana al mover el mismo trabajo de la CPU a la GPU.

Se reportan tres cosas:

1. **El entorno y las marcas de tiempo.** Qué dispositivo ejecutó realmente las operaciones —CPU o
   GPU—, con su nombre, el sistema operativo y la versión del framework, además de la resolución de
   entrada, la **fecha y hora de inicio** de la corrida (la fija la primera celda del notebook), la
   **hora de término** y la duración total. Es el dato que da sentido a todos los tiempos.
2. **Los tiempos de la corrida.** Segundos y segundos por época de cada configuración entrenada, más un
   *benchmark* corto de tres épocas sobre la arquitectura base y una medición de inferencia sobre el
   conjunto de test. El benchmark existe porque los tiempos de las secciones anteriores no son
   comparables entre sí: la parada temprana hace que cada configuración corra un número distinto de
   épocas. Entrenar unas pocas épocas con la misma arquitectura, el mismo batch y sin *callbacks* sí da
   una cifra homogénea. De esas épocas se descarta la primera, que incluye la compilación del grafo y el
   traslado inicial de los datos al dispositivo.
3. **La comparación entre entornos.** Cada notebook guarda su resumen en una carpeta propia dentro de
   `../../resultados/newDataset/`, con la resolución en el nombre del archivo:

   ```
   resultados/newDataset/
   ├── CPU/
   │   ├── rendimiento_cpu_32x32.json
   │   ├── rendimiento_cpu_64x64.json
   │   └── rendimiento_cpu_128x128.json
   └── GPU/
       ├── rendimiento_gpu_32x32.json
       ├── rendimiento_gpu_64x64.json
       └── rendimiento_gpu_128x128.json
   ```

   La celda cruza únicamente el par que comparte resolución, porque comparar una corrida de 64 x 64
   con otra de 128 x 128 mediría el tamaño del problema y no el hardware. Cuando existen los dos,
   muestra la tabla y el gráfico comparativos con la aceleración obtenida en cada medición; si solo
   está el de este entorno, se guarda igual, se avisa de cuál falta y se indica a qué resoluciones sí
   hay corridas guardadas del otro.

Conviene recordar al leer las cifras que una CNN es, sobre todo, convoluciones: la misma operación
repetida en todas las posiciones de la imagen, exactamente el tipo de cálculo que una GPU paraleliza
bien. A diferencia del MLP sobre descriptores, aquí el cómputo crece con el número de píxeles, así que la
diferencia entre los dos dispositivos debería notarse mucho más en el entrenamiento. Las métricas de calidad (accuracy, F1) aparecen en la tabla solo para
verificar que el cambio de hardware **no** cambia lo que el modelo aprende: si difieren mucho, la causa
está en el pre-procesamiento o en los hiperparámetros, no en el dispositivo.

Las dos corridas usan el mismo framework, Keras 3 sobre PyTorch, y el mismo bucle de entrenamiento, así
que la diferencia de tiempos se debe al hardware. Al final de esta sección están la comparación con el
MLP y la de las corridas guardadas de la CNN, con sus conclusiones.

In [ ]:
# Resumen de rendimiento de la corrida: entorno de ejecución, marcas de tiempo,
# tiempos y métricas finales. Cada notebook escribe su resumen en
# ../../resultados/newDataset/<CPU|GPU>/rendimiento_<entorno>_<resolucion>.json y, si existe el
# del otro entorno a la misma resolución, se muestra la comparación lado a lado.
import json
import platform
import subprocess
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import keras

DIRECTORIO_RESULTADOS = Path("../../resultados/newDataset")
DIRECTORIO_RESULTADOS.mkdir(parents=True, exist_ok=True)

# La resolución de entrada forma parte del nombre del archivo: una corrida a 64x64
# y otra a 128x128 entrenan modelos de distinto tamaño y no son comparables entre sí.
RESOLUCION = f"{ALTO_OBJETIVO}x{ANCHO_OBJETIVO}"

# La marca de inicio la fija la primera celda del notebook; si esa celda no se
# ejecutó en esta sesión, se toma el arranque de esta sección como referencia.
INICIO_CORRIDA = globals().get("INICIO_CORRIDA") or datetime.now()

# Re-entrena unas pocas épocas para medir un tiempo por época comparable entre
# entornos (misma arquitectura, mismo batch, sin parada temprana). False para omitirlo.
EJECUTAR_BENCHMARK = True
EPOCAS_BENCHMARK = 3

# Esta es la medicion en CPU de la misma red, con el mismo bucle de entrenamiento que el
# notebook de GPU: la seccion 0 oculta la GPU, asi que Keras y PyTorch trabajan solo en la CPU.


# --- 1. Entorno de ejecución -------------------------------------------------
def describir_cpu():
    """Nombre del procesador, en Windows y en Linux."""
    if platform.system() == "Windows":
        try:
            nombre = subprocess.run(
                ["wmic", "cpu", "get", "name"], capture_output=True, text=True, timeout=10
            ).stdout.strip().splitlines()
            if len(nombre) > 1 and nombre[1].strip():
                return nombre[1].strip()
        except (OSError, subprocess.SubprocessError):
            pass
        return platform.processor() or platform.machine()
    try:
        for linea in Path("/proc/cpuinfo").read_text().splitlines():
            if linea.startswith("model name"):
                return linea.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def describir_gpu():
    """Nombre comercial y capacidad de cómputo de la GPU que ve PyTorch."""
    mayor, menor = torch.cuda.get_device_capability(0)
    return f"{torch.cuda.get_device_name(0)} (CC {mayor}.{menor})"


usa_gpu = torch.cuda.is_available()
ENTORNO = "GPU" if usa_gpu else "CPU"
dispositivo = describir_gpu() if usa_gpu else describir_cpu()

entorno = pd.DataFrame(
    [
        ("Entorno", ENTORNO),
        ("Modelo", "Red neuronal convolucional (CNN)"),
        ("Dispositivo de cómputo", dispositivo),
        ("CPU del equipo", describir_cpu()),
        ("GPUs visibles para PyTorch", torch.cuda.device_count()),
        ("Sistema operativo", f"{platform.system()} {platform.release()}"),
        ("Python", platform.python_version()),
        ("Framework", f"Keras {keras.__version__} sobre PyTorch {torch.__version__}"),
        ("Resolución de trabajo", RESOLUCION),
        ("Entrada del modelo base", f"imagen {RESOLUCION}x3 ({REPRESENTACION_BASE})"),
        ("Entrada del modelo final", f"imagen {RESOLUCION}x3 ({representacion_final})"),
        ("Arquitectura del modelo final", arquitectura_final),
        ("Inicio de la corrida", INICIO_CORRIDA.strftime("%Y-%m-%d %H:%M:%S")),
    ],
    columns=["Concepto", "Valor"],
)
display(entorno)


# --- 2. Tiempos de entrenamiento --------------------------------------------
# El tiempo del modelo base lo mide la celda de entrenamiento; si esa celda no se
# volvió a ejecutar, queda como desconocido en lugar de inventar un valor.
segundos_base = float(globals().get("segundos_entrenamiento_base", np.nan))
epocas_base = len(historial_base.history["loss"])

tiempos = tabla_experimentos[["experimento", "epocas", "parametros", "segundos"]].copy()
# El modelo base entra en la tabla de experimentos sin tiempo: se completa aquí
tiempos.loc[tiempos["experimento"].str.startswith("Base"), "segundos"] = segundos_base
tiempos["segundos_por_epoca"] = (tiempos["segundos"] / tiempos["epocas"]).round(2)
tiempos["imagenes_por_segundo"] = (
    len(X_train) * tiempos["epocas"] / tiempos["segundos"]
).round(0)

display(tiempos)

segundos_experimentos = float(
    tabla_experimentos.loc[
        ~tabla_experimentos["experimento"].str.startswith("Base"), "segundos"
    ].sum()
)
segundos_totales = float(np.nansum([segundos_base, segundos_experimentos]))
segundos_por_epoca_medio = float(tiempos["segundos_por_epoca"].mean(skipna=True))
epocas_totales = int(tiempos["epocas"].sum())


# --- 3. Benchmark comparable: entrenamiento e inferencia ---------------------
def cronometrar_epocas(modelo, X, y, batch, epocas):
    """Entrena `epocas` épocas seguidas con el mismo bucle del entrenamiento y devuelve el
    tiempo medio por época.

    Se descarta la primera: incluye la reserva de memoria y el calentamiento, que no se
    repiten en las siguientes.
    """
    optimizador = torch.optim.Adam(modelo.parameters(), lr=TASA_APRENDIZAJE, fused=True)
    generador = torch.Generator(device=X.device)
    generador.manual_seed(SEMILLA)

    duraciones = []
    for _ in range(epocas):
        inicio = time.perf_counter()
        _pasada(modelo, X, y, batch, True, optimizador, None, generador)
        duraciones.append(time.perf_counter() - inicio)
    utiles = duraciones[1:] or duraciones
    return float(np.mean(utiles))


benchmark_segundos_epoca = np.nan
if EJECUTAR_BENCHMARK:
    keras.utils.set_random_seed(SEMILLA)
    modelo_benchmark = construir_cnn(nombre="cnn_benchmark").to(DISPOSITIVO)
    benchmark_segundos_epoca = cronometrar_epocas(
        modelo_benchmark, X_train, y_train, TAMANO_BATCH, EPOCAS_BENCHMARK
    )
    del modelo_benchmark

# Inferencia sobre el conjunto de evaluación completo, tras una pasada de calentamiento
modelo_final.predict(X_test[:256], batch_size=256, verbose=0)
inicio_inferencia = time.perf_counter()
modelo_final.predict(X_test, batch_size=256, verbose=0)
segundos_inferencia = time.perf_counter() - inicio_inferencia

filas_benchmark = [
    (
        f"Entrenamiento: segundos por epoca (arquitectura base, batch {TAMANO_BATCH})",
        round(benchmark_segundos_epoca, 2) if EJECUTAR_BENCHMARK else "no ejecutado",
    ),
    (
        "Entrenamiento: imágenes por segundo",
        round(len(X_train) / benchmark_segundos_epoca) if EJECUTAR_BENCHMARK else "-",
    ),
    (
        f"Inferencia: segundos para las {len(X_test):,} imagenes de test",
        round(segundos_inferencia, 2),
    ),
    ("Inferencia: imágenes por segundo", round(len(X_test) / segundos_inferencia)),
]
display(pd.DataFrame(filas_benchmark, columns=["Medicion", "Valor"]))


# --- 4. Resumen de la corrida ------------------------------------------------
# El término se marca aquí, después del benchmark y de la medición de inferencia:
# es el último trabajo real de la corrida.
FIN_CORRIDA = datetime.now()
duracion_corrida_segundos = (FIN_CORRIDA - INICIO_CORRIDA).total_seconds()
duracion_corrida = str(timedelta(seconds=round(duracion_corrida_segundos)))

def cuatro_decimales(valor):
    """Métrica redondeada; la de un subconjunto vacío queda como None (null en el JSON)."""
    return None if np.isnan(valor) else round(float(valor), 4)


resumen_corrida = {
    "modelo": "CNN",
    "entorno": ENTORNO,
    "dispositivo": dispositivo,
    "cpu": describir_cpu(),
    "sistema_operativo": f"{platform.system()} {platform.release()}",
    "framework": "PyTorch",
    "framework_version": torch.__version__,
    "keras": keras.__version__,
    "inicio_corrida": INICIO_CORRIDA.strftime("%Y-%m-%d %H:%M:%S"),
    "fin_corrida": FIN_CORRIDA.strftime("%Y-%m-%d %H:%M:%S"),
    "duracion_corrida": duracion_corrida,
    "duracion_corrida_segundos": round(duracion_corrida_segundos, 1),
    "resolucion": RESOLUCION,
    "dimension_entrada": int(DIMENSION_ENTRADA),
    "representacion_base": REPRESENTACION_BASE,
    "representacion_final": representacion_final,
    "dimension_entrada_final": int(DIMENSIONES[representacion_final]),
    "personajes": int(NUMERO_CLASES),
    "proporciones_train_val_test": " / ".join(f"{PROPORCIONES[nombre]:.0%}" for nombre in CONJUNTOS),
    "esquema_pesos": ESQUEMA_PESOS,
    "suavizado_etiquetas": SUAVIZADO_ETIQUETAS,
    "peso_escena": PESO_ESCENA,
    "imagenes_train_alteradas": int(datos.loc[MASCARAS["train"], "alterada"].sum()),
    "imagenes_val_alteradas": int(datos.loc[MASCARAS["val"], "alterada"].sum()),
    "imagenes_test_alteradas": int(datos.loc[MASCARAS["test"], "alterada"].sum()),
    "imagenes_parecidas": int(datos["parecida"].sum()),
    "pares_gemelas": int(n_pares_gemelas),
    "segundos_carga_imagenes": round(segundos_carga, 1),
    # La CNN no calcula descriptores: su entrada son los pixeles
    "segundos_descriptores": 0.0,
    "imagenes_train": int(len(X_train)),
    "imagenes_val": int(len(X_val)),
    "imagenes_test": int(len(X_test)),
    "batch": int(TAMANO_BATCH),
    "configuracion_final": nombre_final,
    "arquitectura_final": arquitectura_final,
    "parametros_modelo_final": int(modelo_final.count_params()),
    "epocas_modelo_base": int(epocas_base),
    "segundos_modelo_base": None if np.isnan(segundos_base) else round(segundos_base, 1),
    "segundos_experimentos": round(segundos_experimentos, 1),
    "segundos_entrenamiento_total": round(segundos_totales, 1),
    "epocas_totales": epocas_totales,
    "segundos_por_epoca_medio": round(segundos_por_epoca_medio, 2),
    "benchmark_segundos_por_epoca": (
        None if np.isnan(benchmark_segundos_epoca) else round(benchmark_segundos_epoca, 2)
    ),
    "segundos_inferencia_test": round(segundos_inferencia, 2),
    "inferencia_imagenes_por_segundo": round(len(X_test) / segundos_inferencia, 1),
    "criterio_acierto": (
        "carpeta o personaje presente (escena + gemelas + verificación por imagen de los errores con confianza alta)"
    ),
    "umbral_confianza_verificacion": UMBRAL_CONFIANZA_VERIFICACION,
    "umbral_similitud_verificacion": UMBRAL_SIMILITUD_VERIFICACION,
    "errores_confiados_test": int(len(verificacion_test)),
    "errores_confiados_verificados_test": int(verificado_test.sum()),
    "accuracy_test": round(float(accuracy_test), 4),
    "accuracy_estricta_test": round(float(accuracy_estricta_test), 4),
    "top3_test": round(float(top3_test), 4),
    "accuracy_tolerante_test": round(accuracy_tolerante_test, 4),
    "accuracy_sin_verificacion_test": round(accuracy_sin_verificacion_test, 4),
    "accuracy_originales_test": cuatro_decimales(accuracy_originales_test),
    "accuracy_parecidas_test": cuatro_decimales(accuracy_parecidas_test),
    "accuracy_alteradas_test": cuatro_decimales(accuracy_alteradas_test),
    "f1_macro_test": round(float(f1_macro_test), 4),
    "f1_macro_estricto_test": round(float(f1_macro_estricto_test), 4),
    "f1_ponderado_test": round(float(f1_ponderado_test), 4),
    "perdida_test": round(float(perdida_test), 4),
}

# Una carpeta por entorno y un archivo por resolución:
# ../../resultados/newDataset/CPU/rendimiento_cpu_128x128.json
directorio_entorno = DIRECTORIO_RESULTADOS / ENTORNO
directorio_entorno.mkdir(parents=True, exist_ok=True)
ruta_resumen = directorio_entorno / f"rendimiento_{ENTORNO.lower()}_{RESOLUCION}.json"
ruta_resumen.write_text(json.dumps(resumen_corrida, indent=2, ensure_ascii=False), encoding="utf-8")

display(
    pd.DataFrame(
        [
            (clave, valor if valor is not None else "desconocido")
            for clave, valor in resumen_corrida.items()
        ],
        columns=["Concepto", "Valor"],
    )
)
print(f"Resumen guardado en: {ruta_resumen.resolve()}")


# --- 5. Comparación CPU vs GPU ----------------------------------------------
# Solo es posible cuando ya se ejecutaron ambos notebooks A LA MISMA RESOLUCIÓN:
# cada uno deja su json en ../../resultados/newDataset/<ENTORNO>/ y esta sección cruza el par
# que comparte resolución, que es el único par que compara hardware y no tamaños.
def ruta_resumen_de(nombre_entorno, resolucion):
    carpeta = DIRECTORIO_RESULTADOS / nombre_entorno.upper()
    return carpeta / f"rendimiento_{nombre_entorno.lower()}_{resolucion}.json"


def resoluciones_guardadas(nombre_entorno):
    carpeta = DIRECTORIO_RESULTADOS / nombre_entorno.upper()
    if not carpeta.is_dir():
        return []
    patron = f"rendimiento_{nombre_entorno.lower()}_*.json"
    return sorted(archivo.stem.rsplit("_", 1)[-1] for archivo in carpeta.glob(patron))


def framework_de(resumen):
    """Los resúmenes antiguos guardaban la versión bajo la clave 'tensorflow'."""
    if "framework" in resumen:
        return f"{resumen['framework']} {resumen.get('framework_version', '')}".strip()
    if "tensorflow" in resumen:
        return f"TensorFlow {resumen['tensorflow']}"
    return "desconocido"


resumenes = {}
for nombre_entorno in ("cpu", "gpu"):
    ruta = ruta_resumen_de(nombre_entorno, RESOLUCION)
    if ruta.exists():
        resumenes[nombre_entorno.upper()] = json.loads(ruta.read_text(encoding="utf-8"))

if len(resumenes) < 2:
    faltante = "GPU" if "GPU" not in resumenes else "CPU"
    print(
        f"\nSolo hay resultados a {RESOLUCION} del entorno {ENTORNO}. Ejecuta tambien "
        f"el notebook del entorno {faltante} con la misma resolucion para ver la "
        f"comparacion; cada uno escribe su resumen en "
        f"{(DIRECTORIO_RESULTADOS / faltante).resolve()}."
    )
    otras = [r for r in resoluciones_guardadas(faltante) if r != RESOLUCION]
    if otras:
        print(
            f"El entorno {faltante} si tiene corridas guardadas a: {', '.join(otras)}. "
            f"No se comparan con esta porque una resolucion distinta cambia el tamano "
            f"de la entrada y del modelo, y los tiempos dejarian de medir el hardware."
        )
else:
    cpu, gpu = resumenes["CPU"], resumenes["GPU"]

    # (etiqueta, clave, "menor" = menos es mejor, "mayor" = más es mejor, None = no se compara)
    filas_comparacion = [
        ("Dispositivo", "dispositivo", None),
        ("Sistema operativo", "sistema_operativo", None),
        ("Framework", None, None),
        ("Inicio de la corrida", "inicio_corrida", None),
        ("Termino de la corrida", "fin_corrida", None),
        ("Duracion de la corrida", "duracion_corrida", None),
        ("Resolución de entrada", "resolucion", None),
        ("Parámetros del modelo final", "parametros_modelo_final", None),
        ("Épocas totales entrenadas", "epocas_totales", None),
        ("Segundos: modelo base", "segundos_modelo_base", "menor"),
        ("Segundos: experimentos", "segundos_experimentos", "menor"),
        ("Segundos: entrenamiento total", "segundos_entrenamiento_total", "menor"),
        ("Segundos por época (medio)", "segundos_por_epoca_medio", "menor"),
        ("Segundos por época (benchmark)", "benchmark_segundos_por_epoca", "menor"),
        ("Segundos: inferencia sobre test", "segundos_inferencia_test", "menor"),
        ("Inferencia: imágenes por segundo", "inferencia_imagenes_por_segundo", "mayor"),
        ("Accuracy en test", "accuracy_test", "mayor"),
        ("Top-3 accuracy en test", "top3_test", "mayor"),
        ("F1 macro en test", "f1_macro_test", "mayor"),
    ]

    registros = []
    for etiqueta, clave, direccion in filas_comparacion:
        if clave is None:  # fila de framework, que no vive en una sola clave
            registros.append({"Concepto": etiqueta, "CPU": framework_de(cpu),
                              "GPU": framework_de(gpu), "Diferencia": "-"})
            continue
        valor_cpu, valor_gpu = cpu.get(clave), gpu.get(clave)
        diferencia = "-"
        comparables = isinstance(valor_cpu, (int, float)) and isinstance(valor_gpu, (int, float))
        if direccion and comparables:
            if direccion == "menor" and valor_gpu > 0:
                diferencia = f"GPU x{valor_cpu / valor_gpu:.2f} mas rapida"
            elif direccion == "mayor" and valor_cpu > 0:
                diferencia = f"GPU x{valor_gpu / valor_cpu:.2f}"
        registros.append(
            {
                "Concepto": etiqueta,
                "CPU": valor_cpu if valor_cpu is not None else "desconocido",
                "GPU": valor_gpu if valor_gpu is not None else "desconocido",
                "Diferencia": diferencia,
            }
        )

    comparacion_entornos = pd.DataFrame(registros)
    display(comparacion_entornos)

    mismo_framework = framework_de(cpu).split()[0] == framework_de(gpu).split()[0]
    print(
        f"\nAmbas corridas entrenan la misma red sobre la misma entrada "
        f"(representacion base a {RESOLUCION}, {cpu['dimension_entrada']} valores por imagen)."
    )
    if mismo_framework:
        print("Las dos usan el mismo framework, asi que la diferencia de tiempos es "
              "atribuible al hardware.")
    else:
        print(
            f"ATENCION: las dos corridas NO usan el mismo framework ({framework_de(cpu)} "
            f"en CPU frente a {framework_de(gpu)} en GPU), asi que la diferencia de "
            f"tiempos mezcla hardware Y framework y no debe leerse como una medida "
            f"limpia de la ganancia del hardware."
        )
        print(
            "Para leer la ganancia del hardware sin la interferencia del framework "
            "habria que ejecutar los dos notebooks sobre el mismo framework."
        )

    # Tiempos lado a lado, en escala logarítmica: los órdenes de magnitud suelen diferir
    metricas_grafico = [
        ("Entrenamiento total", "segundos_entrenamiento_total"),
        ("Por época (medio)", "segundos_por_epoca_medio"),
        ("Por época (benchmark)", "benchmark_segundos_por_epoca"),
        ("Inferencia sobre test", "segundos_inferencia_test"),
    ]
    metricas_grafico = [
        (etiqueta, clave)
        for etiqueta, clave in metricas_grafico
        if isinstance(cpu.get(clave), (int, float)) and isinstance(gpu.get(clave), (int, float))
    ]

    if metricas_grafico:
        posiciones = np.arange(len(metricas_grafico))
        valores_cpu = [cpu[clave] for _, clave in metricas_grafico]
        valores_gpu = [gpu[clave] for _, clave in metricas_grafico]

        figura, eje = plt.subplots(figsize=(10, 4.5))
        barras_cpu = eje.bar(posiciones - 0.2, valores_cpu, width=0.4, label="CPU", color="#4c78a8")
        barras_gpu = eje.bar(posiciones + 0.2, valores_gpu, width=0.4, label="GPU", color="#59a14f")

        eje.set_yscale("log")
        eje.set_xticks(posiciones)
        eje.set_xticklabels([etiqueta for etiqueta, _ in metricas_grafico], fontsize=9)
        eje.set_ylabel("Segundos (escala logarítmica)")
        eje.set_title(f"Tiempos de la corrida a {RESOLUCION}: CPU frente a GPU", fontsize=11)
        eje.legend()
        eje.grid(alpha=0.3, axis="y")

        for barras in (barras_cpu, barras_gpu):
            eje.bar_label(barras, fmt="%.1f", fontsize=8, padding=2)

        plt.tight_layout()
        plt.show()

### Comparación con el MLP

Las dos redes se entrenan con el mismo proceso sobre las mismas imágenes: la misma partición, los mismos pesos
por clase, el mismo bucle de entrenamiento y el mismo criterio de acierto. La celda siguiente busca el resumen
del MLP del mismo entorno y la misma resolución (`PMC/resultados/newDataset/<CPU|GPU>/`) y pone las dos
corridas lado a lado. Si no existe, avisa y muestra a qué resoluciones hay corridas del MLP.

Una diferencia al leer los tiempos de inferencia: los del MLP miden solo la red sobre los descriptores ya
calculados, y los de la CNN, la red sobre los píxeles. El costo completo de clasificar una imagen nueva con el
MLP incluye además calcular sus descriptores, que se hace en la CPU.

In [ ]:
# Resumen del MLP (PMC) con el mismo entorno y la misma resolución que esta corrida
DIRECTORIO_RESULTADOS_PMC = Path("../../../PMC/resultados/newDataset")
directorio_pmc = DIRECTORIO_RESULTADOS_PMC / ENTORNO
ruta_pmc = directorio_pmc / f"rendimiento_{ENTORNO.lower()}_{RESOLUCION}.json"

if not ruta_pmc.exists():
    disponibles = (
        sorted(archivo.stem.rsplit("_", 1)[-1] for archivo in directorio_pmc.glob("rendimiento_*.json"))
        if directorio_pmc.is_dir()
        else []
    )
    print(f"No hay resumen del MLP en {ENTORNO} a {RESOLUCION}: {ruta_pmc.resolve()}")
    if disponibles:
        print(f"El MLP tiene corridas en {ENTORNO} a: {', '.join(disponibles)}.")
else:
    pmc = json.loads(ruta_pmc.read_text(encoding="utf-8"))
    cnn = resumen_corrida

    # (etiqueta, clave, "mayor" = más es mejor, "menor" = menos es mejor, None = no se compara)
    filas_mlp = [
        ("Configuración final", "configuracion_final", None),
        ("Parámetros del modelo final", "parametros_modelo_final", None),
        ("Accuracy en test", "accuracy_test", "mayor"),
        ("Accuracy estricta en test", "accuracy_estricta_test", "mayor"),
        ("Top-3 accuracy en test", "top3_test", "mayor"),
        ("F1 macro en test", "f1_macro_test", "mayor"),
        ("F1 macro estricto en test", "f1_macro_estricto_test", "mayor"),
        ("Accuracy en originales", "accuracy_originales_test", "mayor"),
        ("Accuracy en parecidas", "accuracy_parecidas_test", "mayor"),
        ("Accuracy en alteradas", "accuracy_alteradas_test", "mayor"),
        ("Errores con confianza > 0,80", "errores_confiados_test", None),
        ("Segundos: entrenamiento total", "segundos_entrenamiento_total", "menor"),
        ("Segundos por época (benchmark)", "benchmark_segundos_por_epoca", "menor"),
        ("Inferencia: imágenes por segundo", "inferencia_imagenes_por_segundo", "mayor"),
        ("Duración de la corrida", "duracion_corrida", None),
    ]
    registros_mlp = []
    for etiqueta, clave, direccion in filas_mlp:
        valor_mlp, valor_cnn = pmc.get(clave), cnn.get(clave)
        diferencia = "-"
        comparables = isinstance(valor_mlp, (int, float)) and isinstance(valor_cnn, (int, float))
        if direccion == "mayor" and comparables:
            diferencia = (
                f"{valor_cnn - valor_mlp:+.4f}" if valor_mlp <= 1 and valor_cnn <= 1
                else f"CNN x{valor_cnn / valor_mlp:.2f}"
            )
        elif direccion == "menor" and comparables and valor_mlp > 0:
            diferencia = f"CNN x{valor_cnn / valor_mlp:.2f} el tiempo del MLP"
        registros_mlp.append(
            {
                "Concepto": etiqueta,
                "MLP (PMC)": valor_mlp if valor_mlp is not None else "desconocido",
                "CNN": valor_cnn if valor_cnn is not None else "desconocido",
                "Diferencia": diferencia,
            }
        )
    comparacion_mlp = pd.DataFrame(registros_mlp)
    display(comparacion_mlp)
    print(
        f"MLP: corrida del {pmc.get('inicio_corrida', '?')}, entrada {pmc.get('representacion_final', '?')}"
        f" ({pmc.get('dimension_entrada_final', '?')} valores).   CNN: entrada imagen {RESOLUCION}x3"
        f" ({representacion_final})."
    )

    metricas_mlp = [
        ("Accuracy", "accuracy_test"),
        ("Accuracy estricta", "accuracy_estricta_test"),
        ("Top-3", "top3_test"),
        ("F1 macro", "f1_macro_test"),
    ]
    posiciones = np.arange(len(metricas_mlp))
    figura, eje = plt.subplots(figsize=(9, 4.2))
    barras_mlp = eje.bar(posiciones - 0.2, [pmc[clave] for _, clave in metricas_mlp], width=0.4,
                         label="MLP (PMC)", color="#8fb8de")
    barras_cnn = eje.bar(posiciones + 0.2, [cnn[clave] for _, clave in metricas_mlp], width=0.4,
                         label="CNN", color="#d95f02")
    for barras in (barras_mlp, barras_cnn):
        eje.bar_label(barras, fmt="%.3f", fontsize=8, padding=2)
    eje.set_xticks(posiciones)
    eje.set_xticklabels([etiqueta for etiqueta, _ in metricas_mlp])
    eje.set_ylim(0, 1)
    eje.set_ylabel("Valor en test")
    eje.set_title(f"MLP frente a CNN en test ({ENTORNO}, {RESOLUCION})", fontsize=11)
    eje.legend()
    eje.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()

**Lectura.** Con el mismo proceso, los mismos datos y la misma resolución, la CNN supera al MLP en todas las
métricas de calidad, y por márgenes parecidos en los tres tipos de imagen: entre 25 y 29 puntos de accuracy en
originales, parecidas y alteradas. No aprovecha mejor un tipo particular de imagen: reconoce mejor a los
personajes en general.

El precio está en el entrenamiento: 5,6 veces más tiempo (96 minutos frente a 17) y 5,6 veces más por época en
el benchmark (9,8 s frente a 1,8). En inferencia la comparación depende de qué se mida: la red del MLP sola es
diez veces más rápida que la CNN (63.000 frente a 6.400 imágenes por segundo), pero antes hay que calcular sus
descriptores en la CPU, y de extremo a extremo la CNN clasifica más imágenes por segundo (5.326 frente a
1.285–1.557 en las corridas del MLP que lo midieron). La CNN también comete más errores confiados (64 frente a
35), aunque casi todos nombran a un personaje presente en la imagen (ver «Ejemplos mal clasificados»).

### Corridas guardadas

La CNN se ejecutó completa el 19-10-2026 en la GPU a 64 × 64, sobre el mismo dataset que las corridas del MLP
(64 personajes, 40.123 originales) y en el mismo equipo (Intel Core i7-13620H y NVIDIA GeForce RTX 4060 Laptop,
conectado a la corriente). Las cifras salen del JSON de `resultados/newDataset/GPU/`.

| | GPU 64 × 64 |
|---|---:|
| Configuración final | E2 - 2 convoluciones por bloque |
| Parámetros del modelo final | 1.258.400 |
| Accuracy en test | 0,830 |
| Accuracy estricta en test | 0,777 |
| Top-3 en test | 0,932 |
| F1 macro en test | 0,788 |
| Accuracy en originales / parecidas / alteradas | 0,83 / 0,89 / 0,79 |
| Errores con confianza > 0,80 (verificados por imagen) | 64 (1) |
| Carga de imágenes (min) | 4,7 |
| Entrenamiento total (min) | 95,6 |
| Segundos por época (benchmark) | 9,84 |
| Utilización media de la GPU | 90,7 % |
| Duración de la corrida | 1 h 44 min 33 s |

**Conclusiones**

1. **La GPU trabaja de verdad.** A batch 128 la tarjeta está ocupada el 91 % del tiempo y consume unos 70 W,
   frente al 22–34 % y unos 10 W del MLP. El barrido de tamaño de lote lo confirma: de batch 128 a 512 el
   rendimiento sube apenas un 7 % (de 3.530 a 3.782 imágenes por segundo), cuando en el MLP se multiplicaba por
   14 a 16. Batch 1.024 y 2.048 ya no caben en los 8 GB de la tarjeta.
2. **El entrenamiento domina la corrida.** Cargar las imágenes toma 4,7 minutos y entrenar las catorce
   configuraciones, 96: el 91 % de la corrida. E2, con dos convoluciones por bloque, cuesta el doble por época
   (21,3 s) que las demás.
3. **Corrida de CPU pendiente.** El notebook de CPU (`LosSimpsonsCNN.ipynb`) sigue el mismo proceso, pero a
   64 × 64 cada época le cuesta unos 130 s, trece veces más que en la GPU: una corrida completa tomaría más de
   quince horas. Cuando se ejecute, su JSON quedará en `resultados/newDataset/CPU/` y la celda de resumen de
   rendimiento mostrará la comparación entre los dos dispositivos.